# Prodrugs
### PHS 312 - Drug Discovery, Design, and Development
**Lecture 32 · Wednesday, November 18, 2026 · 50 minutes**

Pre-class reading: Patrick 7e, Section 14.6

**This activity takes 30-35 minutes.** Work in pairs. Run each cell with the
play button on its left, answer the question that follows, and keep going.
You will not write any code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spinner101/phs312-cadd-notebooks/blob/main/notebooks/PHS312_L32_Prodrugs.ipynb)

> **New:** Parts 2–4 are **lecture notes**: key ideas, annotated 2D structures (each prodrug drawn above its active drug), worked examples, step-by-step calculation cards (each with a *Solve it yourself* cell) and calculators. Read them before class or use them afterwards for review. The timed pair activity begins at Part 5; after it, Part 6 is a **design** part (build a prodrug from a stated problem, with a tool and two checks) and Part 7 is a reference table of marketed prodrugs.

---
### Before you start - what this notebook does with your work

* **Nothing you type is sent anywhere.** This notebook makes no network
  calls, writes no files, and does not connect to Google Drive. Your answers
  live in the memory of this browser tab and disappear when you close it.
* **Your instructor cannot see this page.** At the end you will get a short
  report code. You choose to paste it into Canvas - that is the only thing
  that reaches your instructor.
* **Do not type anything personal.** No patient information, no identifiers,
  nothing about a real person. Everything here is simulated.
* Opening this notebook signs you in with your Hampton Google account, which
  is covered by the university's agreement with Google. If you would rather
  not use a Google account, tell your instructor and you will be given an
  offline copy.
---

In [ ]:
#@title Set-up  -  run this cell first { display-mode: "form" }
#@markdown Press the play button once. Nothing is downloaded and nothing
#@markdown leaves this browser tab. Takes a few seconds.
import sys, types, textwrap

_tk = types.ModuleType("toolkit"); _tk.__dict__["__name__"] = "toolkit"
exec(compile("\"\"\"PHS 312 activity toolkit - assessment engine and simulation helpers.\n\nEverything runs in the browser session. Nothing is transmitted anywhere:\nno network calls, no file writes, no analytics. Scores exist only in the\nstudent's own Colab session until they choose to report them in Canvas.\n\"\"\"\nimport textwrap, hashlib, json\nfrom datetime import datetime\n\nW = 86\ndef _wrap(s, indent=\"\"):\n    out = []\n    for para in str(s).split(\"\\n\"):\n        out.extend(textwrap.wrap(para, W, initial_indent=indent,\n                                 subsequent_indent=indent) or [\"\"])\n    return \"\\n\".join(out)\n\ndef _rule(ch=\"-\"):\n    return ch * W\n\n\nclass Item:\n    \"\"\"One multiple-choice item with per-distractor feedback.\"\"\"\n    def __init__(self, qid, stem, options, key, why, bloom=\"\", silo=\"\",\n                 section=\"concept\", topic=\"\"):\n        if key not in options:\n            raise ValueError(f\"{qid}: key {key!r} not among options {list(options)}\")\n        missing = [k for k in options if k != key and k not in why]\n        if missing:\n            raise ValueError(f\"{qid}: no feedback written for distractor(s) {missing}\")\n        self.qid, self.stem, self.options = qid, stem, options\n        self.key, self.why = key, why\n        self.bloom, self.silo, self.section = bloom, silo, section\n        self.topic = topic\n\n\nclass Gradebook:\n    \"\"\"Tracks responses and reports performance. Local to this session.\"\"\"\n\n    SECTIONS = (\"precheck\", \"concept\", \"post\")\n    LABELS = {\"precheck\": \"Prior-knowledge check\",\n              \"concept\": \"Concept checks during the activity\",\n              \"post\": \"End-of-activity assessment\"}\n\n    def __init__(self, lecture, title):\n        self.lecture, self.title = lecture, title\n        self.items, self.responses = {}, {}\n        self.started = datetime.now()\n\n    # ---------- authoring ----------\n    def add(self, item):\n        if item.qid in self.items:\n            raise ValueError(f\"duplicate qid {item.qid}\")\n        self.items[item.qid] = item\n\n    def add_many(self, items):\n        for it in items:\n            self.add(it)\n\n    # ---------- student interaction ----------\n    def ask(self, qid):\n        \"\"\"Print the stem and options for an item.\"\"\"\n        it = self.items[qid]\n        print(_rule(\"=\"))\n        print(_wrap(f\"{qid}.  {it.stem}\"))\n        print()\n        for letter, text in it.options.items():\n            print(_wrap(f\"{letter}) {text}\", indent=\"    \"))\n        print(_rule(\"=\"))\n\n    def answer(self, qid, choice):\n        \"\"\"Record a response and give feedback appropriate to the section.\"\"\"\n        if qid not in self.items:\n            print(f\"[!] Unknown question id {qid!r}. Check the notebook.\")\n            return\n        it = self.items[qid]\n        choice = (choice or \"\").strip().upper()\n        if choice in (\"\", \"-\", \"SELECT\", \"NONE\"):\n            print(\"Choose an option from the dropdown, then press the play button again.\")\n            return\n        if choice not in it.options:\n            print(f\"[!] {choice!r} is not one of {list(it.options)}.\")\n            return\n\n        first = qid not in self.responses\n        if first:\n            self.responses[qid] = choice      # only the first response scores\n        correct = choice == it.key\n\n        if it.section == \"precheck\":\n            # No feedback yet: revealing it now would teach the answer\n            # before the activity and destroy the pre/post comparison.\n            n = len([q for q in self.responses\n                     if self.items[q].section == \"precheck\"])\n            total = len([q for q in self.items\n                         if self.items[q].section == \"precheck\"])\n            print(f\"Recorded. ({n} of {total} answered)\")\n            if n == total:\n                print(\"\\nAll set - run the next cell to begin the activity.\")\n            return\n\n        print(_rule())\n        if correct:\n            print(f\"Correct - {choice}.\")\n            print(_wrap(it.why.get(\"_correct\", \"\"), \"\"))\n        else:\n            print(f\"Not quite. You chose {choice}.\")\n            print(_wrap(it.why[choice]))\n            print()\n            print(_wrap(f\"The defensible answer is {it.key}. \"\n                        + it.why.get(\"_correct\", \"\")))\n        if not first:\n            kept = self.responses[qid]\n            print(f\"\\n(Your first response, {kept}, is the one on record - \"\n                  f\"this retry is for your own understanding.)\")\n        print(_rule())\n\n    # ---------- reporting ----------\n    def _score(self, section):\n        qs = [q for q, it in self.items.items() if it.section == section]\n        got = sum(1 for q in qs if self.responses.get(q) == self.items[q].key)\n        return got, len(qs)\n\n    def summary(self, show_pre_post=True):\n        print(_rule(\"=\"))\n        print(f\"  {self.lecture} - {self.title}\")\n        print(f\"  Performance summary\")\n        print(_rule(\"=\"))\n        any_missing = False\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            if tot == 0:\n                continue\n            answered = len([q for q, it in self.items.items()\n                            if it.section == sec and q in self.responses])\n            if answered < tot:\n                any_missing = True\n            pct = 100 * got / tot if tot else 0\n            bar = \"#\" * int(round(pct / 5)) + \".\" * (20 - int(round(pct / 5)))\n            print(f\"  {self.LABELS[sec]:<38} {got:>2}/{tot:<3} {pct:5.1f}%  [{bar}]\")\n            if answered < tot:\n                print(f\"      ({tot - answered} unanswered)\")\n        print(_rule())\n\n        if show_pre_post:\n            pre_g, pre_t = self._score(\"precheck\")\n            post_g, post_t = self._score(\"post\")\n            if pre_t and post_t:\n                pre = 100 * pre_g / pre_t\n                post = 100 * post_g / post_t\n                print(f\"  Before the activity: {pre:5.1f}%     \"\n                      f\"After the activity: {post:5.1f}%\")\n                if post >= pre:\n                    if pre < 100:\n                        g = (post - pre) / (100 - pre)\n                        print(f\"  Normalised learning gain  g = {g:.2f}\"\n                              f\"   {self._gain_note(g)}\")\n                    else:\n                        print(\"  You came in at ceiling on the prior check.\")\n                else:\n                    print(\"  Your post score is below your pre score - look back at the\")\n                    print(\"  concept checks you missed before you leave.\")\n                print(_rule())\n\n        self._missed()\n        if any_missing:\n            print(\"\\n  Some questions are unanswered. Scroll back and complete them\")\n            print(\"  before copying your report code.\")\n        return self\n\n    @staticmethod\n    def _gain_note(g):\n        if g >= 0.7: return \"(high)\"\n        if g >= 0.3: return \"(medium)\"\n        return \"(low - review before the exam)\"\n\n    def _missed(self):\n        missed = [(q, it) for q, it in self.items.items()\n                  if it.section != \"precheck\"\n                  and q in self.responses and self.responses[q] != it.key]\n        if not missed:\n            print(\"  Nothing missed in the graded sections.\")\n            return\n        print(\"  Worth reviewing before the exam:\")\n        for q, it in missed:\n            tag = f\"[{it.bloom}\" + (f\" | SILO {it.silo}\" if it.silo else \"\") + \"]\"\n            print(_wrap(f\"- {q} {tag}: {it.stem}\", indent=\"    \"))\n\n    def report_code(self, student_name=\"\"):\n        \"\"\"A short code the student pastes into Canvas.\n\n        This is a participation signal, not a secure exam score: it is\n        generated in the browser and a determined student could edit it.\n        Weight it accordingly.\n        \"\"\"\n        tags = {\"precheck\": \"PRE\", \"concept\": \"CON\", \"post\": \"POST\"}\n        parts = []\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            parts.append(f\"{tags[sec]}{got}/{tot}\" if tot else f\"{tags[sec]}-\")\n        body = f\"{self.lecture}|\" + \",\".join(parts)\n        digest = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper()\n        code = f\"{body}|{digest}\"\n        print(_rule(\"=\"))\n        print(\"  COPY THIS LINE INTO THE CANVAS ASSIGNMENT\")\n        print(_rule(\"=\"))\n        print(f\"  {code}\")\n        print(_rule(\"=\"))\n        print(_wrap(\"Nothing was sent anywhere. This code exists only in your \"\n                    \"browser until you paste it into Canvas yourself.\"))\n        return code\n\n\ndef verify_code(code):\n    \"\"\"Instructor-side check that a pasted code has not been casually edited.\"\"\"\n    try:\n        body, digest = code.strip().rsplit(\"|\", 1)\n        ok = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper() == digest.upper()\n        return ok, body\n    except Exception:\n        return False, \"\"\n\n\n    # ---------- diagnostic review (used by the L28 review session) ----------\ndef _topic_table(gb, guide=None, section=None):\n    \"\"\"Score by topic and say what to revisit. Attached to Gradebook below.\n\n    `section` limits the table to one section, so the mid-session diagnostic\n    does not count questions the student has not reached yet.\n    \"\"\"\n    rows = {}\n    for qid, it in gb.items.items():\n        if not it.topic:\n            continue\n        if section and it.section != section:\n            continue\n        got = gb.responses.get(qid)\n        r = rows.setdefault(it.topic, [0, 0, 0])\n        r[1] += 1\n        if got is not None:\n            r[2] += 1\n            if got == it.key:\n                r[0] += 1\n    if not rows:\n        print(\"No topic-tagged questions have been answered yet.\")\n        return {}\n    print(_rule(\"=\"))\n    print(\"  Where you stand, by lecture\")\n    print(_rule(\"=\"))\n    print(f\"  {'lecture':<9}{'answered':>10}{'correct':>9}{'':>4}\")\n    weak = []\n    for t in sorted(rows):\n        ok, total, ans = rows[t]\n        pct = (100.0 * ok / ans) if ans else 0.0\n        bar = \"#\" * int(round(pct / 10)) + \".\" * (10 - int(round(pct / 10)))\n        flag = \"\"\n        if ans == 0:\n            bar, flag = \"-\" * 10, \"  not attempted\"\n        elif pct < 67:\n            flag = \"  <- revisit\"\n            weak.append(t)\n        print(f\"  {t:<9}{ans:>5} of {total:<4}{ok:>5}   {bar}{flag}\")\n    print()\n    if weak:\n        print(\"  Before the exam, go back to:\")\n        for t in weak:\n            where = (guide or {}).get(t, \"\")\n            print(f\"    {t}  {where}\" if where else f\"    {t}\")\n    else:\n        done = sum(r[2] for r in rows.values())\n        total = sum(r[1] for r in rows.values())\n        if done < total:\n            print(f\"  Nothing below two thirds so far, with {total - done} \"\n                  f\"question(s) still unanswered.\")\n        else:\n            print(\"  No lecture is below two thirds. Review your missed \"\n                  \"questions above,\\n  then re-run the activities you found \"\n                  \"slowest rather than re-reading.\")\n    print(_rule(\"=\"))\n    return {t: dict(correct=v[0], answered=v[2], total=v[1]) for t, v in rows.items()}\n\n\nGradebook.topic_report = _topic_table\n", "toolkit.py", "exec"), _tk.__dict__)
sys.modules["toolkit"] = _tk
Item, Gradebook, verify_code = _tk.Item, _tk.Gradebook, _tk.verify_code

_sb = types.ModuleType("swissbio"); _sb.__dict__["__name__"] = "swissbio"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"Curated bioisosteric replacement data for PHS 312 Lecture 31.\n\nSOURCE\n    SwissBioisostere, https://www.swissbioisostere.ch\n    Molecular Modelling Group, SIB Swiss Institute of Bioinformatics and\n    the University of Lausanne. Underlying data derived from ChEMBL 36.\n\nLICENCE\n    CC-BY 4.0  https://creativecommons.org/licenses/by/4.0/\n    Redistribution and adaptation permitted with attribution.\n\nCHANGES MADE (required by CC-BY)\n    Retrieved 20 October 2026 through the SwissBioisostere web interface.\n    Two single-attachment-point (\"side chain\") fragment queries were run:\n    carboxylic acid [*:1]C(=O)O  and primary amide [*:1]C(=O)N.\n    The full result tables (7,714 and 6,955 replacements) were reduced to the\n    highest-frequency entries shown below. Column names were shortened and\n    values were not otherwise altered. No structures were recomputed.\n\nCITE\n    Cuozzo A., Daina A., Perez M.A.S., Michielin O., Zoete V.\n    SwissBioisostere: molecular replacements for drug discovery guided by\n    updated structural, bioactivity and physico-chemical data accessible\n    through a reshaped Web interface. Nucleic Acids Research, 2021.\n    Wirth M., Zoete V., Michielin O., Sauer W.H.B. SwissBioisostere: a\n    database of molecular replacements for ligand design. Nucleic Acids\n    Research 2013, 41(D1), D1137-D1143. doi:10.1093/nar/gks1059\n\nCOLUMNS\n    smiles  replacement fragment, [*:1] marks the attachment point\n    freq    number of matched molecular pairs in which this replacement occurs\n    rank    rank by frequency within that query\n    better / similar / worse\n            how the replacement changed measured activity across those pairs\n    dlogp, dtpsa, dmw\n            mean change in calculated logP, topological PSA and molecular\n            weight on making the replacement\n\"\"\"\n\nFIELDS = (\"smiles\", \"freq\", \"rank\", \"better\", \"similar\", \"worse\",\n          \"dlogp\", \"dtpsa\", \"dmw\")\n\n# common names for the fragments students meet in Patrick 7e 13.3.7\nNAMES = {\n    \"COC(=O)[*:1]\": \"methyl ester\",\n    \"NC(=O)[*:1]\": \"primary amide\",\n    \"CCOC(=O)[*:1]\": \"ethyl ester\",\n    \"O[*:1]\": \"hydroxyl\",\n    \"C[*:1]\": \"methyl\",\n    \"OC[*:1]\": \"hydroxymethyl\",\n    \"O=C(O)C[*:1]\": \"acetic acid (homologated)\",\n    \"N#C[*:1]\": \"nitrile\",\n    \"CNC(=O)[*:1]\": \"N-methyl amide\",\n    \"CO[*:1]\": \"methoxy\",\n    \"N[*:1]\": \"amine\",\n    \"CN(C)C(=O)[*:1]\": \"N,N-dimethyl amide\",\n    \"O=C(NO)[*:1]\": \"hydroxamic acid\",\n    \"c1ccc([*:1])cc1\": \"phenyl\",\n    \"F[*:1]\": \"fluorine\",\n    \"Cl[*:1]\": \"chlorine\",\n    \"n1n[nH]c([*:1])n1\": \"tetrazol-5-yl\",\n    \"FC(F)(F)[*:1]\": \"trifluoromethyl\",\n    \"n1nc([*:1])n[nH]1\": \"tetrazol-5-yl (tautomer)\",\n    \"O=[N+]([O-])[*:1]\": \"nitro\",\n    \"NS(=O)(=O)[*:1]\": \"sulfonamide\",\n    \"CS(=O)(=O)NC(=O)[*:1]\": \"acylsulfonamide\",\n    \"O=C(O)CC[*:1]\": \"propanoic acid (homologated)\",\n    \"O=C(NCCO)[*:1]\": \"N-(2-hydroxyethyl) amide\",\n    \"CC(=O)N[*:1]\": \"acetamido\",\n    \"Br[*:1]\": \"bromine\",\n    \"CS(=O)(=O)[*:1]\": \"methylsulfonyl\",\n    \"O=C(N1CCOCC1)[*:1]\": \"morpholine amide\",\n    \"CC(=O)[*:1]\": \"acetyl\",\n    \"CN(C)[*:1]\": \"dimethylamino\",\n    \"O=C(O)[*:1]\": \"carboxylic acid\",\n    \"CCNC(=O)[*:1]\": \"N-ethyl amide\",\n    \"O=C(NC1CC1)[*:1]\": \"N-cyclopropyl amide\",\n    \"CS(=O)(=O)N[*:1]\": \"methanesulfonamido\",\n}\n\n# ---- carboxylic acid: what replaces [*:1]C(=O)O  (7,714 replacements total)\nACID_TOTAL = 7714\nACID = [\n (\"COC(=O)[*:1]\", 4530, 1, 1238, 2353, 939, 0.09, -11.00, 14.03),\n (\"NC(=O)[*:1]\", 2781, 2, 1000, 1334, 447, -0.60, 5.79, -0.99),\n (\"CCOC(=O)[*:1]\", 2449, 3, 703, 1024, 722, 0.48, -11.00, 28.06),\n (\"O[*:1]\", 1626, 4, 604, 793, 229, 0.01, -17.07, -28.01),\n (\"C[*:1]\", 1478, 5, 620, 616, 242, 0.61, -37.30, -29.98),\n (\"OC[*:1]\", 1144, 6, 395, 546, 203, -0.20, -17.07, -13.98),\n (\"O=C(O)C[*:1]\", 956, 7, 253, 530, 173, -0.08, 0.00, 14.02),\n (\"N#C[*:1]\", 877, 8, 346, 353, 178, 0.17, -13.51, -19.00),\n (\"CNC(=O)[*:1]\", 818, 9, 298, 412, 108, -0.34, -8.20, 13.04),\n (\"CO[*:1]\", 788, 10, 331, 334, 123, 0.31, -28.07, -13.98),\n (\"N[*:1]\", 721, 11, 201, 368, 152, -0.51, -11.28, -29.00),\n (\"CN(C)C(=O)[*:1]\", 692, 12, 230, 331, 131, 0.00, -16.99, 27.07),\n (\"O=C(NO)[*:1]\", 654, 13, 440, 157, 57, -0.58, 12.03, 15.02),\n (\"c1ccc([*:1])cc1\", 635, 14, 271, 262, 102, 1.96, -37.30, 32.09),\n (\"F[*:1]\", 625, 15, 265, 259, 101, 0.44, -37.30, -26.02),\n (\"Cl[*:1]\", 604, 16, 276, 240, 88, 0.95, -37.30, -9.57),\n (\"n1n[nH]c([*:1])n1\", 545, 17, 105, 325, 115, -0.52, 17.16, 24.03),\n (\"FC(F)(F)[*:1]\", 503, 18, 214, 206, 83, 1.32, -37.30, 23.98),\n (\"n1nc([*:1])n[nH]1\", 484, 19, 135, 256, 93, -0.51, 17.16, 24.03),\n (\"O=[N+]([O-])[*:1]\", 384, 20, 166, 167, 51, 0.21, 5.84, 0.99),\n (\"NS(=O)(=O)[*:1]\", 382, 21, 157, 168, 57, -1.05, 22.86, 35.07),\n (\"CS(=O)(=O)NC(=O)[*:1]\", 369, 22, 36, 226, 107, -1.01, 25.94, 77.11),\n (\"O=C(O)CC[*:1]\", 348, 23, 112, 191, 45, 0.32, 0.00, 28.06),\n (\"O=C(NCCO)[*:1]\", 341, 24, 148, 142, 51, -0.98, 12.03, 43.07),\n (\"CC(=O)N[*:1]\", 331, 25, 111, 160, 60, 0.26, -8.20, 13.04),\n (\"Br[*:1]\", 330, 26, 153, 125, 52, 1.06, -37.30, 34.89),\n (\"CS(=O)(=O)[*:1]\", 327, 27, 132, 141, 54, -0.43, -3.16, 34.08),\n (\"O=C(N1CCOCC1)[*:1]\", 315, 28, 93, 151, 71, -0.23, -7.76, 69.10),\n (\"CC(=O)[*:1]\", 296, 29, 120, 118, 58, 0.50, -20.23, -1.97),\n (\"CN(C)[*:1]\", 283, 30, 93, 116, 74, 0.08, -34.06, -0.94),\n]\n\n# ---- primary amide: what replaces [*:1]C(=O)N  (6,955 replacements total)\nAMIDE_TOTAL = 6955\nAMIDE = [\n (\"O=C(O)[*:1]\", 2781, 1, 447, 1334, 1000, 0.60, -5.79, 0.99),\n (\"N#C[*:1]\", 2384, 2, 586, 1189, 609, 1.04, -19.30, -18.01),\n (\"CNC(=O)[*:1]\", 2202, 3, 192, 1591, 419, 0.26, -13.99, 14.03),\n (\"O[*:1]\", 1622, 4, 398, 1003, 221, 0.11, -22.86, -27.02),\n (\"COC(=O)[*:1]\", 1269, 5, 259, 699, 311, 0.68, -16.79, 15.02),\n (\"C[*:1]\", 1238, 6, 361, 668, 209, 1.21, -43.09, -28.99),\n (\"CN(C)C(=O)[*:1]\", 1112, 7, 128, 590, 394, 0.61, -22.78, 28.06),\n (\"OC[*:1]\", 1060, 8, 233, 651, 176, 0.51, -22.86, -13.00),\n (\"CO[*:1]\", 1003, 9, 235, 554, 214, 0.91, -33.86, -12.99),\n (\"F[*:1]\", 720, 10, 195, 360, 165, 1.04, -43.09, -25.04),\n (\"Cl[*:1]\", 669, 11, 262, 281, 126, 1.55, -43.09, -8.58),\n (\"N[*:1]\", 646, 12, 151, 332, 163, 0.08, -17.07, -28.01),\n (\"FC(F)(F)[*:1]\", 635, 13, 194, 272, 169, 1.92, -43.09, 24.97),\n (\"CCOC(=O)[*:1]\", 612, 14, 146, 344, 122, 1.08, -16.79, 29.03),\n (\"CS(=O)(=O)[*:1]\", 599, 15, 90, 403, 106, 0.17, -8.95, 35.06),\n (\"CC(=O)N[*:1]\", 546, 16, 80, 349, 117, 0.26, -13.99, 14.03),\n (\"O=C(NCCO)[*:1]\", 532, 17, 42, 385, 105, -0.38, 6.24, 44.05),\n (\"c1ccc([*:1])cc1\", 476, 18, 128, 246, 102, 2.37, -43.09, 33.07),\n (\"O=C(N1CCOCC1)[*:1]\", 443, 19, 45, 250, 148, 0.37, -13.55, 70.09),\n (\"CC(=O)[*:1]\", 441, 20, 112, 235, 94, 1.10, -26.02, -0.98),\n (\"CCNC(=O)[*:1]\", 438, 21, 65, 269, 104, 0.65, -13.99, 28.06),\n (\"O=C(NC1CC1)[*:1]\", 431, 22, 94, 246, 91, 0.80, -13.99, 40.06),\n (\"NS(=O)(=O)[*:1]\", 382, 23, 72, 232, 78, -0.45, 17.07, 36.06),\n (\"CS(=O)(=O)N[*:1]\", 359, 24, 58, 225, 76, 0.27, 3.08, 50.08),\n (\"Br[*:1]\", 339, 25, 137, 146, 56, 1.66, -43.09, 35.88),\n]\n\nQUERIES = {\n    \"carboxylic acid\": dict(query=\"[*:1]C(=O)O\", total=ACID_TOTAL, rows=ACID),\n    \"primary amide\":   dict(query=\"[*:1]C(=O)N\", total=AMIDE_TOTAL, rows=AMIDE),\n}\n\n\ndef table(which=\"carboxylic acid\"):\n    q = QUERIES[which]\n    return [dict(zip(FIELDS, r), name=NAMES.get(r[0], r[0])) for r in q[\"rows\"]]\n\n\ndef success_rate(row):\n    \"\"\"Fraction of matched pairs where activity did not get worse.\"\"\"\n    n = row[\"better\"] + row[\"similar\"] + row[\"worse\"]\n    return (row[\"better\"] + row[\"similar\"]) / n if n else 0.0\n", "swissbio.py", "exec"), _sb.__dict__)
sys.modules["swissbio"] = _sb
swissbio = _sb


_sm = types.ModuleType("sims"); _sm.__dict__["__name__"] = "sims"
exec(compile("\"\"\"PHS 312 simulations. NumPy + matplotlib only - no install, no network.\n\nEvery function returns the numbers it plotted so the notebook can quiz on them.\n\"\"\"\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nPRIMARY, SECOND, MUTED = \"#0B6E8F\", \"#C2410C\", \"#6B7280\"\n\n\ndef _tw(text, width=76, indent=\"\"):\n    import textwrap\n    return \"\\n\".join(textwrap.fill(p, width, initial_indent=indent,\n                                   subsequent_indent=indent)\n                     for p in text.split(\"\\n\"))\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\n# ---------------------------------------------------------------- L22\ndef force_field_terms(bond_k=300., bond_r0=1.52, ang_k=60., ang_t0=109.5,\n                      tor_V=14., vdw_eps=0.3, vdw_sigma=3.6, q1q2=-0.3):\n    \"\"\"Each force-field term plotted over its own coordinate.\"\"\"\n    f, axes = plt.subplots(1, 4, figsize=(15, 3.4))\n    r = np.linspace(1.1, 2.1, 200); Es = bond_k * (r - bond_r0) ** 2\n    t = np.linspace(80, 140, 200);  Eb = ang_k * np.radians(t - ang_t0) ** 2\n    d = np.linspace(-180, 180, 361); Et = tor_V / 2 * (1 + np.cos(np.radians(3 * d)))\n    x = np.linspace(3.0, 8.0, 300)\n    Ev = 4 * vdw_eps * ((vdw_sigma / x) ** 12 - (vdw_sigma / x) ** 6)\n    Ec = 332. * q1q2 / x\n    for ax, (xx, yy, xl, ti) in zip(axes, [\n            (r, Es, \"bond length (A)\", \"stretch\"),\n            (t, Eb, \"bond angle (deg)\", \"bend\"),\n            (d, Et, \"torsion (deg)\", \"torsion\"),\n            (x, Ev + Ec, \"separation (A)\", \"vdW + electrostatic\")]):\n        ax.plot(xx, yy, color=PRIMARY, lw=2)\n        ax.set_title(ti, fontsize=10); ax.set_xlabel(xl, fontsize=8)\n        ax.grid(alpha=.25); ax.axhline(0, color=MUTED, lw=.8)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    axes[0].set_ylabel(\"energy (kcal/mol)\", fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(stretch_min=float(r[Es.argmin()]), torsion_barrier=float(Et.max()),\n                vdw_min_sep=float(x[(Ev + Ec).argmin()]),\n                vdw_min_E=float((Ev + Ec).min()))\n\n\n# ---------------------------------------------------------------- L23\ndef torsion_profile(V3=14.0, gauche_strain=3.8, step=10, start_angle=100.):\n    \"\"\"Butane-like relaxed torsion profile, plus where minimisation stops.\"\"\"\n    d = np.arange(-180, 181, step)\n    E = V3 / 2 * (1 + np.cos(np.radians(3 * d)))\n    E += gauche_strain * np.exp(-((d - 60) / 32.) ** 2)\n    E += gauche_strain * np.exp(-((d + 60) / 32.) ** 2)\n    E += 2.2 * gauche_strain * np.exp(-(d / 30.) ** 2)\n    E = E - E.min()\n    # steepest descent from start_angle on this surface\n    fine = np.arange(-180, 180.01, 0.5)\n    Ef = np.interp(fine, d, E)\n    i = int(np.argmin(np.abs(fine - start_angle)))\n    while True:\n        lo, hi = max(i - 1, 0), min(i + 1, len(fine) - 1)\n        j = lo if Ef[lo] < Ef[i] else (hi if Ef[hi] < Ef[i] else i)\n        if j == i: break\n        i = j\n    f, ax = _fig()\n    ax.plot(d, E, \"-o\", ms=3.5, color=PRIMARY, label=\"relaxed scan\")\n    ax.axvline(start_angle, color=MUTED, ls=\":\", label=f\"start {start_angle:.0f}\")\n    ax.plot([fine[i]], [Ef[i]], \"v\", ms=12, color=SECOND,\n            label=f\"minimisation stops at {fine[i]:.0f} deg\")\n    ax.set_xlabel(\"C1-C2-C3-C4 dihedral (degrees)\")\n    ax.set_ylabel(\"relative steric energy (kJ/mol)\")\n    ax.set_xticks(range(-180, 181, 60)); ax.legend(fontsize=8)\n    plt.tight_layout(); plt.show()\n    mins = [float(d[k]) for k in range(1, len(d) - 1)\n            if E[k] < E[k - 1] and E[k] < E[k + 1]]\n    return dict(global_min=float(d[E.argmin()]), local_minima=mins,\n                barrier_at_0=float(E[d == 0][0]), stops_at=float(fine[i]))\n\n\ndef search_cost(n_rot=7, step=30):\n    n = int(round(360 / step)) ** n_rot\n    secs = n * 0.05\n    return dict(conformations=n, hours=secs / 3600,\n                years=secs / 3600 / 24 / 365)\n\n\n# ---------------------------------------------------------------- L24\ndef anneal(T=300., n_steps=4000, seed=0):\n    \"\"\"Metropolis walk on a double-well surface: does it escape the local min?\n\n    Global minima sit at +/-180 deg (0 kJ/mol); a local minimum sits at\n    +60 deg (4 kJ/mol) behind a ~26 kJ/mol barrier. At low T the walker is\n    trapped; raising T lets it cross.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    PLATEAU, DEEP, SHALLOW, W = 30.0, 30.0, 26.0, 26.0\n\n    def E(a):\n        a = np.asarray(a, float)\n        return (PLATEAU\n                - DEEP * np.exp(-((a - 180) / W) ** 2)\n                - DEEP * np.exp(-((a + 180) / W) ** 2)\n                - SHALLOW * np.exp(-((a - 60) / W) ** 2))\n\n    pos = 60.0\n    visited = [pos]\n    kT = max(T, 1.) * 0.0083145                      # kJ/mol per K\n    for _ in range(n_steps):\n        trial = ((pos + rng.normal(0, 20) + 180) % 360) - 180\n        dE = float(E(trial) - E(pos))\n        if dE < 0 or rng.random() < np.exp(-min(dE / kT, 700)):\n            pos = trial\n        visited.append(pos)\n    visited = np.array(visited)\n\n    second_half = visited[len(visited) // 2:]\n    near_global = float(np.mean(np.abs(np.abs(second_half) - 180) < 50) * 100)\n    ever = bool(np.any(np.abs(np.abs(visited) - 180) < 50))\n\n    x = np.linspace(-180, 180, 721)\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))\n    a1.plot(x, E(x), color=PRIMARY, lw=2)\n    a1.set_xlabel(\"dihedral (deg)\"); a1.set_ylabel(\"energy (kJ/mol)\")\n    a1.set_title(\"energy surface\")\n    a1.plot([60], [E(60)], \"o\", color=SECOND, ms=10)\n    a1.annotate(\"start (local, 4 kJ/mol)\", (60, E(60) + 1.5), fontsize=8, ha=\"center\")\n    a1.annotate(\"global (0 kJ/mol)\", (180, E(180) + 1.5), fontsize=8, ha=\"right\")\n    a2.hist(visited, bins=72, color=PRIMARY, alpha=.85)\n    a2.set_xlabel(\"dihedral visited (deg)\"); a2.set_ylabel(\"count\")\n    a2.set_title(f\"T = {T:.0f} K   -   \"\n                 + (\"escaped the local minimum\" if near_global > 15\n                    else \"stayed trapped\"))\n    for ax in (a1, a2):\n        ax.grid(alpha=.25)\n        for sp in (\"top\", \"right\"): ax.spines[sp].set_visible(False)\n    plt.tight_layout(); plt.show()\n    return dict(escaped=near_global > 15, pct_near_global=round(near_global, 1),\n                ever_reached_global=ever, final=round(float(visited[-1]), 1))\n\n\ndef rmsd_overlay(dx=0.0, dy=0.0, rot_deg=0.0):\n    \"\"\"Rigid-body move of a probe onto a template; reports RMSD.\"\"\"\n    tmpl = np.array([[0., 0.], [1.4, 0.3], [2.6, -0.4], [3.9, 0.2], [1.2, 1.7]])\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    probe = (tmpl - tmpl.mean(0)) @ R.T + tmpl.mean(0) + np.array([dx, dy])\n    rmsd = float(np.sqrt(((probe - tmpl) ** 2).sum(1).mean()))\n    f, ax = _fig(6.4, 4.6)\n    ax.plot(*tmpl.T, \"o-\", color=PRIMARY, ms=11, lw=2, label=\"template (rigid active)\")\n    ax.plot(*probe.T, \"s--\", color=SECOND, ms=10, lw=2, label=\"your analogue\")\n    for p, q in zip(tmpl, probe):\n        ax.plot([p[0], q[0]], [p[1], q[1]], color=MUTED, lw=.9, ls=\":\")\n    ax.set_title(f\"RMSD = {rmsd:.2f} A\"); ax.legend(fontsize=8); ax.set_aspect(\"equal\")\n    plt.tight_layout(); plt.show()\n    return dict(rmsd=rmsd, good=rmsd < 0.5)\n\n\n# ---------------------------------------------------------------- L25\n# A snug pocket, 12.0 x 8.0 A, holding a ligand that spans 5.0 A, with a\n# gap in the wall where a polar residue sits. Orientation matters: turned\n# across the pocket the ligand cannot fit without clashing.\n_A, _B = 6.0, 4.1\n_TH = np.linspace(0, 2 * np.pi, 25)[:-1]\n_TH = np.array([t for t in _TH\n                if abs(np.degrees(t) - 90) > 16])          # gap for the residue\nSITE = np.c_[_A * np.cos(_TH), _B * np.sin(_TH)]           # cavity wall atoms\nHBOND_POINT = np.array([0.0, 3.7])                         # polar partner\nLIGAND = np.array([[-2.5, -0.3], [-0.8, 0.3], [0.9, 0.2],\n                   [2.5, -0.4], [0.0, 1.3]])               # last atom is polar\nCLASH_AT, CONTACT_AT, HBOND_AT = 3.0, 3.9, 2.7\n\n\ndef dock_pose(x=0.0, y=0.0, rot_deg=0.0, show=True):\n    \"\"\"Place a rigid 5-atom ligand in a 2-D pocket it very nearly fills.\n\n    Score = -(shape contacts + H-bond) + steric clash penalty, so a MORE\n    NEGATIVE score is a better pose. A pose outside the pocket earns nothing\n    and tends to zero; a pose driven into the wall goes sharply positive.\n    \"\"\"\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    pose = LIGAND @ R.T + np.array([x, y])\n\n    d = np.linalg.norm(pose[:, None, :] - SITE[None, :, :], axis=2)\n    nearest = d.min(1)\n    bad = nearest < CLASH_AT\n    clash = float(np.sum(np.clip(CLASH_AT - nearest, 0, None) ** 2) * 12)\n    contact = float(np.sum(np.exp(-((nearest - CONTACT_AT) / 1.25) ** 2)) * 2.6)\n    hb_dist = float(np.linalg.norm(pose[4] - HBOND_POINT))\n    hbond = float(6.0 * np.exp(-((hb_dist - HBOND_AT) / 0.95) ** 2))\n    score = -(contact + hbond) + clash\n\n    if show:\n        f, ax = _fig(7.0, 5.4)\n        _full = np.linspace(0, 2 * np.pi, 160)\n        ax.fill(_A * np.cos(_full), _B * np.sin(_full), color=\"#DCE9EF\",\n                zorder=0)\n        ax.plot(np.append(SITE[:, 0], SITE[0, 0]),\n                np.append(SITE[:, 1], SITE[0, 1]), \"o\", color=PRIMARY,\n                ms=15, label=\"binding-site wall\", zorder=1)\n        ax.plot(*HBOND_POINT, \"*\", ms=22, color=\"#B45309\",\n                label=\"polar residue\", zorder=3)\n        chain = pose[:4]\n        ax.plot(chain[:, 0], chain[:, 1], \"-\", color=SECOND, lw=2.6,\n                alpha=.8, zorder=4)\n        ax.plot([chain[1, 0], pose[4, 0]], [chain[1, 1], pose[4, 1]], \"-\",\n                color=SECOND, lw=2.6, alpha=.8, zorder=4)\n        ok = ~bad\n        if ok.any():\n            ax.plot(pose[ok][:, 0], pose[ok][:, 1], \"o\", ms=14,\n                    color=SECOND, label=\"ligand atom\", zorder=5)\n        if bad.any():\n            ax.plot(pose[bad][:, 0], pose[bad][:, 1], \"X\", ms=19,\n                    color=\"#DC2626\", label=\"steric clash\", zorder=6)\n        ax.set_aspect(\"equal\")\n        ax.set_xlim(-9.5, 9.5); ax.set_ylim(-7, 7)\n        ax.legend(fontsize=8, loc=\"lower left\")\n        ax.set_title(f\"score {score:+.1f}    \"\n                     f\"contacts {contact:.1f}   H-bond {hbond:.1f}   \"\n                     f\"clash {clash:.1f}\")\n        plt.tight_layout(); plt.show()\n    return dict(score=round(score, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int(bad.sum()))\n\n\n# ---------------------------------------------------------------- L26\ndef score_vs_truth(cutoff=-7.0, seed=3):\n    \"\"\"Docking scores against crystallographic truth: the ranking problem.\"\"\"\n    rng = np.random.default_rng(seed)\n    n = 60\n    active = rng.random(n) < 0.25\n    rmsd = np.where(active, rng.gamma(2, .9, n), rng.gamma(5, 1.1, n))\n    score = -4.5 - 3.2 * active - rng.normal(0, 1.5, n)\n    picked = score <= cutoff\n    tp = int(np.sum(picked & active)); fp = int(np.sum(picked & ~active))\n    tot_act = int(active.sum())\n    hit = 100 * tp / max(picked.sum(), 1)\n    base = 100 * tot_act / n\n    f, ax = _fig(8.6, 4.6)\n    ax.scatter(score[~active], rmsd[~active], s=46, color=MUTED,\n               label=\"decoy\", alpha=.75)\n    ax.scatter(score[active], rmsd[active], s=56, color=PRIMARY,\n               label=\"true active\", alpha=.9)\n    ax.axvline(cutoff, color=SECOND, lw=2, ls=\"--\",\n               label=f\"your cutoff {cutoff:+.1f}\")\n    ax.axhline(2.0, color=\"#B45309\", lw=1.2, ls=\":\",\n               label=\"2 A - pose counts as correct\")\n    ax.set_xlabel(\"docking score (more negative = better)\")\n    ax.set_ylabel(\"RMSD to crystal pose (A)\")\n    ax.legend(fontsize=8)\n    ax.set_title(f\"picked {int(picked.sum())} compounds: {tp} actives, {fp} decoys\"\n                 f\"   |   hit rate {hit:.0f}% vs {base:.0f}% at random\")\n    plt.tight_layout(); plt.show()\n    return dict(picked=int(picked.sum()), true_actives_found=tp, decoys=fp,\n                total_actives=tot_act, hit_rate=round(hit, 1),\n                random_rate=round(base, 1),\n                enrichment=round(hit / base, 2) if base else 0.0)\n\n\n# ---------------------------------------------------------------- L27\ndef de_novo(h_bond_group=\"amidine\", near_group=\"phenyl\", far_group=\"isopropyl\",\n            linker_bonds=5, fill_site=False):\n    \"\"\"Grow and link fragments into a site with one polar and two greasy pockets.\"\"\"\n    HB = {\"amidine\": 5.0, \"tetrazole\": 4.2, \"phenol\": 3.0, \"methyl\": 0.2,\n          \"isopropyl\": 0.0}\n    GREASE = {\"phenyl\": 3.6, \"cyclohexyl\": 3.9, \"isopropyl\": 2.6,\n              \"methyl\": 1.2, \"amidine\": -1.5, \"tetrazole\": -1.8, \"phenol\": 0.6}\n    SYNTH = {\"amidine\": 1.0, \"tetrazole\": 1.6, \"phenol\": 0.3, \"phenyl\": 0.2,\n             \"cyclohexyl\": 0.5, \"isopropyl\": 0.1, \"methyl\": 0.0}\n    inter = HB.get(h_bond_group, 0) + GREASE.get(near_group, 0) + GREASE.get(far_group, 0)\n    ideal = 6\n    strain = 0.9 * abs(linker_bonds - ideal)\n    synth = SYNTH.get(h_bond_group, 0) + SYNTH.get(near_group, 0) + SYNTH.get(far_group, 0) \\\n        + 0.25 * max(linker_bonds - 8, 0)\n    overfill = 3.0 if fill_site else 0.0\n    total = inter - strain - synth - overfill\n    labels = [\"H-bond region\", \"near pocket\", \"far pocket\",\n              \"linker strain\", \"synthetic cost\", \"over-fill\"]\n    vals = [HB.get(h_bond_group, 0), GREASE.get(near_group, 0),\n            GREASE.get(far_group, 0), -strain, -synth, -overfill]\n    f, ax = _fig(9.2, 3.8)\n    cols = [PRIMARY if v >= 0 else SECOND for v in vals]\n    ax.barh(labels, vals, color=cols)\n    ax.axvline(0, color=\"k\", lw=.9)\n    ax.set_xlabel(\"contribution to design score\")\n    ax.set_title(f\"design score {total:+.1f}\"\n                 + (\"   (over-filled: no room left to optimise)\" if fill_site else \"\"))\n    plt.tight_layout(); plt.show()\n    return dict(score=round(total, 2), interactions=round(inter, 2),\n                strain=round(strain, 2), synthetic_cost=round(synth, 2),\n                overfill_penalty=overfill)\n\n\n# ---------------------------------------------------------------- L29 / L30\nCOUMARIN = dict(\n    X=[\"H\", \"Me\", \"OH\", \"OMe\", \"OCF3\", \"NH2\", \"F\", \"Cl\", \"CF3\", \"CN\", \"NO2\"],\n    pIC50=np.array([8.36, 8.38, 8.01, 8.44, 7.94, 7.46, 8.55, 8.48, 8.24, 7.97, 8.59]),\n    pi=np.array([0., .49, -.67, -.09, 1.02, -1.23, .14, .71, .88, -.57, -.26]),\n    Vdw=np.array([.08, 1.01, .49, 1.49, 1.60, .67, .36, 1.07, 1.11, 1.14, 1.20]),\n    F=np.array([0., -.04, .28, .25, .37, .02, .42, .40, .37, .50, .66]))\n\n_PI = {\"H\": 0., \"Me\": .56, \"Cl\": .71, \"NO2\": -.28, \"OMe\": -.02, \"NH2\": -1.23, \"F\": .14}\n_MR = {\"H\": .103, \"Me\": .565, \"Cl\": .603, \"NO2\": .736, \"OMe\": .787, \"NH2\": .542, \"F\": .092}\n_F = {\"H\": 0., \"Me\": .01, \"Cl\": .42, \"NO2\": .65, \"OMe\": .29, \"NH2\": .08, \"F\": .45}\n_R = {\"H\": 0., \"Me\": -.18, \"Cl\": -.19, \"NO2\": .13, \"OMe\": -.56, \"NH2\": -.74, \"F\": -.39}\n_TAC = [(\"tacrine\", \"H\", \"H\", 6.59), (\"I\", \"Me\", \"H\", 5.09), (\"II\", \"H\", \"Me\", 7.00),\n        (\"III\", \"Cl\", \"H\", 6.26), (\"IV\", \"H\", \"Cl\", 8.00), (\"V\", \"NO2\", \"H\", 5.52),\n        (\"VI\", \"H\", \"NO2\", 7.55), (\"VII\", \"H\", \"OMe\", 6.45), (\"VIII\", \"NH2\", \"H\", 5.42),\n        (\"IX\", \"H\", \"F\", 7.06), (\"X\", \"Cl\", \"Cl\", 6.32), (\"XI\", \"OMe\", \"OMe\", 5.29)]\n\ndef tacrine_table():\n    rows = []\n    for cid, X, Y, p in _TAC:\n        rows.append(dict(cpd=cid, X=X, Y=Y, pIC50=p, piX=_PI[X], piY=_PI[Y],\n                         MRX=_MR[X], MRY=_MR[Y], F_XY=_F[X] + _F[Y], R_XY=_R[X] + _R[Y]))\n    return rows\n\n\ndef _lstsq(X, y):\n    X = np.column_stack([np.asarray(X, float).reshape(len(y), -1), np.ones(len(y))])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    pred = X @ beta\n    ss_res = float(((y - pred) ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())\n    r2 = 1 - ss_res / ss_tot\n    n, p = len(y), X.shape[1]\n    s = float(np.sqrt(ss_res / (n - p))) if n > p else float(\"nan\")\n    return beta, pred, r2, s\n\n\ndef qsar_single(param=\"pi\", quadratic=False):\n    \"\"\"One descriptor at a time on the coumarin MAO-B series.\"\"\"\n    y = COUMARIN[\"pIC50\"]; x = COUMARIN[param]\n    cols = np.column_stack([x, x ** 2]) if quadratic else x\n    beta, pred, r2, s = _lstsq(cols, y)\n    f, ax = _fig(7.6, 4.4)\n    ax.scatter(x, y, s=70, color=PRIMARY, zorder=3)\n    for xi, yi, t in zip(x, y, COUMARIN[\"X\"]):\n        ax.annotate(t, (xi, yi), fontsize=8, xytext=(4, 4), textcoords=\"offset points\")\n    xs = np.linspace(x.min(), x.max(), 200)\n    ys = beta[0] * xs + beta[1] * xs ** 2 + beta[2] if quadratic else beta[0] * xs + beta[1]\n    ax.plot(xs, ys, color=SECOND, lw=2,\n            label=(\"quadratic\" if quadratic else \"linear\") + f\"   r = {np.sqrt(max(r2,0)):.3f}   s = {s:.3f}\")\n    ax.set_xlabel({\"pi\": \"substituent hydrophobicity  pi\",\n                   \"Vdw\": \"van der Waals volume\", \"F\": \"inductive factor  F\"}[param])\n    ax.set_ylabel(\"pIC50 (MAO-B)\"); ax.legend(fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(param=param, quadratic=quadratic,\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3))\n\n\ndef qsar_multi(use=(\"MRX\",), show=True):\n    \"\"\"Add descriptors and watch r rise while leave-one-out gets worse.\"\"\"\n    rows = tacrine_table()\n    y = np.array([r[\"pIC50\"] for r in rows])\n    X = np.column_stack([[r[c] for r in rows] for c in use])\n    beta, pred, r2, s = _lstsq(X, y)\n    loo = []\n    for i in range(len(y)):\n        m = np.ones(len(y), bool); m[i] = False\n        b, _, _, _ = _lstsq(X[m], y[m])\n        loo.append(float(np.dot(np.append(X[i], 1.0), b)))\n    loo = np.array(loo)\n    q2 = 1 - float(((y - loo) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())\n    if show:\n        f, ax = _fig(5.6, 5.2)\n        ax.scatter(pred, y, s=72, color=PRIMARY, label=\"fitted\", zorder=3)\n        ax.scatter(loo, y, s=52, color=SECOND, marker=\"s\",\n                   label=\"left-out prediction\", alpha=.85, zorder=3)\n        lim = [min(y.min(), pred.min(), loo.min()) - .4,\n               max(y.max(), pred.max(), loo.max()) + .4]\n        ax.plot(lim, lim, \"k--\", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)\n        ax.set_xlabel(\"predicted pIC50\"); ax.set_ylabel(\"observed pIC50\")\n        ax.set_title(\" + \".join(use) + f\"\\nr = {np.sqrt(max(r2,0)):.3f}    \"\n                     f\"s = {s:.3f}    q2 = {q2:.3f}\")\n        ax.legend(fontsize=8); plt.tight_layout(); plt.show()\n    return dict(descriptors=list(use), n=len(y), k=len(use),\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3),\n                q2=round(q2, 3), compounds_per_parameter=round(len(y) / len(use), 1))\n\n\n# ---------------------------------------------------------------- L31\nBIOISOSTERE = {\n    \"carboxylic acid (COOH)\": dict(pKa=4.2, logP=0.0, MW=45, HBD=1, HBA=2, note=\"parent\"),\n    \"tetrazole\":              dict(pKa=4.9, logP=0.5, MW=69, HBD=1, HBA=4, note=\"classic acid mimic\"),\n    \"acylsulfonamide\":        dict(pKa=4.5, logP=0.6, MW=121, HBD=1, HBA=4, note=\"acid mimic, bulkier\"),\n    \"2,6-difluorophenol\":     dict(pKa=7.1, logP=1.9, MW=130, HBD=1, HBA=1, note=\"lipophilic acid mimic\"),\n    \"hydroxamic acid\":        dict(pKa=9.0, logP=-0.4, MW=61, HBD=2, HBA=3, note=\"weaker acid, metal binder\"),\n    \"primary amide (CONH2)\":  dict(pKa=None, logP=-1.5, MW=44, HBD=2, HBA=2, note=\"neutral\"),\n    \"methyl ester\":           dict(pKa=None, logP=0.6, MW=59, HBD=0, HBA=2, note=\"neutral, hydrolysable\"),\n}\n\ndef bioisostere_swap(replacement=\"tetrazole\", need_acidic=True, need_permeable=True):\n    p = BIOISOSTERE[\"carboxylic acid (COOH)\"]; r = BIOISOSTERE[replacement]\n    keeps_acid = r[\"pKa\"] is not None and r[\"pKa\"] < 6.5\n    more_lipo = r[\"logP\"] > p[\"logP\"]\n    verdict = []\n    if need_acidic:\n        verdict.append((\"keeps the ionised acid at pH 7.4\", keeps_acid))\n    if need_permeable:\n        verdict.append((\"raises lipophilicity for permeability\", more_lipo))\n    verdict.append((\"molecular weight increase under 60 Da\", r[\"MW\"] - p[\"MW\"] < 60))\n    f, ax = _fig(8.8, 3.6)\n    labels = [\"pKa\", \"logP\", \"MW (Da)\", \"HBD\", \"HBA\"]\n    par = [p[\"pKa\"] if p[\"pKa\"] else 0, p[\"logP\"], p[\"MW\"] / 20, p[\"HBD\"], p[\"HBA\"]]\n    rep = [r[\"pKa\"] if r[\"pKa\"] else 0, r[\"logP\"], r[\"MW\"] / 20, r[\"HBD\"], r[\"HBA\"]]\n    i = np.arange(len(labels)); w = .38\n    ax.bar(i - w / 2, par, w, label=\"carboxylic acid (parent)\", color=MUTED)\n    ax.bar(i + w / 2, rep, w, label=replacement, color=PRIMARY)\n    ax.set_xticks(i); ax.set_xticklabels(labels)\n    ax.set_title(f\"{replacement} - {r['note']}   (MW shown /20)\")\n    ax.legend(fontsize=8); ax.axhline(0, color=\"k\", lw=.8)\n    plt.tight_layout(); plt.show()\n    for text, ok in verdict:\n        print((\"  MET      \" if ok else \"  NOT MET  \") + text)\n    return dict(replacement=replacement, pKa=r[\"pKa\"], logP=r[\"logP\"],\n                dMW=r[\"MW\"] - p[\"MW\"], requirements_met=sum(1 for _, o in verdict if o),\n                requirements_total=len(verdict))\n\n\n# ---------------------------------------------------------------- L32\ndef prodrug_kinetics(promoiety=\"ethyl ester\", esterase_activity=1.0, hours=8.0):\n    \"\"\"First-order activation: prodrug -> active drug, and patient variability.\"\"\"\n    K = {\"ethyl ester\": 0.45, \"5-indanyl ester (EWG)\": 1.60,\n         \"pivaloyl ester (hindered)\": 0.10, \"amide\": 0.02, \"no promoiety\": 0.0}\n    k = K[promoiety] * max(esterase_activity, 0.0)\n    t = np.linspace(0, hours, 400)\n    pro = np.exp(-k * t); act = 1 - pro\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))\n    a1.plot(t, pro * 100, color=MUTED, lw=2, label=\"prodrug (inactive)\")\n    a1.plot(t, act * 100, color=PRIMARY, lw=2.4, label=\"active drug\")\n    a1.set_xlabel(\"hours\"); a1.set_ylabel(\"% of dose\"); a1.legend(fontsize=8)\n    a1.set_title(f\"{promoiety}   k = {k:.2f} /h\")\n    for lab, mult, col in [(\"low enzyme (x0.3)\", .3, SECOND),\n                           (\"typical (x1)\", 1., PRIMARY),\n                           (\"high enzyme (x3)\", 3., \"#15803D\")]:\n        a2.plot(t, (1 - np.exp(-K[promoiety] * mult * t)) * 100, lw=2.2,\n                color=col, label=lab)\n    a2.set_xlabel(\"hours\"); a2.set_ylabel(\"% converted to active drug\")\n    a2.set_title(\"same prodrug, three patients\"); a2.legend(fontsize=8)\n    for ax in (a1, a2):\n        ax.grid(alpha=.25); ax.set_ylim(-3, 103)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    plt.tight_layout(); plt.show()\n    half = float(np.log(2) / k) if k > 0 else float(\"inf\")\n    at2 = float((1 - np.exp(-k * 2)) * 100)\n    spread = float(abs((1 - np.exp(-K[promoiety] * 3 * 2)) -\n                       (1 - np.exp(-K[promoiety] * .3 * 2))) * 100)\n    return dict(promoiety=promoiety, k_per_h=round(k, 3),\n                half_life_h=(round(half, 2) if np.isfinite(half) else None),\n                pct_active_at_2h=round(at2, 1),\n                patient_spread_at_2h_pct=round(spread, 1))\n\n\n# ---------------------------------------------------------------- L22 extra\ndef method_choice(atoms=20, want=\"partial charges and an electrostatic map\"):\n    \"\"\"Which level of theory answers the question at an affordable cost.\"\"\"\n    needs_electrons = want in (\"partial charges and an electrostatic map\",\n                               \"heat of formation\",\n                               \"transition state of a reaction\")\n    if not needs_electrons:\n        rec = \"Molecular mechanics\"\n        why = (\"The question is about geometry and sterics only. A force field \"\n               \"treats atoms as spheres on springs and never computes an \"\n               \"electron distribution, which is all the speed comes from.\")\n    elif atoms <= 30:\n        rec = \"Ab initio quantum mechanics\"\n        why = (\"Electron distribution is required and the molecule is small \"\n               \"enough that a first-principles calculation is affordable.\")\n    elif atoms <= 150:\n        rec = \"Semi-empirical quantum mechanics\"\n        why = (\"Electron distribution is required but the molecule is too large \"\n               \"for ab initio, so experimentally fitted parameters are used to \"\n               \"simplify the integrals.\")\n    else:\n        rec = \"Molecular mechanics, and reconsider the question\"\n        why = (\"No quantum method is affordable at this size. Either cut the \"\n               \"molecule down to the fragment that matters, or accept that \"\n               \"electronic properties are out of reach.\")\n    cost = {\"Molecular mechanics\": 1, \"Semi-empirical quantum mechanics\": 1e3,\n            \"Ab initio quantum mechanics\": 1e6}\n    f, ax = _fig(8.6, 2.9)\n    names = list(cost); vals = [cost[n] for n in names]\n    cols = [PRIMARY if n in rec else MUTED for n in names]\n    ax.barh(names, vals, color=cols)\n    ax.set_xscale(\"log\"); ax.set_xlabel(\"relative cost (log scale)\")\n    ax.set_title(f\"{atoms} atoms   -   {want}\")\n    plt.tight_layout(); plt.show()\n    print(\"RECOMMEND:\", rec); print()\n    print(_tw(why))\n    return dict(atoms=atoms, question=want, recommend=rec,\n                needs_electron_distribution=needs_electrons)\n\n\n# ---------------------------------------------------------------- L25 extra\ndef site_surface(probe_radius=1.4, show_probe=True):\n    \"\"\"van der Waals surface versus the solvent-accessible (Connolly) surface.\n\n    Two gate atoms are placed so the opening between their van der Waals\n    surfaces is exactly GAP angstroms. A spherical probe passes only if its\n    radius does not exceed GAP/2, so the narration below is derived from the\n    coordinates rather than asserted.\n    \"\"\"\n    R = 1.7\n    GAP = 1.60                       # surface-to-surface opening, angstroms\n    d = 2 * R + GAP                  # required centre-to-centre separation\n    gate = [(-d / 2, 0.0, R), (d / 2, 0.0, R)]\n    wall = [(-5.6, -1.0, R), (-5.0, -3.0, R), (-3.4, -4.4, R),\n            (-1.2, -5.0, R), (1.2, -5.0, R), (3.4, -4.4, R),\n            (5.0, -3.0, R), (5.6, -1.0, R)]\n    atoms = gate + wall\n    half_gap = GAP / 2\n    fits = probe_radius <= half_gap\n\n    gx, gy = np.meshgrid(np.linspace(-8, 8, 460), np.linspace(-7.5, 4.5, 360))\n    vdw = np.zeros_like(gx, bool); excluded = np.zeros_like(gx, bool)\n    for (cx, cy, r) in atoms:\n        dd = (gx - cx) ** 2 + (gy - cy) ** 2\n        vdw |= dd <= r ** 2\n        excluded |= dd <= (r + probe_radius) ** 2\n\n    f, ax = _fig(7.4, 6.0)\n    ax.contourf(gx, gy, excluded.astype(float), levels=[.5, 1.5],\n                colors=[\"#CBD5E1\"], alpha=.8)\n    ax.contour(gx, gy, excluded.astype(float), levels=[.5], colors=[SECOND],\n               linewidths=2.2)\n    ax.contourf(gx, gy, vdw.astype(float), levels=[.5, 1.5], colors=[MUTED],\n                alpha=.95)\n    ax.contour(gx, gy, vdw.astype(float), levels=[.5], colors=[\"#334155\"],\n               linewidths=1.3)\n    if show_probe:\n        ax.add_patch(plt.Circle((0.0, 0.0), probe_radius, fill=False,\n                                ec=PRIMARY, lw=2.4, ls=\"--\"))\n        ax.plot([0], [0], \"x\", color=PRIMARY, ms=9)\n    ax.annotate(\"\", xy=(-d / 2 + R, 0), xytext=(d / 2 - R, 0),\n                arrowprops=dict(arrowstyle=\"<->\", color=PRIMARY, lw=1.6))\n    ax.text(0, .45, f\"{GAP:.2f} A opening\", ha=\"center\", color=PRIMARY,\n            fontsize=9)\n    ax.set_aspect(\"equal\"); ax.set_xlim(-8, 8); ax.set_ylim(-7.5, 4.5)\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.set_title(f\"probe radius {probe_radius:.2f} A\\n\"\n                 \"dark grey = atoms (van der Waals)    \"\n                 \"orange outline = surface this probe can reach\")\n    plt.tight_layout(); plt.show()\n\n    print(f\"opening between the gate atoms : {GAP:.2f} A\")\n    print(f\"largest probe that passes      : {half_gap:.2f} A radius\")\n    print(f\"this probe                     : {probe_radius:.2f} A radius\")\n    print()\n    print(_tw(\n        (\"The probe fits through the opening, so the cavity behind the gate is \"\n         \"mapped as accessible and a ligand atom may be placed there.\"\n         if fits else\n         \"The probe is too large to pass, so the opening is sealed over. The \"\n         \"cavity behind it never appears on the docking surface, which is the \"\n         \"point: no ligand atom could reach it either.\")))\n    return dict(probe_radius=probe_radius, opening_angstrom=GAP,\n                max_probe_radius_that_fits=half_gap,\n                probe_passes=bool(fits),\n                excluded_volume_grid_points=int(excluded.sum()))\n\n\n# ------------------------------------------------- L31, real SwissBioisostere\ndef bioisostere_real(parent=\"carboxylic acid\", replacement=\"primary amide\",\n                     need_not_worse=0.80, max_mw_increase=60.0, show=True):\n    \"\"\"Judge a replacement against real matched-pair data.\n\n    Data: SwissBioisostere (CC-BY 4.0), ChEMBL 36. See swissbio.py for the\n    full citation and for what was changed on retrieval.\n    \"\"\"\n    import swissbio as sb\n    rows = {r[\"name\"]: r for r in sb.table(parent)}\n    if replacement not in rows:\n        print(f\"'{replacement}' is not in the curated {parent} list.\")\n        print(\"Available:\", \", \".join(sorted(rows)[:12]), \"...\")\n        return None\n    r = rows[replacement]\n    n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n    not_worse = (r[\"better\"] + r[\"similar\"]) / n\n    total = sb.QUERIES[parent][\"total\"]\n\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.6, 4.2))\n        # how activity changed across the real pairs\n        a1.barh([\"worse\", \"similar\", \"better\"],\n                [r[\"worse\"], r[\"similar\"], r[\"better\"]],\n                color=[SECOND, MUTED, \"#15803D\"])\n        a1.set_xlabel(f\"matched molecular pairs (n = {n})\")\n        a1.set_title(f\"{replacement}\\nreplacing {parent} in real compounds\")\n        for s in (\"top\", \"right\"): a1.spines[s].set_visible(False)\n\n        # where this replacement sits against the others by frequency\n        tab = sb.table(parent)\n        ranks = [x[\"rank\"] for x in tab]; freqs = [x[\"freq\"] for x in tab]\n        cols = [PRIMARY if x[\"name\"] == replacement else \"#CBD5E1\" for x in tab]\n        a2.bar(ranks, freqs, color=cols)\n        a2.set_xlabel(f\"rank by frequency (top {len(tab)} of {total:,})\")\n        a2.set_ylabel(\"matched pairs\")\n        a2.set_title(f\"this replacement is rank {r['rank']} of {total:,}\")\n        for s in (\"top\", \"right\"): a2.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n\n    print(f\"SMILES              {r['smiles']}\")\n    print(f\"frequency           {r['freq']:,} matched pairs  (rank {r['rank']} of {total:,})\")\n    print(f\"activity outcome    {r['better']} better / {r['similar']} similar / {r['worse']} worse\")\n    print(f\"not worse           {not_worse*100:.0f}% of pairs\")\n    print(f\"mean change         logP {r['dlogp']:+.2f}   tPSA {r['dtpsa']:+.1f}   MW {r['dmw']:+.1f}\")\n    print()\n    checks = [(f\"at least {need_not_worse*100:.0f}% of pairs not worse\", not_worse >= need_not_worse),\n              (f\"molecular weight increase under {max_mw_increase:.0f} Da\",\n               r[\"dmw\"] < max_mw_increase)]\n    for text, okd in checks:\n        print((\"  MET      \" if okd else \"  NOT MET  \") + text)\n    return dict(name=replacement, smiles=r[\"smiles\"], freq=r[\"freq\"], rank=r[\"rank\"],\n                not_worse=round(not_worse, 3), dlogp=r[\"dlogp\"], dtpsa=r[\"dtpsa\"],\n                dmw=r[\"dmw\"], met=sum(1 for _, o in checks if o), total_checks=len(checks))\n\n\ndef bioisostere_ranking(parent=\"carboxylic acid\", min_freq=300, show=True):\n    \"\"\"Frequency is not success: rank the common replacements by outcome.\"\"\"\n    import swissbio as sb\n    tab = [r for r in sb.table(parent) if r[\"freq\"] >= min_freq]\n    for r in tab:\n        n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n        r[\"not_worse\"] = (r[\"better\"] + r[\"similar\"]) / n if n else 0\n    by_freq = sorted(tab, key=lambda r: -r[\"freq\"])\n    by_ok = sorted(tab, key=lambda r: -r[\"not_worse\"])\n    if show:\n        f, ax = _fig(9.6, 5.0)\n        ax.scatter([r[\"freq\"] for r in tab], [r[\"not_worse\"] * 100 for r in tab],\n                   s=70, color=PRIMARY, zorder=3)\n        for r in tab:\n            if r[\"freq\"] > 600 or r[\"not_worse\"] > 0.88:\n                ax.annotate(r[\"name\"], (r[\"freq\"], r[\"not_worse\"] * 100),\n                            fontsize=8, xytext=(5, 4), textcoords=\"offset points\")\n        ax.set_xscale(\"log\")\n        ax.set_xlabel(f\"how often it was done (matched pairs, log scale)\")\n        ax.set_ylabel(\"% of pairs where activity was not worse\")\n        ax.set_title(f\"replacing {parent}: popularity against outcome\")\n        plt.tight_layout(); plt.show()\n    print(f\"{'most common':<30}{'most reliable':<30}\")\n    print(\"-\" * 60)\n    for i in range(min(6, len(tab))):\n        print(f\"{by_freq[i]['name'][:28]:<30}{by_ok[i]['name'][:28]:<30}\")\n    return dict(n=len(tab),\n                most_common=by_freq[0][\"name\"], most_reliable=by_ok[0][\"name\"],\n                same=by_freq[0][\"name\"] == by_ok[0][\"name\"])\n", "sims.py", "exec"), _sm.__dict__)
sys.modules["sims"] = _sm
sims = _sm

ITEMS = [
    Item("P1", "In a carrier-linked prodrug, what does the promoiety do?",
         {"A": "It protects the drug from the immune system during transport", "B": "It binds the target directly once the drug has been absorbed", "C": "It changes the physical properties and is later cleaved off", "D": "It provides the pharmacological activity until the drug acts"},
         "C", {"B": "The promoiety is removed before activity; it does not bind the target.", "D": "A carrier-linked prodrug is inactive until the promoiety is cleaved.", "A": "Immune evasion is not the role a promoiety plays.", "_correct": "The promoiety is attached to alter properties such as solubility, stability or taste, and is removed in the body to release the active drug."},
         bloom="Understand", silo="", section="precheck"),
    Item("P2", "An ester prodrug is activated faster when the alcohol carries an electron-withdrawing aryl group. Why?",
         {"A": "The compound becomes more water soluble and so hydrolyses", "B": "The ester bond becomes longer and therefore weaker overall", "C": "The aryl group attracts esterase enzymes to the molecule", "D": "The carbonyl becomes more electrophilic and easier to attack"},
         "D", {"B": "Bond length changes negligibly and is not the operative factor.", "C": "Esterases are not recruited by an aryl group in this way.", "A": "Greater solubility does not itself accelerate ester hydrolysis.", "_correct": "Withdrawing electron density from the carbonyl carbon makes it more electrophilic, so nucleophilic attack and hydrolysis proceed faster."},
         bloom="Understand", silo="", section="precheck"),
    Item("P3", "A prodrug behaves consistently in plasma but gives variable exposure across patients. What does that suggest?",
         {"A": "Activation depends on an enzyme whose activity differs between people", "B": "The prodrug is chemically unstable and decomposes before it is taken", "C": "The assay used to measure plasma concentrations is poorly calibrated", "D": "The drug itself has a narrow therapeutic window in these patients"},
         "A", {"B": "Chemical instability would show up in the plasma incubations as well.", "C": "A miscalibrated assay would distort the plasma results too.", "D": "A narrow window makes variability dangerous but does not cause it.", "_correct": "Consistency in plasma rules out chemical instability, so the variability must come from enzymes or gut flora that differ between individuals."},
         bloom="Understand", silo="", section="precheck"),
    Item("C1", "The ethyl ester showed a far larger between-patient spread at two hours than the indanyl ester did. Why?",
         {"A": "The ethyl ester is chemically unstable and decomposes unpredictably", "B": "At an intermediate rate, two hours falls where the curves differ most", "C": "Esterase activity varies more widely for ethyl esters than aryl ones", "D": "The indanyl ester is activated by a route that needs no enzyme at all"},
         "B", {"C": "The same threefold range of enzyme activity was applied to both compounds.", "D": "Both are hydrolysed enzymatically; only the rate constant differs between them.", "A": "Both are modelled as clean first-order hydrolysis with no decomposition.", "_correct": "The indanyl ester is nearly complete in every patient by two hours, so the curves converge, while an intermediate rate leaves the three patients at very different extents."},
         bloom="Analyze", silo="4", section="concept"),
    Item("C2", "You must choose a promoiety for a drug with a narrow therapeutic window. Which consideration should dominate?",
         {"A": "Reliability of activation across patients, not raw speed alone", "B": "The lowest possible rate constant, to avoid any sudden release", "C": "The highest possible rate constant, to minimise exposure time", "D": "The lowest molecular weight promoiety that can be attached"},
         "A", {"B": "A very slow promoiety may never deliver a therapeutic concentration at all.", "C": "Speed alone ignores whether every patient achieves the same exposure.", "D": "Promoiety mass is a minor consideration beside unpredictable activation.", "_correct": "With a narrow window the danger is between-patient variability in exposure, so a promoiety whose activation is consistent matters more than one that is merely fast."},
         bloom="Evaluate", silo="4", section="concept"),
    Item("Q1", "SAR data: OCH2 active, NHCH2 active, SCH2 inactive, CH2CH2 inactive, CH=CH inactive. What does the pattern establish about the oxygen?",
         {"A": "It contributes bulk that the binding site sterically requires", "B": "It stabilizes the active conformation through its bond angle", "C": "It raises lipophilicity enough to drive membrane permeation", "D": "It acts as a hydrogen-bond acceptor at that position in the site"},
         "D", {"A": "is ruled out by the similarly sized CH2CH2", "C": "is ruled out by the direction of pi values", "B": "is not distinguished by this series", "_correct": "Only O and NH can accept a hydrogen bond at that position, and the three groups that cannot accept one are exactly the inactive analogues."},
         bloom="Analyze", silo="4", section="post"),
    Item("Q2", "A CF3 group is introduced two bonds from a basic amine that must engage an anionic site residue. Potency falls. Why?",
         {"A": "CF3 sterically blocks the amine's approach to the site residue", "B": "CF3 lowers the amine's pKa, reducing the protonated fraction", "C": "CF3 is metabolized to an acid that competes for the same site", "D": "CF3 raises lipophilicity, which excludes the drug from plasma"},
         "B", {"A": "is possible but not the chapter's mechanism at that distance", "C": "is wrong because CF3 is metabolically robust", "D": "is unrelated to the binding step", "_correct": "A strongly electron-withdrawing CF3 lowers the amine's basicity, so less of it is protonated and available for the ionic interaction."},
         bloom="Apply", silo="4", section="post"),
    Item("Q3", "Two esters of one drug differ only in the alcohol: ethanol versus 5-indanol. How will activation compare, and why?",
         {"A": "The indanyl ester hydrolyzes faster, as the aryl group withdraws density", "B": "Neither hydrolyzes in plasma, since aryl esters require liver enzymes", "C": "Both hydrolyze identically, since esterases are insensitive to structure", "D": "The ethyl ester hydrolyzes faster, as smaller esters are less hindered"},
         "A", {"C": "is false", "B": "contradicts plasma esterase activity", "D": "applies a sterics-only argument", "_correct": "The indanyl group is aryl and electron-withdrawing, which makes the ester carbonyl more electrophilic and speeds hydrolysis."},
         bloom="Apply", silo="4", section="post"),
    Item("Q4", "Chloramphenicol is formulated as the palmitate ester for oral suspension and the succinate ester for injection. What does this contrast demonstrate?",
         {"A": "Two promoieties are required before any prodrug can be marketed", "B": "Ester prodrugs invariably increase the aqueous solubility of a drug", "C": "Promoiety choice is dictated by the route and the dosage form needed", "D": "The parent drug is inactive until both esters have been hydrolyzed"},
         "C", {"B": "is contradicted by the palmitate", "D": "misstates the parent's activity", "A": "is an invented requirement", "_correct": "The palmitate is poorly soluble and tasteless, suiting an oral suspension, while the water-soluble succinate suits an injection."},
         bloom="Analyze", silo="4", section="post"),
    Item("Q5", "A prodrug performs consistently in plasma incubations but gives highly variable exposure across patients. Which explanation fits best?",
         {"A": "The promoiety itself is pharmacologically active in some patients", "B": "Activating enzyme or gut flora activity differs between individuals", "C": "The linkage hydrolyzes chemically rather than enzymatically in vivo", "D": "Plasma incubation overestimates the drug's intrinsic potency in vivo"},
         "B", {"A": "would not produce variable parent exposure", "C": "is wrong because chemical hydrolysis would be more uniform", "D": "confuses potency with exposure", "_correct": "Plasma incubation tests only chemical and plasma-esterase hydrolysis, so between-patient variability must come from enzymes or gut flora that differ."},
         bloom="Evaluate", silo="4", section="post"),
]

gb = Gradebook("L32", "Prodrugs")
gb.add_many(ITEMS)
print("Ready - 10 questions loaded. Scroll down and begin.")

# ===== BEGIN structure-drawing and calculator kit (lecture notes) =====
# ---------------------------------------------------------------------------
# PHS 511 notebook helper library (pure Python + matplotlib; no RDKit, no network)
# Draws textbook-oriented 2D skeletal structures from explicit coordinates so
# every figure has a fixed, reviewed orientation:
#   * acyl chains: C1 (carbonyl) on the LEFT, omega (methyl) end on the RIGHT
#   * glycerol / sphingosine backbones vertical (sn-1 top, sn-3 bottom)
#   * steroids: rings A-B-C-D left to right, C3-OH lower left, C18/C19 up
# ---------------------------------------------------------------------------
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch, FancyBboxPatch, Circle

COL = {
    "ink": "#1f2430", "muted": "#6b7280",
    "ester": "#1d6fb8", "amide": "#c2410c", "ether": "#7c3aed",
    "phos": "#15803d", "oh": "#b91c1c", "dbl": "#d97706",
    "delta": "#1d6fb8", "omega": "#b91c1c", "drug": "#be123c",
    "polar": "#0e7490", "nonpolar": "#a16207", "hl": "#fde68a",
}
S30, C30 = 0.5, math.cos(math.radians(30))
UNIT_IN = 0.36   # inches per bond length -> identical scale in every figure


class Mol:
    """Atoms with explicit 2D coordinates; bonds with order/style/highlight."""

    def __init__(self, name=""):
        self.name, self.atoms, self.bonds, self.notes, self.boxes = name, [], [], [], []

    def add(self, x, y, label="", color=None, ha="center", size=None, anchor=None):
        """anchor: index of the label character that sits on the atom (default: first C/N/O/P/S)."""
        self.atoms.append(dict(x=x, y=y, label=label, color=color, ha=ha, size=size, anchor=anchor))
        return len(self.atoms) - 1

    def bond(self, i, j, order=1, style="plain", hl=None, color=None, side=None):
        self.bonds.append(dict(i=i, j=j, order=order, style=style, hl=hl, color=color, side=side))
        return len(self.bonds) - 1

    def note(self, x, y, text, color=None, size=8.5, ha="center", va="center", weight="normal"):
        self.notes.append(dict(x=x, y=y, text=text, color=color, size=size, ha=ha, va=va, weight=weight))

    def pos(self, i):
        a = self.atoms[i]
        return a["x"], a["y"]



def _turtle(m, start, n, d0, turns, orders, hl=None):
    idx = [start]
    x, y = m.pos(start)
    d = d0
    for k in range(1, n + 1):
        x, y = x + math.cos(math.radians(d)), y + math.sin(math.radians(d))
        idx.append(m.add(x, y))
        m.bond(idx[-2], idx[-1], order=orders.get(k, 1), hl=hl)
        if k < n:
            d += turns[k]
    return idx


def chain(m, start, n, heading=0.0, first=+1, cis=(), trans=(), hl=None, flip=()):
    """Zigzag chain of n carbons grown from atom `start` (start = carbon 0).
    Bond k joins chain carbons k-1 and k. The turn at carbon k normally
    alternates sign (zigzag). If bond k is a cis C=C, the turn at carbon k
    repeats the previous sign so both substituents sit on the same side.
    `flip` lists carbons where a single bond is drawn rotated (same-sign turn)
    to steer a long chain back into its row; it does not change the molecule."""
    turns, orders = {}, {}
    sign = -first
    for k in range(1, n):
        if k >= 2:
            sign = sign if (k in cis or k in flip) else -sign
        turns[k] = 60 * sign
    for k in list(cis) + list(trans):
        orders[k] = 2
    return _turtle(m, start, n, heading + 30 * first, turns, orders, hl)


# ---------------------------------------------------------------------------
# Rendering
# ---------------------------------------------------------------------------
SUBS = set("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089\u207a\u207b\u00b9\u00b2\u00b3\u2074")
EM = 1.0 / (72 * UNIT_IN)   # data units per point at the fixed drawing scale


def _cw(ch):
    """Approximate glyph advance (em) for DejaVu Sans."""
    if ch in SUBS:
        return 0.40
    if ch in "()":
        return 0.39
    if ch.isupper():
        return 0.72
    if ch.isdigit():
        return 0.64
    return 0.56


def _label_box(lab, fs, anchor=None):
    """(left, right, half-height) of a label in data units, measured from the
    anchor glyph centre (first C/N/O/P/S), which sits exactly on the atom."""
    em = fs * EM
    ws = [_cw(c) * em for c in lab]
    k = anchor if anchor is not None else next((i for i, c in enumerate(lab) if c in "CNOPS"), 0)
    left = sum(ws[:k]) + ws[k] / 2
    right = sum(ws[k + 1:]) + ws[k] / 2
    return left, right, 0.36 * fs * EM * 1.0


def _label_radius(lab, ux, uy, fs=10.5, anchor=None):
    """Distance from atom centre to the label box edge along (ux, uy), plus a gap."""
    if not lab:
        return 0.0
    left, right, hh = _label_box(lab, fs, anchor)
    tx = (right if ux > 0 else left) / abs(ux) if abs(ux) > 1e-9 else 1e9
    ty = hh / abs(uy) if abs(uy) > 1e-9 else 1e9
    return min(tx, ty) + 0.08


def draw(ax, m, fs=10.5):
    A = m.atoms
    nb = {i: [] for i in range(len(A))}
    for b in m.bonds:
        nb[b["i"]].append(b["j"]); nb[b["j"]].append(b["i"])
    # highlights first (under everything)
    for b in m.bonds:
        if b["hl"]:
            (x1, y1), (x2, y2) = m.pos(b["i"]), m.pos(b["j"])
            ax.plot([x1, x2], [y1, y2], color=b["hl"], lw=9, alpha=0.30,
                    solid_capstyle="round", zorder=1)
    for b in m.bonds:
        i, j = b["i"], b["j"]
        (x1, y1), (x2, y2) = m.pos(i), m.pos(j)
        L = math.hypot(x2 - x1, y2 - y1)
        ux, uy = (x2 - x1) / L, (y2 - y1) / L
        r1 = _label_radius(A[i]["label"], ux, uy, A[i]["size"] or fs, A[i]["anchor"])
        r2 = _label_radius(A[j]["label"], ux, uy, A[j]["size"] or fs, A[j]["anchor"])
        ax1, ay1 = x1 + ux * r1, y1 + uy * r1
        ax2, ay2 = x2 - ux * r2, y2 - uy * r2
        c = b["color"] or COL["ink"]
        st = b["style"]
        if st == "wedge":
            px, py = -uy * 0.11, ux * 0.11
            ax.add_patch(Polygon([(ax1, ay1), (ax2 + px, ay2 + py), (ax2 - px, ay2 - py)],
                                 closed=True, fc=c, ec=c, lw=0.6, zorder=3))
            continue
        if st == "hash":
            nseg = 7
            for s in range(nseg + 1):
                f = s / nseg
                cx, cy = ax1 + (ax2 - ax1) * f, ay1 + (ay2 - ay1) * f
                w = 0.02 + 0.10 * f
                ax.plot([cx - uy * w, cx + uy * w], [cy + ux * w, cy - ux * w],
                        color=c, lw=1.1, zorder=3, solid_capstyle="butt")
            continue
        if st == "wavy":
            n = 40
            xs = [ax1 + (ax2 - ax1) * k / n - uy * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ys = [ay1 + (ay2 - ay1) * k / n + ux * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ax.plot(xs, ys, color=c, lw=1.4, zorder=3)
            continue
        px, py = -uy, ux
        if b["order"] == 1:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round",
                    ls="--" if st == "dashed" else "-")
            continue
        side = b["side"]
        if side is None:
            if A[i]["label"] or A[j]["label"]:
                side = 0                      # C=O, C=N+: centred pair
            else:
                sx = sy = 0.0
                for k in nb[i]:
                    if k != j:
                        sx += A[k]["x"] - x1; sy += A[k]["y"] - y1
                for k in nb[j]:
                    if k != i:
                        sx += A[k]["x"] - x2; sy += A[k]["y"] - y2
                cross = ux * sy - uy * sx
                side = 1 if cross >= -1e-6 else -1
        if side == 0:
            for sgn in (+1, -1):
                o = 0.075 * sgn
                ax.plot([ax1 + px * o, ax2 + px * o], [ay1 + py * o, ay2 + py * o],
                        color=c, lw=1.5, zorder=3, solid_capstyle="round")
        else:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round")
            o, sh = 0.15 * side, 0.16
            ax.plot([ax1 + px * o + ux * sh, ax2 + px * o - ux * sh],
                    [ay1 + py * o + uy * sh, ay2 + py * o - uy * sh],
                    color=c, lw=1.5, zorder=3, solid_capstyle="round")
    for a in A:
        if a["label"]:
            l_, r_, _ = _label_box(a["label"], a["size"] or fs, a["anchor"])
            ax.text(a["x"] + (r_ - l_) / 2, a["y"], a["label"], ha="center", va="center",
                    fontsize=a["size"] or fs, color=a["color"] or COL["ink"],
                    zorder=5, fontweight="normal")
    for n_ in m.notes:
        ax.text(n_["x"], n_["y"], n_["text"], ha=n_["ha"], va=n_["va"], fontsize=n_["size"],
                color=n_["color"] or COL["muted"], zorder=6, fontweight=n_["weight"])


def extent(m, pad=0.7):
    xs = [a["x"] for a in m.atoms] + [n["x"] for n in m.notes]
    ys = [a["y"] for a in m.atoms] + [n["y"] for n in m.notes]
    return min(xs) - pad, max(xs) + pad, min(ys) - pad, max(ys) + pad




def show(m, title=None, unit=UNIT_IN, extra=None, caption=None, show_now=True):
    """One structure at the fixed scale (UNIT_IN inches per bond)."""
    x0, x1, y0, y1 = extent(m)
    if extra and hasattr(extra, "bounds"):
        bx0, bx1, by0, by1 = extra.bounds
        x0, x1, y0, y1 = min(x0, bx0), max(x1, bx1), min(y0, by0), max(y1, by1)
    th = 0.42 if title else 0.0
    ch = 0.40 if caption else 0.0
    W, H = (x1 - x0) * unit, (y1 - y0) * unit
    fig = plt.figure(figsize=(W, H + th + ch))
    ax = fig.add_axes([0, ch / (H + th + ch), 1, H / (H + th + ch)])
    draw(ax, m)
    if extra:
        extra(ax)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        fig.text(0.0, 1 - (th / 2) / (H + th + ch), title, fontsize=11.5, fontweight="bold",
                 color=COL["ink"], va="center", ha="left")
    if caption:
        fig.text(0.0, (ch / 2) / (H + th + ch), caption, fontsize=9, color=COL["muted"],
                 va="center", ha="left")
    if show_now:
        plt.show()
    return fig


def grid(mols, titles, ncols=2, unit=UNIT_IN, align="left", suptitle=None, captions=None, show_now=True, bottom=0.0):
    """Panels drawn at one common scale; each panel box has the same width and
    the row height of its tallest member, so structures line up cleanly."""
    exts = [extent(m) for m in mols]
    W = max(max(e[1] - e[0] for e in exts), 3.6 / unit)   # panels at least 3.6 in wide
    nrows = math.ceil(len(mols) / ncols)
    import textwrap as _tw
    maxch = max(12, int(W * unit * 72 / (10.5 * 0.60)) - 1)
    wrapped = [_tw.fill(t, maxch) for t in titles]
    rowH = []
    for r in range(nrows):
        rowH.append(max(exts[k][3] - exts[k][2] for k in range(r * ncols, min(len(mols), (r + 1) * ncols))))
    row_th = []
    for r in range(nrows):
        nl = max(wrapped[k].count("\n") + 1 for k in range(r * ncols, min(len(mols), (r + 1) * ncols)))
        row_th.append(0.27 + 0.19 * nl)
    fig_w = W * unit * ncols
    fig_h = sum(h * unit + th for h, th in zip(rowH, row_th)) + (0.5 if suptitle else 0) + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    top = 1.0 - (0.5 / fig_h if suptitle else 0)
    y_cursor = top
    for r in range(nrows):
        h_frac = (rowH[r] * unit) / fig_h
        t_frac = row_th[r] / fig_h
        y_cursor -= (h_frac + t_frac)
        for c in range(ncols):
            k = r * ncols + c
            if k >= len(mols):
                break
            ax = fig.add_axes([c / ncols, y_cursor, 1 / ncols, h_frac])
            m = mols[k]
            x0, x1, y0, y1 = exts[k]
            draw(ax, m)
            if align == "left":
                ax.set_xlim(x0, x0 + W)
            else:
                cx = (x0 + x1) / 2
                ax.set_xlim(cx - W / 2, cx + W / 2)
            cy = (y0 + y1) / 2
            ax.set_ylim(cy - rowH[r] / 2, cy + rowH[r] / 2)
            ax.set_aspect("equal"); ax.axis("off")
            ax.text(0.01, 1.0 + 0.02, wrapped[k], transform=ax.transAxes, fontsize=10.5, linespacing=1.15,
                    fontweight="bold", color=COL["ink"], va="bottom", ha="left")
    if suptitle:
        fig.text(0.005, 1 - 0.25 / fig_h, suptitle, fontsize=12.5, fontweight="bold", va="center")
    if show_now:
        plt.show()
    return fig


# ---------------------------------------------------------------------------
# Structure builders (coordinates chosen to match textbook orientation)
# ---------------------------------------------------------------------------
def _outer(m, i, nbrs, dist=0.55):
    """Point on the open (outer) side of atom i, away from its neighbours."""
    x, y = m.pos(i)
    sx = sum(m.atoms[k]["x"] - x for k in nbrs); sy = sum(m.atoms[k]["y"] - y for k in nbrs)
    L = math.hypot(sx, sy) or 1.0
    return x - sx / L * dist, y - sy / L * dist


def fatty_acid(n, cis=(), trans=(), name="", numbers=None, ionized=False, origin=(0.0, 0.0),
               hl_double=True, flip=()):
    """Free fatty acid, C1 (COOH) on the LEFT, methyl (omega) end on the RIGHT.
    cis/trans: Delta positions (bond between C_p and C_p+1).
    numbers: None | 'delta' | 'omega' | 'both'"""
    m = Mol(name)
    ox, oy = origin
    c1 = m.add(ox, oy)
    o_dbl = m.add(ox, oy + 1.0, "O")
    o_h = m.add(ox - C30, oy - 0.5, "⁻O" if ionized else "HO")
    m.bond(c1, o_dbl, 2); m.bond(c1, o_h, 1)
    hl = COL["hl"] if hl_double else None
    ids = chain(m, c1, n - 1, heading=0, first=-1, cis=cis, trans=trans, flip=flip)
    if hl_double:
        for b in m.bonds:
            if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
                b["hl"] = COL["dbl"]
    m.carbons = ids  # carbons[0] = C1
    if numbers:
        dbs = sorted(list(cis) + list(trans))
        last = dbs[-1] if dbs else n - 2
        for p, a in enumerate(ids, start=1):
            if p == 1:
                nb = [o_dbl, o_h, ids[1]]
            elif p == n:
                nb = [ids[p - 2]]
            else:
                nb = [ids[p - 2], ids[p]]
            if numbers in ("delta", "both"):
                x, y = _outer(m, a, nb, 0.50)
                m.note(x, y, str(p), color=COL["delta"], size=7.5)
            w = n - p + 1
            if numbers == "omega" or (numbers == "both" and p >= last):
                if p == n:
                    x, y = _outer(m, a, nb, 0.50)
                    y -= 0.42 if numbers == "both" else 0.0
                else:
                    x, y = _outer(m, a, nb, 0.50 if numbers == "omega" else -0.62)
                m.note(x, y, ("\u03c9" + str(w)) if w == 1 else str(w), color=COL["omega"], size=7.5,
                       weight="bold")
    return m


def acyl_row(m, o_atom, n, cis=(), trans=(), carbonyl_up=True, flip=None):
    """Acyl group C(=O)-chain attached to ester/amide atom o_atom, chain to the right."""
    x, y = m.pos(o_atom)
    c1 = m.add(x + C30, y + (0.5 if carbonyl_up else -0.5))
    m.bond(o_atom, c1, 1)
    od = m.add(x + C30, y + (1.5 if carbonyl_up else -1.5), "O")
    m.bond(c1, od, 2)
    if flip is None:   # keep long chains in their row: rotate the bond 3 carbons past the last cis C=C
        flip = tuple(p + 2 for p in cis)
    ids = chain(m, c1, n - 1, heading=0, first=-1 if carbonyl_up else +1, cis=cis, trans=trans, flip=flip)
    for b in m.bonds:
        if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
            b["hl"] = COL["dbl"]
    return c1, od, ids


ROW = 2.6  # vertical spacing of backbone carbons in E-shaped lipids


def backbone(m, labels, x=0.0, y0=0.0):
    """Vertical 3-carbon backbone (glycerol or sphingosine C1-C3), top->bottom."""
    ids = []
    for k, lab in enumerate(labels):
        ids.append(m.add(x, y0 - k * ROW, lab))
        if k:
            m.bond(ids[k - 1], ids[k], 1)
    return ids


def ester(m, c_backbone, n, cis=(), trans=(), hl=True, cut=None):
    x, y = m.pos(c_backbone)
    o = m.add(x + 1.15, y, "O")
    b0 = m.bond(c_backbone, o, 1, hl=COL["ester"] if hl else None)
    c1, od, ids = acyl_row(m, o, n, cis, trans)
    m.bonds[-(len(ids)) - 1]["hl"] = None
    # highlight O-C(=O) (the ester bond lipases cut)
    for b in m.bonds:
        if b["i"] == o and b["j"] == c1:
            b["hl"] = COL["ester"] if hl else None
            if cut:
                b["color"] = COL["drug"]
    return o, c1, ids


def triacylglycerol(acyls=((16, ()), (18, (9,)), (18, (9, 12))), name="Triacylglycerol",
                    sn=True, cuts=None):
    """E-shaped TAG; glycerol vertical, sn-1 top. acyls: ((n, cis-positions), ...)."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    rows = []
    for k, (n, cis) in enumerate(acyls):
        rows.append(ester(m, g[k], n, cis=cis))
    if sn:
        for k in range(3):
            x, y = m.pos(g[k])
            m.note(x - 0.95, y, "sn-%d" % (k + 1), color=COL["muted"], size=8.5, ha="right")
    m.g, m.rows = g, rows
    return m


def mark_cut(m, o_idx, c_idx, text, color=None, dy=-0.62, dx=0.0):
    x1, y1 = m.pos(o_idx); x2, y2 = m.pos(c_idx)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    m.note(mx, my - 0.05, "✂", color=color or COL["drug"], size=13)
    m.note(mx + dx, my + dy, text, color=color or COL["drug"], size=7.5, ha="left" if dx else "center")


def phosphocholine(m, c_attach, head="choline"):
    """-O-P(=O)(O-)-O-CH2-CH2-N+(CH3)3 to the right of c_attach (horizontal)."""
    x, y = m.pos(c_attach)
    o1 = m.add(x + 1.15, y, "O"); m.bond(c_attach, o1, 1)
    p = m.add(x + 2.3, y, "P", color=COL["phos"]); m.bond(o1, p, 1, hl=COL["phos"])
    po = m.add(x + 2.3, y + 1.05, "O"); m.bond(p, po, 2)
    pm = m.add(x + 2.3, y - 1.05, "O⁻"); m.bond(p, pm, 1)
    o2 = m.add(x + 3.45, y, "O"); m.bond(p, o2, 1, hl=COL["phos"])
    if head == "choline":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "N⁺(CH₃)₃"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.6, y - 1.0, "choline", color=COL["muted"], size=8)
    elif head == "ethanolamine":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "NH₃⁺"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.4, y - 1.0, "ethanolamine", color=COL["muted"], size=8)
    return p


def phosphatidylcholine(name="Phosphatidylcholine (lecithin)", a1=(16, ()), a2=(18, (9,))):
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    ester(m, g[0], *a1); ester(m, g[1], *a2)
    phosphocholine(m, g[2], "choline")
    for k in range(3):
        x, y = m.pos(g[k]); m.note(x - 0.95, y, "sn-%d" % (k + 1), size=8.5, ha="right")
    m.g = g
    return m


def plasmalogen(name="Plasmalogen (ethanolamine)", n1=16, a2=(18, (9,))):
    """sn-1 vinyl ether: CH2-O-CH=CH-R (cis), no carbonyl."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    x, y = m.pos(g[0])
    o = m.add(x + 1.15, y, "O"); m.bond(g[0], o, 1, hl=COL["ether"])
    ids = chain(m, o, n1, heading=0, first=+1, cis=(2,), flip=(4,))
    # bond O-C1' is ether; C1'=C2' vinyl (cis, as in natural plasmalogens)
    for b in m.bonds:
        if b["i"] == o and b["j"] == ids[1]:
            b["hl"] = COL["ether"]
        if b["order"] == 2:
            b["hl"] = COL["ether"]
    ester(m, g[1], *a2)
    phosphocholine(m, g[2], "ethanolamine")
    for k in range(3):
        xx, yy = m.pos(g[k]); m.note(xx - 0.95, yy, "sn-%d" % (k + 1), size=8.5, ha="right")
    return m


def sphingo_core(m, fa_n=18, fa_cis=()):
    """Sphingosine drawn as the same vertical 3-carbon backbone:
    C3 top (CH-OH + trans CH=CH tail), C2 middle (CH-NH-acyl, amide), C1 bottom (CH2-O-head)."""
    s = backbone(m, ["CH", "CH", "CH₂"])
    x, y = m.pos(s[0])
    oh = m.add(x - 1.15, y, "HO"); m.bond(s[0], oh, 1, hl=COL["oh"])
    ids = chain(m, s[0], 15, heading=0, first=+1, trans=(1,))
    # C4=C5 trans: chain bond 1 is C3-C4? -> rebuild explicitly
    m.bonds[[k for k, b in enumerate(m.bonds) if b["i"] == s[0] and b["j"] == ids[1]][0]]["order"] = 1
    for b in m.bonds:
        if b["i"] == ids[1] and b["j"] == ids[2]:
            b["order"] = 2; b["hl"] = COL["dbl"]
    # amide at C2
    x2, y2 = m.pos(s[1])
    nh = m.add(x2 + 1.4, y2, "NH"); m.bond(s[1], nh, 1)
    c1, od, acyl = acyl_row(m, nh, fa_n, cis=fa_cis)
    for b in m.bonds:
        if b["i"] == nh and b["j"] == c1:
            b["hl"] = COL["amide"]
    for k, lab in enumerate(["C3", "C2", "C1"]):
        xx, yy = m.pos(s[k]); m.note(xx - 0.25, yy - 0.55, lab, size=7.5)
    return s


def sphingomyelin(name="Sphingomyelin"):
    m = Mol(name)
    s = sphingo_core(m, 18)
    phosphocholine(m, s[2], "choline")
    return m


def galactosylceramide(name="Galactosylceramide (a cerebroside)"):
    m = Mol(name)
    s = sphingo_core(m, 24, fa_cis=(15,))
    x, y = m.pos(s[2])
    o = m.add(x + 1.15, y, "O"); m.bond(s[2], o, 1, hl=COL["polar"])
    # beta-D-galactopyranose, Haworth-free chair-less hexagon (ring O upper right)
    cx, cy, r = x + 3.0, y - 0.1, 1.0
    ang = [180, 120, 60, 0, -60, -120]
    pts = [m.add(cx + r * math.cos(math.radians(a)), cy + r * math.sin(math.radians(a)),
                 "O" if k == 1 else "") for k, a in enumerate(ang)]
    for k in range(6):
        m.bond(pts[k], pts[(k + 1) % 6])
    m.bond(o, pts[0], 1, hl=COL["polar"])
    m.note(cx, cy - 0.05, "Gal", color=COL["polar"], size=9, weight="bold")
    return m


# --- steroids --------------------------------------------------------------
def _hex(cx, cy):
    return [(cx + math.cos(math.radians(a)), cy + math.sin(math.radians(a))) for a in (90, 30, -30, -90, -150, 150)]


def steroid_core(m, ox=0.0, oy=0.0, dbl=()):
    """Gonane nucleus in standard orientation. Returns dict C1..C17 -> atom index.
    Unit hexagon geometry with vertical shared A/B edge (C5-C10)."""
    r3 = math.sqrt(3)
    P = {}
    A = _hex(ox, oy)                      # top, ur, lr, bottom, ll, ul
    P.update({1: A[0], 10: A[1], 5: A[2], 4: A[3], 3: A[4], 2: A[5]})
    B = _hex(ox + r3, oy)
    P.update({9: B[0], 8: B[1], 6: B[3], 7: B[2]})
    C = _hex(ox + 1.5 * r3, oy + 1.5)
    P.update({12: C[0], 13: C[1], 14: C[2], 11: C[5]})
    # ring D: regular pentagon on edge C13-C14, outward (to the right)
    (x13, y13), (x14, y14) = P[13], P[14]
    mx, my = (x13 + x14) / 2, (y13 + y14) / 2
    s = 1.0
    apo = s / (2 * math.tan(math.radians(36)))
    cx, cy = mx + apo, my
    R = s / (2 * math.sin(math.radians(36)))
    base = math.degrees(math.atan2(y13 - cy, x13 - cx))
    pent = [(cx + R * math.cos(math.radians(base - 72 * k)), cy + R * math.sin(math.radians(base - 72 * k)))
            for k in range(5)]
    # pent[0]=C13, going clockwise: C17, C16, C15, C14
    P.update({17: pent[1], 16: pent[2], 15: pent[3]})
    idx = {k: m.add(v[0], v[1]) for k, v in P.items()}
    ring_bonds = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 10), (10, 1), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
                  (9, 11), (11, 12), (12, 13), (13, 14), (14, 8), (13, 17), (17, 16), (16, 15), (15, 14)]
    for a, b in ring_bonds:
        m.bond(idx[a], idx[b], 2 if (a, b) in dbl or (b, a) in dbl else 1)
    m.st = idx
    return idx


def subst(m, at, dx, dy, label="", style="plain", order=1, hl=None, color=None):
    x, y = m.pos(at)
    k = m.add(x + dx, y + dy, label, color=color)
    m.bond(at, k, order, style=style, hl=hl)
    return k


def side_chain_cholestane(m, st, extra_ethyl=False):
    """C20-C27 side chain on C17 (drawn up-right, zigzag)."""
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 4, heading=0, first=-1)
    c25 = ids[-1]
    x25, y25 = m.pos(c25)
    c26 = m.add(x25 + C30, y25 + 0.5); m.bond(c25, c26)
    c27 = m.add(x25, y25 - 1.0); m.bond(c25, c27)
    if extra_ethyl:
        x24, y24 = m.pos(ids[3])
        ex, ey = _outer(m, ids[3], [ids[2], ids[4]], 1.0)
        e1 = m.add(ex, ey); m.bond(ids[3], e1, hl=COL["nonpolar"])
        e2 = m.add(ex + C30, ey + (0.5 if ey > y24 else -0.5)); m.bond(e1, e2, hl=COL["nonpolar"])
    return ids


def cholesterol(name="Cholesterol", sitosterol=False):
    m = Mol(name)
    st = steroid_core(m, dbl=((5, 6),))
    ho = subst(m, st[3], -C30, -0.5, "HO", style="wedge", hl=COL["oh"])
    subst(m, st[10], 0.0, 0.95, "", style="wedge")     # C19 methyl
    subst(m, st[13], 0.0, 0.95, "", style="wedge")     # C18 methyl
    side_chain_cholestane(m, st, extra_ethyl=sitosterol)
    m.note(m.pos(st[3])[0] - 0.2, m.pos(st[3])[1] + 0.45, "3β", size=7.5)
    for k, lab in ((1, "A"), (2, "B"), (3, "C")):
        pass
    r3 = math.sqrt(3)
    for (cx, cy, lab) in ((0, 0, "A"), (r3, 0, "B"), (1.5 * r3, 1.5, "C")):
        m.note(cx, cy, lab, color="#9aa3b2", size=9, weight="bold")
    x13, y13 = m.pos(st[13]); x14, y14 = m.pos(st[14])
    m.note((x13 + x14) / 2 + 0.68, (y13 + y14) / 2, "D", color="#9aa3b2", size=9, weight="bold")
    m.st = st
    return m


def bile_acid(name="Glycocholate", oh=(("3", "a"), ("7", "a"), ("12", "a")), conj="glycine", ionized=True):
    """5-beta bile acid; oh entries (position, 'a' alpha=hash | 'b' beta=wedge)."""
    m = Mol(name)
    st = steroid_core(m)
    subst(m, st[10], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[13], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[5], 0.0, -0.85, "H", style="wedge")      # 5-beta H (A/B cis)
    vec = {"3": (-C30, -0.5), "7": (C30 * 0.0, -1.0), "12": (0.0, 1.0)}
    for pos, face in oh:
        dx, dy = vec[pos]
        if pos == "12":
            dx, dy = (-0.35, 0.9)
        if pos == "7":
            dx, dy = (0.35, -0.95)
        subst(m, st[int(pos)], dx, dy, "HO" if pos == "3" else "OH",
              style="hash" if face == "a" else "wedge", hl=COL["polar"])
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 3, heading=0, first=-1)
    c24 = ids[-1]
    x24, y24 = m.pos(c24)
    o24 = m.add(x24, y24 + 1.0, "O"); m.bond(c24, o24, 2)
    if conj == "glycine":
        nh = m.add(x24 + C30, y24 - 0.5, "NH"); m.bond(c24, nh, 1, hl=COL["amide"])
        ca = m.add(x24 + 2 * C30, y24); m.bond(nh, ca)
        cc = m.add(x24 + 3 * C30, y24 - 0.5); m.bond(ca, cc)
        m.bond(cc, m.add(x24 + 3 * C30, y24 - 1.5, "O"), 2)
        m.bond(cc, m.add(x24 + 4 * C30, y24, "O⁻" if ionized else "OH"), 1, hl=COL["polar"])
    else:
        m.bond(c24, m.add(x24 + C30, y24 - 0.5, "OH"), 1, hl=COL["polar"])
    m.st = st
    return m


# ---------------------------------------------------------------------------
# General 2D sketch helpers: rings, fused rings, substituents, reaction schemes
# (built on draw.py; every coordinate is explicit so orientation is fixed)
# ---------------------------------------------------------------------------
def _nbrs(m, i):
    out = []
    for b in m.bonds:
        if b["i"] == i:
            out.append(b["j"])
        elif b["j"] == i:
            out.append(b["i"])
    return out


def bond_between(m, i, j):
    for b in m.bonds:
        if (b["i"], b["j"]) in ((i, j), (j, i)):
            return b
    return None


def ring(m, cx, cy, n=6, rot=90.0, labels=None, side=1.0):
    """Regular n-ring with bond length `side`, first vertex at angle `rot` (deg),
    vertices counter-clockwise. labels: list of n labels ('' = carbon).
    Returns atom indices in ring order."""
    R = side / (2 * math.sin(math.pi / n))
    ids = []
    for k in range(n):
        a = math.radians(rot + 360.0 * k / n)
        ids.append(m.add(cx + R * math.cos(a), cy + R * math.sin(a), (labels[k] if labels else "")))
    for k in range(n):
        m.bond(ids[k], ids[(k + 1) % n], 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def fuse(m, a, b, n=6, labels=None, side=1.0):
    """New regular n-ring sharing edge a-b, built on the side away from the
    existing neighbours. labels: for the n-2 NEW atoms, in order from b onward.
    Returns ring atoms [a, b, new...]."""
    (xa, ya), (xb, yb) = m.pos(a), m.pos(b)
    mx, my = (xa + xb) / 2, (ya + yb) / 2
    ex, ey = xb - xa, yb - ya
    L = math.hypot(ex, ey)
    px, py = -ey / L, ex / L
    # decide which side is free: away from centroid of current neighbours
    nb = [k for k in _nbrs(m, a) + _nbrs(m, b) if k not in (a, b)]
    if nb:
        cx_ = sum(m.atoms[k]["x"] for k in nb) / len(nb); cy_ = sum(m.atoms[k]["y"] for k in nb) / len(nb)
        if (cx_ - mx) * px + (cy_ - my) * py > 0:
            px, py = -px, -py
    apo = side / (2 * math.tan(math.pi / n))
    cx, cy = mx + px * apo, my + py * apo
    R = side / (2 * math.sin(math.pi / n))
    # walk from b around the new ring (away from a)
    ang_b = math.atan2(yb - cy, xb - cx)
    ang_a = math.atan2(ya - cy, xa - cx)
    step = 2 * math.pi / n
    # direction: going from b, the next vertex must not be a
    d = (ang_b - ang_a + math.pi) % (2 * math.pi) - math.pi
    sgn = 1 if d > 0 else -1
    ids = [a, b]
    prev = b
    for k in range(1, n - 1):
        ang = ang_b + sgn * step * k
        lab = labels[k - 1] if labels else ""
        i = m.add(cx + R * math.cos(ang), cy + R * math.sin(ang), lab)
        m.bond(prev, i, 1)
        ids.append(i); prev = i
    m.bond(prev, a, 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def set_double(m, pairs):
    """pairs: iterable of (i, j) atom-index pairs to make double bonds."""
    for i, j in pairs:
        b = bond_between(m, i, j)
        if b is None:
            raise ValueError("no bond %d-%d" % (i, j))
        b["order"] = 2


def kekule(m, ids, start=0):
    """Alternate double bonds around ring `ids` beginning with bond start->start+1."""
    n = len(ids)
    for k in range(start, start + n, 2):
        bond_between(m, ids[k % n], ids[(k + 1) % n])["order"] = 2


def outward(m, i, dist=1.0):
    """Unit outward direction from atom i (away from its neighbours)."""
    x, y = m.pos(i)
    nb = _nbrs(m, i)
    sx = sum(m.atoms[k]["x"] - x for k in nb); sy = sum(m.atoms[k]["y"] - y for k in nb)
    L = math.hypot(sx, sy)
    if L < 1e-9:
        return (dist, 0.0)
    return (-sx / L * dist, -sy / L * dist)


def sub(m, i, label="", angle=None, dist=1.0, order=1, style="plain", hl=None, color=None, anchor=None):
    """Attach a substituent to atom i: outward by default, or at an absolute angle (deg)."""
    x, y = m.pos(i)
    if angle is None:
        dx, dy = outward(m, i, dist)
    else:
        dx, dy = dist * math.cos(math.radians(angle)), dist * math.sin(math.radians(angle))
    k = m.add(x + dx, y + dy, label, color=color, anchor=anchor)
    m.bond(i, k, order, style=style, hl=hl)
    return k


def grow(m, i, n, angle=None, first=+1, cis=(), trans=(), flip=()):
    """Zigzag chain of n atoms from atom i; heading = outward unless angle given."""
    if angle is None:
        dx, dy = outward(m, i)
        angle = math.degrees(math.atan2(dy, dx))
    return chain(m, i, n, heading=angle, first=first, cis=cis, trans=trans, flip=flip)


def label(m, i, text, anchor=None):
    m.atoms[i]["label"] = text
    m.atoms[i]["anchor"] = anchor


def hl_bonds(m, pairs, color):
    for i, j in pairs:
        bond_between(m, i, j)["hl"] = color


def hl_atoms_note(m, i, text, color=None, dist=0.75, size=8):
    x, y = m.pos(i)
    dx, dy = outward(m, i, dist)
    m.note(x + dx, y + dy, text, color=color or COL["muted"], size=size)


def shift(m, dx, dy):
    for a in m.atoms:
        a["x"] += dx; a["y"] += dy
    for n in m.notes:
        n["x"] += dx; n["y"] += dy
    return m


def rotate(m, deg, about=None):
    """Rotate a whole drawing (use to fix an orientation, e.g. to put a ring flat)."""
    if about is None:
        xs = [a["x"] for a in m.atoms]; ys = [a["y"] for a in m.atoms]
        about = ((min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2)
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    for obj in m.atoms + m.notes:
        x, y = obj["x"] - about[0], obj["y"] - about[1]
        obj["x"], obj["y"] = about[0] + c * x - s * y, about[1] + s * x + c * y
    return m


def mirror(m, axis="x"):
    """Mirror left-right (axis='x') or top-bottom ('y'); wedges keep their sense."""
    for obj in m.atoms + m.notes:
        if axis == "x":
            obj["x"] = -obj["x"]
        else:
            obj["y"] = -obj["y"]
    return m


def merge(dst, src, dx=0.0, dy=0.0):
    """Copy all atoms/bonds/notes of src into dst with an offset; returns index map."""
    base = len(dst.atoms)
    for a in src.atoms:
        b = dict(a); b["x"] += dx; b["y"] += dy
        dst.atoms.append(b)
    for b in src.bonds:
        c = dict(b); c["i"] += base; c["j"] += base
        dst.bonds.append(c)
    for n in src.notes:
        c = dict(n); c["x"] += dx; c["y"] += dy
        dst.notes.append(c)
    return base


# ---------------------------------------------------------------------------
# Reaction / pathway scheme: molecules in a row (wrapping), arrows with labels
# ---------------------------------------------------------------------------
def scheme(items, title=None, per_row=3, unit=UNIT_IN, gap=3.2, caption=None, show_now=True):
    """items: [Mol, ("enzyme", "cofactor note"), Mol, (...), Mol ...]
    Molecules are drawn at one scale; arrows carry an enzyme label above and
    an optional cofactor/regulation note below. Rows wrap after per_row molecules."""
    mols = [x for x in items if isinstance(x, Mol)]
    arrows = [x for x in items if not isinstance(x, Mol)]
    exts = [extent(mm, pad=0.4) for mm in mols]
    rows = [list(range(k, min(k + per_row, len(mols)))) for k in range(0, len(mols), per_row)]
    row_h = [max(exts[k][3] - exts[k][2] for k in r) for r in rows]
    row_w = [sum(exts[k][1] - exts[k][0] for k in r) + gap * (len(r) - 1) for r in rows]
    W = max(row_w) + (1.6 if len(rows) > 1 else 0)
    H = sum(row_h) + 1.6 * (len(rows) - 1)
    th = 0.45 if title else 0.0
    ch = 0.4 if caption else 0.0
    fig = plt.figure(figsize=(W * unit, H * unit + th + ch))
    ax = fig.add_axes([0, ch / (H * unit + th + ch), 1, H * unit / (H * unit + th + ch)])
    ax.set_xlim(0, W); ax.set_ylim(-H, 0); ax.set_aspect("equal"); ax.axis("off")
    y_top = 0.0
    pos = {}
    for r, ids in enumerate(rows):
        x = 0.0 if r == 0 else 1.6
        cy = y_top - row_h[r] / 2
        for k in ids:
            x0, x1, y0, y1 = exts[k]
            dx = x - x0; dy = cy - (y0 + y1) / 2
            mm = Mol(); merge(mm, mols[k], dx, dy)
            draw(ax, mm)
            pos[k] = (x, x + (x1 - x0), cy, y1 - y0)
            x += (x1 - x0) + gap
        y_top -= row_h[r] + 1.6
    for k, lab in enumerate(arrows):
        if k + 1 >= len(mols):
            break
        enz, note = (lab, "") if isinstance(lab, str) else (lab[0], lab[1] if len(lab) > 1 else "")
        xa0, xa1, ya, ha = pos[k]; xb0, xb1, yb, hb = pos[k + 1]
        same = abs(ya - yb) < 1e-6
        if same:
            p, q = (xa1 + 0.25, ya), (xb0 - 0.25, yb)
        else:  # wrap: from under the last molecule of a row to above the first of the next
            p, q = ((xa0 + xa1) / 2, ya - ha / 2 + 0.1), ((xb0 + xb1) / 2, yb + hb / 2 - 0.1)
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=14, lw=1.4, color=COL["ink"],
                                     connectionstyle="arc3,rad=0" if same else "arc3,rad=-0.12"))
        mx, my = (p[0] + q[0]) / 2, (p[1] + q[1]) / 2
        box = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9)
        if same:
            ax.text(mx, my + 0.3, enz, ha="center", va="bottom", fontsize=8.5, color=COL["ester"],
                    fontweight="bold", bbox=box)
            if note:
                ax.text(mx, my - 0.3, note, ha="center", va="top", fontsize=8, color=COL["muted"], bbox=box)
        else:
            ax.text(mx, my, enz + ("\n" + note if note else ""), ha="center", va="center", fontsize=8.5,
                    color=COL["ester"], bbox=box)
    if title:
        fig.text(0.0, 1 - (th / 2) / (H * unit + th + ch), title, fontsize=11.5, fontweight="bold", va="center")
    if caption:
        fig.text(0.0, (ch / 2) / (H * unit + th + ch), caption, fontsize=9, color=COL["muted"], va="center")
    if show_now:
        plt.show()
    return fig


try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    def Markdown(s): return s


def _md(lines):
    display(Markdown("\n".join(lines) if isinstance(lines, (list, tuple)) else lines))


# ---------------------------------------------------------------------------
# Calculator engines (no UI). Two families:
#   RPNEngine  - HP-15C / HP 35s RPN behaviour: 4-level stack X Y Z T, LAST X,
#                stack-lift rules, f/g (or left/right) shifted functions.
#   AlgEngine  - algebraic entry line with textbook precedence (TI-30XS,
#                TI-84 Plus CE, Casio fx-991EX/115ES, HP 35s ALG mode).
# Keys are canonical function ids (see FUNCS); the layout tables map each
# physical key + shift state to one of these ids.
# ---------------------------------------------------------------------------
import math

_FACT_MAX = 170


def _fact(x):
    if x < 0 or abs(x - round(x)) > 1e-12:
        return math.gamma(x + 1)                       # HP-15C x! gives Γ(x+1) for non-integers
    if x > _FACT_MAX:
        raise OverflowError("overflow")
    return float(math.factorial(int(round(x))))


def _perm(n, r):
    return math.perm(int(n), int(r))


def _comb(n, r):
    return math.comb(int(n), int(r))


def _hms(h):          # decimal hours -> H.MMSS
    sign = -1 if h < 0 else 1
    h = abs(h)
    H = int(h); m = (h - H) * 60; M = int(m); s = (m - M) * 60
    return sign * (H + M / 100 + s / 10000)


def _hr(hms):         # H.MMSS -> decimal hours
    sign = -1 if hms < 0 else 1
    x = abs(hms)
    H = int(x); MM = int(round((x - H) * 100, 8)); SS = ((x - H) * 100 - MM) * 100
    return sign * (H + MM / 60 + SS / 3600)


class AngleMixin:
    angle = "DEG"

    def to_rad(self, x):
        return {"DEG": math.radians(x), "RAD": x, "GRD": x * math.pi / 200}[self.angle]

    def from_rad(self, r):
        return {"DEG": math.degrees(r), "RAD": r, "GRD": r * 200 / math.pi}[self.angle]


def fmt_number(x, mode="FIX", digits=4, width=10):
    """HP-style display formatting (FIX/SCI/ENG/ALL)."""
    if isinstance(x, str):
        return x
    if x != x:
        return "Error"
    if math.isinf(x):
        return "Overflow"
    if mode == "ALL":
        s = "%.10g" % x
        return s
    if mode == "FIX":
        if x != 0 and (abs(x) >= 1e10 or abs(x) < 10 ** (-digits)):
            return "%.*e" % (digits, x)
        return "%.*f" % (digits, x)
    if mode == "SCI":
        return "%.*e" % (digits, x)
    if mode == "ENG":
        if x == 0:
            return "%.*fe+00" % (digits, 0.0)
        e = int(math.floor(math.log10(abs(x)) / 3) * 3)
        m = x / 10 ** e
        return "%.*fe%+03d" % (max(0, digits - (len(str(int(abs(m)))) - 1)), m, e)
    return str(x)


# =============================================================================
class RPNEngine(AngleMixin):
    """HP-15C-compatible RPN core. Unsupported programming keys report a message."""

    def __init__(self, model="HP-15C"):
        self.model = model
        self.X = self.Y = self.Z = self.T = 0.0
        self.LASTX = 0.0
        self.lift = True
        self.entry = None          # string being typed, or None
        self.exp_entry = None      # exponent digits being typed after EEX
        self.regs = {}
        self.pending = None        # 'STO' | 'RCL' | 'FIX' | 'SCI' | 'ENG' (+ optional op)
        self.pending_op = None
        self.disp_mode, self.disp_digits = "FIX", 4
        self.angle = "DEG"
        self.hyp = None            # None | 'HYP' | 'AHYP'
        self.message = ""
        self.stats_clear()

    # ---------------- stack primitives
    def _lift(self):
        self.T, self.Z, self.Y = self.Z, self.Y, self.X

    def _drop(self):
        self.Y, self.Z = self.Z, self.T

    def _finish_entry(self):
        if self.entry is not None:
            self.X = self._entry_value()
            self.entry = None
            self.exp_entry = None

    def _entry_value(self):
        s = self.entry or "0"
        if s in ("", "-", ".", "-."):
            s = s.replace(".", "") + "0"
        v = float(s if not s.endswith(".") else s[:-1] or "0")
        if self.exp_entry is not None:
            e = self.exp_entry or "0"
            if e in ("-", "+"):
                e = "0"
            v = v * 10 ** int(e)
        return v

    def stats_clear(self):
        self.S = dict(n=0.0, sx=0.0, sx2=0.0, sy=0.0, sy2=0.0, sxy=0.0)

    # ---------------- display
    def display(self):
        if self.message:
            return self.message
        if self.entry is not None:
            s = self.entry if self.entry else "0"
            if self.exp_entry is not None:
                e = self.exp_entry
                sign = "-" if e.startswith("-") else ""
                e = e.lstrip("-")
                s = "%s  %s%s" % (s, sign, e.rjust(2, "0") if e else "00")
            return s
        return fmt_number(self.X, self.disp_mode, self.disp_digits)

    def stack(self):
        x = self._entry_value() if self.entry is not None else self.X
        return dict(X=x, Y=self.Y, Z=self.Z, T=self.T, LASTX=self.LASTX)

    # ---------------- key dispatcher
    def press(self, key):
        self.message = ""
        if self.pending:
            return self._pending_key(key)
        if key in "0123456789" or key == ".":
            return self._digit(key)
        h = getattr(self, "k_" + key, None)
        if h is None:
            self._finish_entry()
            self.message = "%s: not simulated" % key
            return
        h()

    # ---------------- entry keys
    def _digit(self, d):
        if self.entry is None:
            if self.lift:
                self._lift()
            self.entry = ""
            self.exp_entry = None
        if self.exp_entry is not None:
            digits = self.exp_entry.lstrip("-")
            if d == ".":
                return
            digits = (digits + d)[-2:]
            self.exp_entry = ("-" if self.exp_entry.startswith("-") else "") + digits
            return
        if d == "." and "." in self.entry:
            return
        if len(self.entry.replace("-", "").replace(".", "")) >= 10 and d != ".":
            return
        self.entry += d
        self.lift = True

    def k_EEX(self):
        if self.entry is None:
            self._digit("1")
        if self.exp_entry is None:
            self.exp_entry = ""

    def k_CHS(self):
        if self.entry is not None:
            if self.exp_entry is not None:
                self.exp_entry = self.exp_entry[1:] if self.exp_entry.startswith("-") else "-" + self.exp_entry
            else:
                self.entry = self.entry[1:] if self.entry.startswith("-") else "-" + self.entry
        else:
            self.X = -self.X
            self.lift = True

    def k_BSP(self):
        if self.entry is not None:
            if self.exp_entry is not None:
                if self.exp_entry.lstrip("-"):
                    self.exp_entry = self.exp_entry[:-1]
                else:
                    self.exp_entry = None
                return
            self.entry = self.entry[:-1]
            if self.entry in ("", "-"):
                self.entry = None
                self.X = 0.0
                self.lift = False
            return
        self.k_CLX()

    def k_ENTER(self):
        self._finish_entry()
        self._lift()
        self.lift = False

    def k_CLX(self):
        self.entry = None
        self.exp_entry = None
        self.X = 0.0
        self.lift = False

    def k_CLSTK(self):
        self.entry = None
        self.X = self.Y = self.Z = self.T = 0.0
        self.lift = False

    # ---------------- generic helpers
    def _unary(self, f):
        self._finish_entry()
        try:
            r = f(self.X)
        except (ValueError, ZeroDivisionError, OverflowError) as e:
            self.message = "Error 0" if not isinstance(e, OverflowError) else "Overflow"
            return
        if isinstance(r, complex):
            self.message = "Error 0"
            return
        self.LASTX = self.X
        self.X = r
        self.lift = True

    def _binary(self, f):
        self._finish_entry()
        try:
            r = f(self.Y, self.X)
        except (ValueError, ZeroDivisionError, OverflowError) as e:
            self.message = "Error 0" if not isinstance(e, OverflowError) else "Overflow"
            return
        if isinstance(r, complex):
            self.message = "Error 0"
            return
        self.LASTX = self.X
        self.X = r
        self._drop()
        self.lift = True

    def _push(self, v):
        self._finish_entry()
        if self.lift:
            self._lift()
        self.X = v
        self.lift = True

    # ---------------- arithmetic
    def k_ADD(self): self._binary(lambda y, x: y + x)
    def k_SUB(self): self._binary(lambda y, x: y - x)
    def k_MUL(self): self._binary(lambda y, x: y * x)
    def k_DIV(self): self._binary(lambda y, x: y / x)
    def k_YX(self): self._binary(lambda y, x: y ** x)
    def k_XROOTY(self): self._binary(lambda y, x: y ** (1.0 / x))

    def k_PCT(self):        # % : X = Y*X/100, Y kept
        self._finish_entry()
        self.LASTX = self.X
        self.X = self.Y * self.X / 100.0
        self.lift = True

    def k_DPCT(self):       # Δ% : 100 (X - Y) / Y, Y kept
        self._finish_entry()
        try:
            r = 100.0 * (self.X - self.Y) / self.Y
        except ZeroDivisionError:
            self.message = "Error 0"; return
        self.LASTX = self.X
        self.X = r
        self.lift = True

    # ---------------- unary functions
    def k_SQRT(self): self._unary(lambda x: math.sqrt(x))
    def k_SQ(self): self._unary(lambda x: x * x)
    def k_EXP(self): self._unary(math.exp)
    def k_LN(self): self._unary(math.log)
    def k_10X(self): self._unary(lambda x: 10.0 ** x)
    def k_LOG(self): self._unary(math.log10)
    def k_INV(self): self._unary(lambda x: 1.0 / x)
    def k_ABS(self): self._unary(abs)
    def k_INT(self): self._unary(lambda x: float(math.trunc(x)))
    def k_FRAC(self): self._unary(lambda x: x - math.trunc(x))
    def k_FACT(self): self._unary(_fact)
    def k_RND(self): self._unary(lambda x: float(fmt_number(x, self.disp_mode, self.disp_digits).replace("e", "E")) if self.disp_mode != "ALL" else x)
    def k_HMS(self): self._unary(_hms)
    def k_HR(self): self._unary(_hr)
    def k_TORAD(self): self._unary(math.radians)
    def k_TODEG(self): self._unary(math.degrees)
    # HP 35s conversion keys (exact factors)
    def k_TOF(self): self._unary(lambda c: c * 9.0 / 5.0 + 32.0)
    def k_TOC(self): self._unary(lambda f: (f - 32.0) * 5.0 / 9.0)
    def k_TOLB(self): self._unary(lambda kg: kg / 0.45359237)
    def k_TOKG(self): self._unary(lambda lb: lb * 0.45359237)
    def k_TOMILE(self): self._unary(lambda km: km / 1.609344)
    def k_TOKM(self): self._unary(lambda mi: mi * 1.609344)
    def k_TOIN(self): self._unary(lambda cm: cm / 2.54)
    def k_TOCM(self): self._unary(lambda inch: inch * 2.54)
    def k_TOGAL(self): self._unary(lambda l: l / 3.785411784)
    def k_TOL(self): self._unary(lambda gal: gal * 3.785411784)

    def _trig(self, name):
        hyp = self.hyp
        self.hyp = None
        if hyp == "HYP":
            f = {"SIN": math.sinh, "COS": math.cosh, "TAN": math.tanh}[name]
            return self._unary(f)
        if hyp == "AHYP":
            f = {"SIN": math.asinh, "COS": math.acosh, "TAN": math.atanh}[name]
            return self._unary(f)
        f = {"SIN": math.sin, "COS": math.cos, "TAN": math.tan}[name]
        self._unary(lambda x: f(self.to_rad(x)))

    def k_SIN(self): self._trig("SIN")
    def k_COS(self): self._trig("COS")
    def k_TAN(self): self._trig("TAN")
    def k_ASIN(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("SIN")
        self._unary(lambda x: self.from_rad(math.asin(x)))
    def k_ACOS(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("COS")
        self._unary(lambda x: self.from_rad(math.acos(x)))
    def k_ATAN(self):
        if self.hyp:
            self.hyp = "AHYP"; return self._trig("TAN")
        self._unary(lambda x: self.from_rad(math.atan(x)))
    def k_HYP(self): self._finish_entry(); self.hyp = "HYP"
    def k_AHYP(self): self._finish_entry(); self.hyp = "AHYP"

    # ---------------- stack manipulation
    def k_SWAP(self):
        self._finish_entry()
        self.X, self.Y = self.Y, self.X
        self.lift = True

    def k_RDN(self):
        self._finish_entry()
        self.X, self.Y, self.Z, self.T = self.Y, self.Z, self.T, self.X
        self.lift = True

    def k_RUP(self):
        self._finish_entry()
        self.X, self.Y, self.Z, self.T = self.T, self.X, self.Y, self.Z
        self.lift = True

    def k_LASTX(self): self._push(self.LASTX)
    def k_PI(self): self._push(math.pi)
    def k_E(self): self._push(math.e)

    # ---------------- polar / rectangular (15C: X=x? -> f→R uses X=r, Y=θ)
    def k_TOP(self):     # rectangular (x in X, y in Y) -> polar: X = r, Y = θ
        self._finish_entry()
        x, y = self.X, self.Y
        self.LASTX = self.X
        self.X = math.hypot(x, y)
        self.Y = self.from_rad(math.atan2(y, x))
        self.lift = True

    def k_TOR(self):     # polar (r in X, θ in Y) -> rectangular: X = x, Y = y
        self._finish_entry()
        r, th = self.X, self.to_rad(self.Y)
        self.LASTX = self.X
        self.X, self.Y = r * math.cos(th), r * math.sin(th)
        self.lift = True

    def k_PERM(self): self._binary(_perm)
    def k_COMB(self): self._binary(_comb)

    # ---------------- statistics (Σ registers as on HP-15C)
    def k_SIGMAP(self, sign=1):
        self._finish_entry()
        x, y = self.X, self.Y
        S = self.S
        S["n"] += sign; S["sx"] += sign * x; S["sx2"] += sign * x * x
        S["sy"] += sign * y; S["sy2"] += sign * y * y; S["sxy"] += sign * x * y
        self.LASTX = x
        self.X = S["n"]
        self.lift = False

    def k_SIGMAM(self): self.k_SIGMAP(-1)

    def _need(self, k):
        if self.S["n"] < k:
            self.message = "Error 2"
            return False
        return True

    def k_MEAN(self):
        self._finish_entry()
        if not self._need(1): return
        n = self.S["n"]
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = self.S["sx"] / n, self.S["sy"] / n
        self.lift = True

    def k_SDEV(self):
        self._finish_entry()
        if not self._need(2): return
        n, S = self.S["n"], self.S
        sx = math.sqrt((n * S["sx2"] - S["sx"] ** 2) / (n * (n - 1)))
        sy = math.sqrt(max(0.0, (n * S["sy2"] - S["sy"] ** 2) / (n * (n - 1))))
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = sx, sy
        self.lift = True

    def _lr(self):
        n, S = self.S["n"], self.S
        M = n * S["sx2"] - S["sx"] ** 2
        N = n * S["sy2"] - S["sy"] ** 2
        P = n * S["sxy"] - S["sx"] * S["sy"]
        slope = P / M
        icpt = (M * S["sy"] - P * S["sx"]) / (n * M)
        r = P / math.sqrt(M * N) if N > 0 else float("nan")
        return slope, icpt, r

    def k_LR(self):         # X = intercept, Y = slope (HP-15C L.R.)
        self._finish_entry()
        if not self._need(2): return
        slope, icpt, _ = self._lr()
        if self.lift: self._lift()
        self._lift()
        self.X, self.Y = icpt, slope
        self.lift = True

    def k_YHATR(self):      # X = ŷ at x in X, Y = r
        self._finish_entry()
        if not self._need(2): return
        slope, icpt, r = self._lr()
        self.LASTX = self.X
        yhat = icpt + slope * self.X
        self._lift()
        self.X, self.Y = yhat, r
        self.lift = True

    def k_CLSIGMA(self):
        self.stats_clear()
        self.k_CLSTK()

    # ---------------- modes and prefixes
    def k_DEG(self): self.angle = "DEG"
    def k_RAD(self): self.angle = "RAD"
    def k_GRD(self): self.angle = "GRD"
    def k_FIX(self): self._finish_entry(); self.pending = "FIX"
    def k_SCI(self): self._finish_entry(); self.pending = "SCI"
    def k_ENG(self): self._finish_entry(); self.pending = "ENG"
    def k_ALL(self): self.disp_mode = "ALL"
    def k_STO(self): self._finish_entry(); self.pending = "STO"; self.pending_op = None
    def k_RCL(self): self._finish_entry(); self.pending = "RCL"; self.pending_op = None

    def _pending_key(self, key):
        p = self.pending
        if p in ("FIX", "SCI", "ENG"):
            if key in "0123456789":
                self.disp_mode, self.disp_digits = p, int(key)
            self.pending = None
            return
        if p in ("STO", "RCL"):
            if key in ("ADD", "SUB", "MUL", "DIV") and self.pending_op is None:
                self.pending_op = key
                return
            if key == ".":
                self.pending_op = (self.pending_op or "") + "."
                return
            if key in "0123456789":
                op = self.pending_op or ""
                dot = "." if op.endswith(".") else ""
                op = op.rstrip(".")
                name = dot + key
                self.pending, self.pending_op = None, None
                if p == "STO":
                    cur = self.regs.get(name, 0.0)
                    v = {"": self.X, "ADD": cur + self.X, "SUB": cur - self.X,
                         "MUL": cur * self.X, "DIV": (cur / self.X) if self.X else float("nan")}[op]
                    self.regs[name] = v
                    self.lift = True
                else:
                    v = self.regs.get(name, 0.0)
                    if op:
                        self.LASTX = self.X
                        self.X = {"ADD": self.X + v, "SUB": self.X - v, "MUL": self.X * v,
                                  "DIV": (self.X / v) if v else float("nan")}[op]
                        self.lift = True
                    else:
                        self._push(v)
                return
            self.pending, self.pending_op = None, None
            return
        self.pending = None


# =============================================================================
class AlgEngine(AngleMixin):
    """Algebraic entry-line calculator with textbook precedence.
    Keys append tokens; '=' (ENTER/EXE) evaluates. ANS holds the last result."""

    FUNCS1 = {"sin(": "sin", "cos(": "cos", "tan(": "tan", "sin⁻¹(": "asin", "cos⁻¹(": "acos", "tan⁻¹(": "atan",
              "sinh(": "sinh", "cosh(": "cosh", "tanh(": "tanh", "sinh⁻¹(": "asinh", "cosh⁻¹(": "acosh",
              "tanh⁻¹(": "atanh", "ln(": "ln", "log(": "log", "√(": "sqrt", "e^(": "exp", "10^(": "pow10",
              "abs(": "abs", "∛(": "cbrt"}

    def __init__(self, model="TI-30XS MultiView", implicit_mult=True, neg_power_first=True):
        self.model = model
        self.line = []            # list of tokens (strings)
        self.ans = 0.0
        self.result = None
        self.history = []
        self.vars = {}
        self.angle = "DEG"
        self.just_evaluated = False
        self.message = ""
        self.implicit = implicit_mult
        self.pending_sto = False
        self.disp_mode, self.disp_digits = "ALL", 10

    # ---------------- display helpers
    def entry_text(self):
        return "".join(self.line)

    def display(self):
        if self.message:
            return self.message
        return fmt_number(self.result, self.disp_mode, self.disp_digits) if self.result is not None else ""

    # ---------------- keys
    def press(self, key):
        self.message = ""
        if self.pending_sto:
            self.pending_sto = False
            if key.startswith("VAR_"):
                v = self._evaluate(store=False)
                if v is not None:
                    self.vars[key[4:]] = v
                    self.result = v
                    self.ans = v
                    self.history.append((list(self.line), v))
                    self.just_evaluated = True
                return
        if key in ("EQ", "ENTER", "EXE"):
            if self.line == [] and self.history:
                self.line = list(self.history[-1][0])
            v = self._evaluate()
            return
        if key in ("CLEAR", "AC"):
            if self.line:
                self.line = []
            else:
                self.result = None
            self.just_evaluated = False
            return
        if key in ("DEL", "BSP"):
            if self.line:
                self.line.pop()
            return
        if key == "STO":
            self.pending_sto = True
            return
        if key == "ENTRY":               # TI-84 2nd ENTRY: recall the last entry line
            if self.history:
                self.line = list(self.history[-1][0]); self.just_evaluated = False
            return
        if key == "DEG": self.angle = "DEG"; return
        if key == "RAD": self.angle = "RAD"; return
        if key == "GRD": self.angle = "GRD"; return
        tok = self._token_for(key)
        if tok is None:
            self.message = "%s: not simulated" % key
            return
        if self.just_evaluated:
            # an operator continues from Ans; anything else starts a new line
            if tok in ("+", "−", "×", "÷", "^", "²", "³", "⁻¹", "!", "E", "%"):
                self.line = ["Ans"]
            else:
                self.line = []
            self.just_evaluated = False
        self.line.append(tok)

    TOKENS = {"ADD": "+", "SUB": "−", "MUL": "×", "DIV": "÷", "NEG": "(−)", "LP": "(", "RP": ")",
              "YX": "^", "SQ": "²", "INV": "⁻¹", "SQRT": "√(", "SIN": "sin(", "COS": "cos(", "TAN": "tan(",
              "ASIN": "sin⁻¹(", "ACOS": "cos⁻¹(", "ATAN": "tan⁻¹(", "SINH": "sinh(", "COSH": "cosh(",
              "TANH": "tanh(", "ASINH": "sinh⁻¹(", "ACOSH": "cosh⁻¹(", "ATANH": "tanh⁻¹(",
              "LN": "ln(", "LOG": "log(", "EXP": "e^(", "10X": "10^(", "PI": "π", "E": "e", "EEX": "E",
              "ANS": "Ans", "FACT": "!", "PERM": " nPr ", "COMB": " nCr ", "ABS": "abs(", "COMMA": ",",
              "PCT": "%", "XROOTY": "ˣ√(", "CBRT": "∛(", "CUBE": "³"}

    def _token_for(self, key):
        if key in "0123456789" or key == ".":
            return key
        if key.startswith("VAR_"):
            return key[4:]
        return self.TOKENS.get(key)

    # ---------------- evaluation: tokenizer + recursive descent
    def _evaluate(self, store=True):
        text = self.entry_text()
        try:
            toks = self._lex(text)
            self._toks, self._i = toks, 0
            v = self._expr()
            if self._i != len(self._toks):
                raise SyntaxError("unexpected %r" % (self._toks[self._i],))
            if isinstance(v, complex) or v != v:
                raise ValueError("domain")
        except ZeroDivisionError:
            self.message = "DIVIDE BY 0"; return None
        except (ValueError, OverflowError):
            self.message = "DOMAIN" if True else ""; return None
        except (SyntaxError, IndexError, KeyError):
            self.message = "SYNTAX ERROR"; return None
        if store:
            self.history.append((list(self.line), v))
            self.ans = v
            self.result = v
            self.just_evaluated = True
        return v

    def _lex(self, s):
        out, i = [], 0
        words = sorted(list(self.FUNCS1) + ["Ans", " nPr ", " nCr ", "(−)", "⁻¹", "ˣ√("], key=len, reverse=True)
        while i < len(s):
            c = s[i]
            if c == " " and not any(s.startswith(w, i) for w in (" nPr ", " nCr ")):
                i += 1; continue
            m = next((w for w in words if s.startswith(w, i)), None)
            if m:
                out.append(("FN", self.FUNCS1[m]) if m in self.FUNCS1 else ("SYM", m.strip()))
                i += len(m); continue
            if c in "0123456789.":
                j = i
                while j < len(s) and s[j] in "0123456789.":
                    j += 1
                num = s[i:j]
                if num.count(".") > 1:
                    raise SyntaxError("number")
                out.append(("NUM", float(num if num != "." else "0")))
                i = j; continue
            if c in "+−×÷^²³!(),%E":
                out.append(("SYM", c)); i += 1; continue
            if c == "-":
                out.append(("SYM", "−")); i += 1; continue
            if c in "πe":
                out.append(("NUM", math.pi if c == "π" else math.e)); i += 1; continue
            if c.isalpha():
                out.append(("VAR", c)); i += 1; continue
            raise SyntaxError(c)
        return out

    def _peek(self):
        return self._toks[self._i] if self._i < len(self._toks) else (None, None)

    def _take(self):
        t = self._toks[self._i]; self._i += 1; return t

    def _expr(self):
        v = self._term()
        while self._peek() in (("SYM", "+"), ("SYM", "−")):
            op = self._take()[1]
            r = self._term()
            v = v + r if op == "+" else v - r
        return v

    def _starts_factor(self):
        k, t = self._peek()
        return k in ("NUM", "FN", "VAR") or (k == "SYM" and t in ("(", "(−)", "√")) or (k == "SYM" and t == "Ans")

    def _term(self):
        v = self._factor()
        while True:
            k, t = self._peek()
            if (k, t) in (("SYM", "×"), ("SYM", "÷")):
                self._take()
                r = self._factor()
                v = v * r if t == "×" else v / r
            elif (k, t) in (("SYM", "nPr"), ("SYM", "nCr")):
                self._take()
                r = self._factor()
                v = float(_perm(v, r) if t == "nPr" else _comb(v, r))
            elif self.implicit and self._starts_factor():
                v = v * self._factor()
            else:
                return v

    def _factor(self):
        k, t = self._peek()
        if (k, t) == ("SYM", "(−)") or (k, t) == ("SYM", "−"):
            self._take()
            return -self._factor()          # negation binds looser than ^ : (−)2² = −4
        return self._power()

    def _power(self):
        base = self._postfix()
        if self._peek() == ("SYM", "^"):
            self._take()
            expo = self._factor()            # right-associative, allows (−) in exponent
            return base ** expo
        return base

    def _postfix(self):
        v = self._primary()
        while True:
            k, t = self._peek()
            if (k, t) == ("SYM", "²"):
                self._take(); v = v * v
            elif (k, t) == ("SYM", "³"):
                self._take(); v = v * v * v
            elif (k, t) == ("SYM", "⁻¹"):
                self._take(); v = 1.0 / v
            elif (k, t) == ("SYM", "!"):
                self._take(); v = _fact(v)
            elif (k, t) == ("SYM", "ˣ√("):
                self._take()
                inner = self._expr()
                if self._peek() == ("SYM", ")"):
                    self._take()
                v = inner ** (1.0 / v)
            elif (k, t) == ("SYM", "%"):
                self._take(); v = v / 100.0
            elif (k, t) == ("SYM", "E"):
                self._take()
                sign = 1
                if self._peek() in (("SYM", "(−)"), ("SYM", "−")):
                    self._take(); sign = -1
                kk, ee = self._take()
                if kk != "NUM":
                    raise SyntaxError("EE")
                v = v * 10 ** (sign * ee)
            else:
                return v

    def _primary(self):
        k, t = self._take()
        if k == "NUM":
            return t
        if k == "VAR":
            return self.vars.get(t, 0.0)
        if (k, t) == ("SYM", "Ans"):
            return self.ans
        if (k, t) == ("SYM", "("):
            v = self._expr()
            if self._peek() == ("SYM", ")"):
                self._take()
            return v                          # closing parenthesis optional at the end, as on TI/Casio
        if (k, t) == ("SYM", "ˣ√("):
            raise SyntaxError("x-root needs index first")
        if k == "FN":
            v = self._expr()
            if self._peek() == ("SYM", ")"):
                self._take()
            return self._apply(t, v)
        raise SyntaxError(t)

    def _apply(self, fn, v):
        r = self.to_rad
        f = {"sin": lambda x: math.sin(r(x)), "cos": lambda x: math.cos(r(x)), "tan": lambda x: math.tan(r(x)),
             "asin": lambda x: self.from_rad(math.asin(x)), "acos": lambda x: self.from_rad(math.acos(x)),
             "atan": lambda x: self.from_rad(math.atan(x)), "sinh": math.sinh, "cosh": math.cosh,
             "tanh": math.tanh, "asinh": math.asinh, "acosh": math.acosh, "atanh": math.atanh,
             "ln": math.log, "log": math.log10, "sqrt": math.sqrt, "exp": math.exp,
             "pow10": lambda x: 10.0 ** x, "abs": abs, "cbrt": lambda x: math.copysign(abs(x) ** (1 / 3), x)}[fn]
        return f(v)


def run_keys(engine, keys):
    """Feed a space-separated key string to an engine; returns the engine."""
    for k in keys.split():
        engine.press(k)
    return engine


# ---------------------------------------------------------------------------
# Step-by-step calculation cards (equation -> case -> worked solution)
#
#   1. Equation: name, colour-coded LaTeX equation, table of variables and
#      constants (meaning, role, units, source).
#   2. Case: pharmacy / pharmaceutical-sciences narrative with each given
#      value highlighted in the same colour as its symbol.
#   3. Solution table: general equation | values from the case | unit check
#      (prefixes expanded, cancellations noted, conversions) | substitution |
#      arithmetic steps | answer | what the answer means.
# The same card renders static Markdown (build time) and an interactive
# solver (Colab), and logs every solve to the lab report.
# ---------------------------------------------------------------------------
import math
import re

PALETTE = ["#1d6fb8", "#c2410c", "#15803d", "#7c3aed", "#b91c1c", "#0e7490", "#a16207", "#be185d"]

# ---- units ---------------------------------------------------------------
PREFIX = {"p": (-12, "pico"), "n": (-9, "nano"), "µ": (-6, "micro"), "u": (-6, "micro"),
          "m": (-3, "milli"), "c": (-2, "centi"), "d": (-1, "deci"), "k": (3, "kilo"), "M": (6, "mega")}
BASE = {"g": "gram", "L": "litre", "mol": "mole", "M": "molar (mol/L)", "Da": "dalton", "s": "second",
        "J": "joule", "cal": "calorie", "eq": "equivalent", "m": "metre", "Pa": "pascal", "U": "enzyme unit",
        "kat": "katal", "Hz": "hertz", "V": "volt", "W": "watt"}
NO_PREFIX = {"min", "h", "day", "week", "yr", "%", "mmHg", "°C", "K", "bp", "kb", "Å", "ppm", "dL"}
SPECIAL = {"dL": (-1, "L"), "mmHg": None, "kb": None}


def parse_unit_token(tok):
    """'mg' -> (exp=-3, base='g', prefix='milli'); 'min' -> (0, 'min', '')."""
    t = tok.strip()
    p = 1
    m = re.match(r"^(.*?)(\^(-?\d+)|²|³)?$", t)
    core = m.group(1)
    if m.group(2):
        p = 2 if m.group(2) == "²" else 3 if m.group(2) == "³" else int(m.group(3))
    if core in NO_PREFIX and core not in SPECIAL:
        return 0, core, "", p
    if core == "dL":
        return -1, "L", "deci", p
    if core in BASE:
        return 0, core, "", p
    if len(core) > 1 and core[0] in PREFIX and core[1:] in BASE:
        e, name = PREFIX[core[0]]
        return e, core[1:], name, p
    return 0, core, "", p


def unit_factor(unit):
    """Return (power of ten, base-unit string) for a compound unit like 'mg/dL'."""
    if not unit:
        return 0, ""
    num, _, den = unit.partition("/")
    e_tot, parts_n, parts_d = 0, [], []
    for tok in re.split(r"[·*\s]+", num.strip()):
        if tok:
            e, b, _, p = parse_unit_token(tok)
            e_tot += e * p; parts_n.append(b + ("^%d" % p if p != 1 else ""))
    for tok in re.split(r"[·*/\s]+", den.strip()):
        if tok:
            e, b, _, p = parse_unit_token(tok)
            e_tot -= e * p; parts_d.append(b + ("^%d" % p if p != 1 else ""))
    base = "·".join(parts_n) + ("/" + "·".join(parts_d) if parts_d else "")
    return e_tot, base


def _plain(x, sig=4):
    """sig significant figures without exponent notation (1e-3 <= |x| < 1e5)."""
    d = int(math.floor(math.log10(abs(x))))
    v = round(x, sig - 1 - d)
    dec = max(0, sig - 1 - d)
    t = ("%.*f" % (dec, v))
    if "." in t:
        t = t.rstrip("0").rstrip(".")
    return t


def sci(x, sig=4):
    """LaTeX scientific notation with sig figs: 1.23 \\times 10^{-4}."""
    if x == 0:
        return "0"
    if isinstance(x, int) and abs(x) < 1e5 or (abs(x) >= 1e-3 and abs(x) < 1e5):
        return _plain(x, sig)
    e = int(math.floor(math.log10(abs(x))))
    m = x / 10 ** e
    return r"%s \times 10^{%d}" % (("%.*g" % (sig, m)), e)


def fmt(x, sig=4):
    if isinstance(x, str):
        return x
    if x == 0:
        return "0"
    if abs(x) >= 1e5 or abs(x) < 1e-3:
        return sci(x, sig)
    return _plain(x, sig)


def utex(u):
    """Unit string to LaTeX (upright)."""
    if not u:
        return ""
    u = u.replace("µ", r"\mu ").replace("·", r"\cdot ").replace("%", r"\%")
    u = re.sub(r"\^(-?\d+)", r"^{\1}", u)
    u = u.strip().replace(" ", r"\ ")
    return r"\,\mathrm{%s}" % u


def expand_prefix(value, unit):
    """LaTeX showing the prefix written out, e.g. 5 mg/dL = 5 × 10⁻³ g / 10⁻¹ L."""
    e, base = unit_factor(unit)
    if e == 0:
        return None
    return r"%s%s = %s \times 10^{%d}\,\mathrm{%s} = %s%s" % (
        fmt(value), utex(unit), fmt(value), e, base.replace("µ", r"\mu "), sci(value * 10 ** e), utex(base))


# ---- card model -------------------------------------------------------------
CARDS = {}


class Var:
    def __init__(self, key, sym, name, unit="", role="input", source="", value=None, color=None, const=False):
        self.key, self.sym, self.name, self.unit, self.role = key, sym, name, unit, role
        self.source, self.value, self.color, self.const = source, value, color, const

    def tex(self):
        return r"{\color{%s}{%s}}" % (self.color, self.sym)

    def hl(self, value=None, unit=None):
        v = self.value if value is None else value
        u = self.unit if unit is None else unit
        return r"$\mathbf{{\color{%s}{%s%s}}}$" % (self.color, fmt(v), utex(u))


class Card:
    """A worked calculation. `solve(card, vals)` must return a Solution."""

    def __init__(self, id, title, equation, variables, case, solve, constants=(), conversions=(),
                 result_name="", result_unit="", source="", inputs=None):
        self.id, self.title, self.equation = id, title, equation
        self.vars = list(variables) + list(constants)
        for k, v in enumerate(self.vars):
            if v.color is None:
                v.color = "#4b5563" if v.const else PALETTE[k % len(PALETTE)]
        self.V = {v.key: v for v in self.vars}
        self.case, self.solve_fn = case, solve
        self.conversions = list(conversions)
        self.result_name, self.result_unit, self.source = result_name, result_unit, source
        self.inputs = inputs or [v.key for v in self.vars if v.role == "input" and not v.const]
        CARDS[id] = self

    # ---- LaTeX helpers
    def eq_tex(self, template=None):
        t = template or self.equation
        return re.sub(r"\{\{(\w+)\}\}", lambda m: self.V[m.group(1)].tex(), t)

    def defaults(self):
        return {v.key: v.value for v in self.vars if v.value is not None}

    def solve(self, **vals):
        d = self.defaults()
        d.update(vals)
        return self.solve_fn(self, d)

    # ---- static markdown
    def md_equation(self):
        rows = ["| Symbol | Meaning | Role in the calculation | Units | Source / value |", "|---|---|---|---|---|"]
        for v in self.vars:
            role = "constant" if v.const else v.role
            src = v.source
            if v.const and v.value is not None:
                src = "%s = %s %s; %s" % ("value", fmt(v.value), v.unit, v.source)
            rows.append("| $%s$ | %s | %s | %s | %s |" % (v.tex(), v.name, role, (v.unit or "—").replace("|", "/"), src))
        return ("**1. %s**\n\n$$%s$$\n\n" % (self.title, self.eq_tex())) + "\n".join(rows) + (
            "\n\n*Source of the relationship:* %s" % self.source if self.source else "")

    def md_case(self, vals=None):
        d = self.defaults()
        if vals:
            d.update(vals)
        txt = re.sub(r"\[\[(\w+)\]\]", lambda m: self.V[m.group(1)].hl(d.get(m.group(1))), self.case)
        return "**2. Practice problem (case)**\n\n" + txt

    def markdown(self, vals=None, collapse=True):
        sol = self.solve(**(vals or {}))
        body = sol.markdown()
        s3 = "**3. Step-by-step solution**\n\n" + body
        if collapse:
            s3 = "<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>\n\n" + body + "\n</details>"
        return self.md_equation() + "\n\n" + self.md_case(vals) + "\n\n" + s3


class Solution:
    def __init__(self, card, vals):
        self.card, self.vals, self.rows, self.result, self.unit, self.meaning = card, vals, [], None, "", ""
        self.extra_conversions = []

    # rows are (label, latex-or-markdown, kind)
    def general(self, template=None):
        self.rows.append(("General equation", "$%s$" % self.card.eq_tex(template), "tex"))
        return self

    def given(self):
        parts = []
        for k in self.card.inputs:
            v = self.card.V[k]
            parts.append("$%s = %s%s$" % (v.tex(), fmt(self.vals[k]), utex(v.unit)))
        consts = [v for v in self.card.vars if v.const]
        for v in consts:
            parts.append("$%s = %s%s$ (constant)" % (v.tex(), fmt(self.vals.get(v.key, v.value)), utex(v.unit)))
        self.rows.append(("Values from the case", "; ".join(parts), "md"))
        return self

    def unit_check(self, note=""):
        """Expand every prefix; say whether units cancel or carry through."""
        bits = []
        for v in self.card.vars:
            if v.key not in self.vals or not v.unit:
                continue
            ex = expand_prefix(self.vals[v.key], v.unit)
            if ex:
                bits.append("$%s$" % ex)
        units = sorted({v.unit for v in self.card.vars if v.key in self.vals and v.unit})
        txt = ("Prefixes written out: " + "; ".join(bits) + ". ") if bits else "No unit prefixes to expand. "
        txt += note or ("Units present: " + ", ".join(units) + ". " if units else "")
        self.rows.append(("Unit check", txt, "md"))
        return self

    def sub(self, latex, label="Substitute"):
        self.rows.append((label, "$%s$" % latex, "tex"))
        return self

    def step(self, label, latex):
        self.rows.append((label, "$%s$" % latex, "tex"))
        return self

    def answer(self, value, unit="", meaning="", label=None):
        self.result, self.unit, self.meaning = value, unit, meaning
        self.rows.append((label or "Answer", r"$\boxed{%s = %s%s}$" % (
            self.card.result_name or "x", fmt(value), utex(unit)), "tex"))
        if meaning:
            self.rows.append(("What it means", meaning, "md"))
        return self

    def convert(self, text):
        self.extra_conversions.append(text)
        return self

    def markdown(self):
        out = ["| Step | Work |", "|---|---|"]
        for k, (lab, s, kind) in enumerate(self.rows, 1):
            out.append("| %d. %s | %s |" % (k, lab, s.replace("|", r"\vert ")))
        conv = self.card.conversions + self.extra_conversions
        if conv:
            out.append("\n**Common conversions for this calculation:** " + "; ".join(conv))
        return "\n".join(out)

    def plain_lines(self):
        """[(label, latex_or_text)] for the report renderer."""
        return [(lab, s) for lab, s, _ in self.rows]


# ---- interactive solver (Colab) ------------------------------------------
def solver_widget(card, report=None, calculator=None):
    """Input boxes for the card's inputs + 'Solve step by step' + optional calculator."""
    try:
        import ipywidgets as W
        from IPython.display import display, Markdown, clear_output
    except Exception:                                     # plain Python (validator)
        sol = card.solve()
        print(sol.markdown())
        if report is not None:
            report.add_calc(card, sol)
        return sol
    boxes = {}
    for k in card.inputs:
        v = card.V[k]
        boxes[k] = W.FloatText(value=float(v.value if v.value is not None else 0.0),
                               description="%s (%s)" % (v.name[:28], v.unit or "–"),
                               style={"description_width": "230px"}, layout=W.Layout(width="420px"))
    btn = W.Button(description="Solve step by step", button_style="success", icon="list-ol",
                   layout=W.Layout(width="200px"))
    cbtn = W.Button(description="Open calculator", icon="calculator", layout=W.Layout(width="200px"))
    out, cout = W.Output(), W.Output()

    def go(_=None):
        with out:
            clear_output()
            try:
                sol = card.solve(**{k: b.value for k, b in boxes.items()})
            except (ValueError, ZeroDivisionError) as e:
                print("Check the inputs:", e)
                return
            display(Markdown(card.md_case({k: b.value for k, b in boxes.items()}) + "\n\n" + sol.markdown()))
            if report is not None:
                report.add_calc(card, sol)

    def calc(_=None):
        with cout:
            clear_output()
            if calculator is not None:
                calculator()
    btn.on_click(go)
    cbtn.on_click(calc)
    display(W.VBox(list(boxes.values()) + [W.HBox([btn, cbtn]), out, cout]))
    go()


# ---------------------------------------------------------------------------
# Lab report collector: every figure shown, every step-by-step calculation
# and every self-check answer is logged in this browser session; at the end
# the student downloads a formatted report (.html and .docx). Nothing is
# sent anywhere; files are created only when the student asks.
# ---------------------------------------------------------------------------
import io, re, base64, zipfile, datetime, html as _html

_LATEX_FIX = [(r"\\textcolor\{[^}]*\}", ""), (r"\\color\{[^}]*\}", ""),
              (r"\\text\{", r"\\mathrm{"), (r"\\boxed\{", "{"), (r"\\quad", r"\ \ "), (r"\\%", r"\%"),
              (r"\\vert", "|"), (r"\\mathbf\{", r"\\mathbf{")]


def latex_png(tex, fontsize=12, dpi=170):
    """Render one LaTeX line to PNG bytes with matplotlib mathtext (colours stripped)."""
    import matplotlib
    import matplotlib.pyplot as plt
    t = tex
    for a, b in _LATEX_FIX:
        t = re.sub(a, b, t)
    t = t.replace(r"\,", r"\ ")
    fig = plt.figure(figsize=(0.01, 0.01))
    try:
        fig.text(0, 0, "$%s$" % t, fontsize=fontsize)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight", pad_inches=0.04, transparent=False,
                    facecolor="white")
        return buf.getvalue()
    except Exception:
        return None
    finally:
        plt.close(fig)


def _split_math(s):
    """Split '... $a$ ... $b$' into [('text', ..), ('math', ..)]."""
    out, pos = [], 0
    for m in re.finditer(r"\$([^$]+)\$", s):
        if m.start() > pos:
            out.append(("text", s[pos:m.start()]))
        out.append(("math", m.group(1)))
        pos = m.end()
    if pos < len(s):
        out.append(("text", s[pos:]))
    return out


class LabReport:
    def __init__(self, course, session, title):
        self.course, self.session, self.title = course, session, title
        self.entries = []          # dicts: kind, section, title, ...
        self.section = ""
        self.student = ""
        self.started = datetime.datetime.now()

    # ---- collection
    def set_section(self, label):
        self.section = label

    def add_figure(self, fig, title=None):
        buf = io.BytesIO()
        try:
            fig.savefig(buf, format="png", dpi=110, bbox_inches="tight", facecolor="white")
        except Exception:
            return
        t = title
        if not t:
            st = getattr(fig, "_suptitle", None)
            t = st.get_text() if st is not None else ""
            if not t:
                for ax in fig.axes:
                    tt = ax.get_title() or ax.get_title(loc="left") or ax.get_title(loc="right")
                    if tt:
                        t = tt; break
            if not t and fig.texts:
                t = fig.texts[0].get_text()
        self.entries.append(dict(kind="figure", section=self.section, title=t or "Figure", png=buf.getvalue()))

    def add_calc(self, card, sol):
        self.entries.append(dict(kind="calc", section=self.section, title=card.title, card_id=card.id,
                                 case=card.md_case(sol.vals), rows=sol.plain_lines(),
                                 result=sol.result, unit=sol.unit, meaning=sol.meaning))

    def add_answer(self, qid, stem, choice, correct, feedback=""):
        self.entries.append(dict(kind="answer", section=self.section, qid=qid, stem=stem, choice=choice,
                                 correct=correct, feedback=feedback))

    def add_note(self, title, text):
        self.entries.append(dict(kind="note", section=self.section, title=title, text=text))

    def counts(self):
        c = {}
        for e in self.entries:
            c[e["kind"]] = c.get(e["kind"], 0) + 1
        return c

    # ---- HTML
    def to_html(self, summary_md=""):
        css = ("body{font-family:Georgia,serif;max-width:860px;margin:2em auto;color:#1f2430;line-height:1.5}"
               "h1{font-size:1.6em}h2{border-bottom:2px solid #1d6fb8;padding-bottom:4px;margin-top:1.6em}"
               "table{border-collapse:collapse;width:100%;margin:.6em 0}td,th{border:1px solid #cbd5e1;padding:6px;"
               "vertical-align:top;font-size:.95em}th{background:#eef4fb}.fig{text-align:center;margin:1em 0}"
               ".fig img{max-width:100%}.cap{font-size:.9em;color:#4b5563}.ok{color:#15803d}.no{color:#b91c1c}"
               ".meta{color:#4b5563}")
        h = ['<!doctype html><html><head><meta charset="utf-8"><title>%s</title><style>%s</style>'
             '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.9/MathJax.js?config=TeX-AMS_HTML"></script>'
             '<script type="text/x-mathjax-config">MathJax.Hub.Config({tex2jax:{inlineMath:[["$","$"]]}});</script>'
             '</head><body>' % (_html.escape(self.title), css)]
        h.append("<h1>%s · %s</h1><p class='meta'>%s<br>Student: %s<br>Generated %s</p>" % (
            _html.escape(self.course), _html.escape(self.title), _html.escape(self.session),
            _html.escape(self.student or "________________"), datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
        sec = None
        nfig = ncalc = 0
        for e in self.entries:
            if e["section"] != sec:
                sec = e["section"]
                h.append("<h2>%s</h2>" % _html.escape(sec or "Work"))
            if e["kind"] == "figure":
                nfig += 1
                h.append("<div class='fig'><img src='data:image/png;base64,%s'><div class='cap'>Figure %d. %s</div></div>"
                         % (base64.b64encode(e["png"]).decode(), nfig, _html.escape(e["title"])))
            elif e["kind"] == "calc":
                ncalc += 1
                h.append("<h3>Calculation %d. %s</h3><p>%s</p><table><tr><th>Step</th><th>Work</th></tr>" % (
                    ncalc, _html.escape(e["title"]), e["case"].replace("**2. Practice problem (case)**", "").strip()))
                for k, (lab, s) in enumerate(e["rows"], 1):
                    h.append("<tr><td>%d. %s</td><td>%s</td></tr>" % (k, _html.escape(lab), s))
                h.append("</table>")
            elif e["kind"] == "answer":
                h.append("<p><b>%s.</b> %s<br>Answer: %s <span class='%s'>(%s)</span></p>" % (
                    _html.escape(e["qid"]), _html.escape(e["stem"]), _html.escape(e["choice"]),
                    "ok" if e["correct"] else "no", "correct" if e["correct"] else "review"))
            elif e["kind"] == "note":
                h.append("<h3>%s</h3><p>%s</p>" % (_html.escape(e["title"]), _html.escape(e["text"])))
        if summary_md:
            h.append("<h2>Summary</h2><pre>%s</pre>" % _html.escape(summary_md))
        h.append("</body></html>")
        return "\n".join(h)

    # ---- DOCX (minimal WordprocessingML writer; equations as images)
    def to_docx(self):
        D = _Docx()
        D.heading("%s · %s" % (self.course, self.title), 1)
        D.para("%s    Student: %s    Generated %s" % (self.session, self.student or "________________",
                                                      datetime.datetime.now().strftime("%Y-%m-%d %H:%M")), size=18)
        sec = None
        nfig = ncalc = 0
        for e in self.entries:
            if e["section"] != sec:
                sec = e["section"]
                D.heading(sec or "Work", 2)
            if e["kind"] == "figure":
                nfig += 1
                D.image(e["png"], width_in=6.0)
                D.para("Figure %d. %s" % (nfig, e["title"]), size=18, italic=True)
            elif e["kind"] == "calc":
                ncalc += 1
                D.heading("Calculation %d. %s" % (ncalc, e["title"]), 3)
                D.rich(e["case"].replace("**2. Practice problem (case)**", "").strip())
                rows = []
                for k, (lab, s) in enumerate(e["rows"], 1):
                    rows.append(("%d. %s" % (k, lab), s))
                D.table(rows)
            elif e["kind"] == "answer":
                D.para("%s. %s — answer %s (%s)" % (e["qid"], e["stem"], e["choice"],
                                                     "correct" if e["correct"] else "review"), size=20)
            elif e["kind"] == "note":
                D.heading(e["title"], 3)
                D.para(e["text"])
        return D.build()

    # ---- delivery
    def download(self, basename=None):
        base = basename or ("%s_%s_lab_report" % (self.course.replace(" ", ""), self.session.split()[0]))
        html_bytes = self.to_html().encode("utf-8")
        docx_bytes = self.to_docx()
        with open(base + ".html", "wb") as f:
            f.write(html_bytes)
        with open(base + ".docx", "wb") as f:
            f.write(docx_bytes)
        try:
            from google.colab import files
            files.download(base + ".docx")
            files.download(base + ".html")
            return "Downloading %s.docx and %s.html" % (base, base)
        except Exception:
            return "Saved %s.docx and %s.html in the notebook's folder" % (base, base)


# ---------------------------------------------------------------------------
_W = 'xmlns:w="http://schemas.openxmlformats.org/wordprocessingml/2006/main" ' \
     'xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships" ' \
     'xmlns:wp="http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing" ' \
     'xmlns:a="http://schemas.openxmlformats.org/drawingml/2006/main" ' \
     'xmlns:pic="http://schemas.openxmlformats.org/drawingml/2006/picture"'


def _x(s):
    return _html.escape(s, quote=False)


class _Docx:
    def __init__(self):
        self.body, self.media = [], []

    def _run(self, text, bold=False, italic=False, size=None, color=None):
        rpr = ""
        if bold: rpr += "<w:b/>"
        if italic: rpr += "<w:i/>"
        if color: rpr += '<w:color w:val="%s"/>' % color.lstrip("#")
        if size: rpr += '<w:sz w:val="%d"/>' % size
        return '<w:r>%s<w:t xml:space="preserve">%s</w:t></w:r>' % ("<w:rPr>%s</w:rPr>" % rpr if rpr else "", _x(text))

    def heading(self, text, level=1):
        size = {1: 36, 2: 28, 3: 24}.get(level, 22)
        self.body.append('<w:p><w:pPr><w:spacing w:before="240" w:after="80"/></w:pPr>%s</w:p>'
                         % self._run(text, bold=True, size=size, color="1d3557" if level < 3 else None))

    def para(self, text, size=22, italic=False):
        self.body.append("<w:p>%s</w:p>" % self._run(text, size=size, italic=italic))

    def _img_run(self, png, width_in):
        from PIL import Image
        im = Image.open(io.BytesIO(png))
        w, h = im.size
        cx = int(width_in * 914400)
        cy = int(cx * h / w)
        self.media.append(png)
        rid = "rIdImg%d" % len(self.media)
        n = len(self.media)
        return ('<w:r><w:drawing><wp:inline><wp:extent cx="%d" cy="%d"/><wp:docPr id="%d" name="img%d"/>'
                '<a:graphic><a:graphicData uri="http://schemas.openxmlformats.org/drawingml/2006/picture">'
                '<pic:pic><pic:nvPicPr><pic:cNvPr id="%d" name="img%d.png"/><pic:cNvPicPr/></pic:nvPicPr>'
                '<pic:blipFill><a:blip r:embed="%s"/><a:stretch><a:fillRect/></a:stretch></pic:blipFill>'
                '<pic:spPr><a:xfrm><a:off x="0" y="0"/><a:ext cx="%d" cy="%d"/></a:xfrm>'
                '<a:prstGeom prst="rect"><a:avLst/></a:prstGeom></pic:spPr></pic:pic></a:graphicData></a:graphic>'
                '</wp:inline></w:drawing></w:r>' % (cx, cy, n, n, n, n, rid, cx, cy))

    def image(self, png, width_in=6.0):
        self.body.append('<w:p><w:pPr><w:jc w:val="center"/></w:pPr>%s</w:p>' % self._img_run(png, width_in))

    def _rich_runs(self, s, size=22):
        runs = []
        for kind, part in _split_math(s):
            if kind == "text":
                txt = re.sub(r"\*\*(.*?)\*\*", r"\1", part)
                runs.append(self._run(txt, size=size))
            else:
                png = latex_png(part, fontsize=11)
                if png:
                    from PIL import Image
                    w, h = Image.open(io.BytesIO(png)).size
                    runs.append(self._img_run(png, min(6.0, w / 170.0)))
                else:
                    runs.append(self._run(part, size=size, italic=True))
        return "".join(runs)

    def rich(self, s):
        self.body.append("<w:p>%s</w:p>" % self._rich_runs(s))

    def table(self, rows):
        trs = []
        for a, b in rows:
            trs.append('<w:tr><w:tc><w:tcPr><w:tcW w:w="2400" w:type="dxa"/></w:tcPr><w:p>%s</w:p></w:tc>'
                       '<w:tc><w:tcPr><w:tcW w:w="7000" w:type="dxa"/></w:tcPr><w:p>%s</w:p></w:tc></w:tr>'
                       % (self._run(a, bold=True, size=20), self._rich_runs(b, size=20)))
        borders = "".join('<w:%s w:val="single" w:sz="4" w:space="0" w:color="CBD5E1"/>' % e
                          for e in ("top", "left", "bottom", "right", "insideH", "insideV"))
        self.body.append('<w:tbl><w:tblPr><w:tblW w:w="9400" w:type="dxa"/><w:tblBorders>%s</w:tblBorders>'
                         '</w:tblPr><w:tblGrid><w:gridCol w:w="2400"/><w:gridCol w:w="7000"/></w:tblGrid>%s</w:tbl>'
                         '<w:p/>' % (borders, "".join(trs)))

    def build(self):
        doc = ('<?xml version="1.0" encoding="UTF-8" standalone="yes"?><w:document %s><w:body>%s'
               '<w:sectPr><w:pgSz w:w="12240" w:h="15840"/><w:pgMar w:top="1080" w:right="1080" w:bottom="1080" '
               'w:left="1080" w:header="720" w:footer="720" w:gutter="0"/></w:sectPr></w:body></w:document>'
               % (_W, "".join(self.body)))
        rels = ['<Relationship Id="rIdImg%d" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/image" '
                'Target="media/image%d.png"/>' % (k, k) for k in range(1, len(self.media) + 1)]
        buf = io.BytesIO()
        with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
            z.writestr("[Content_Types].xml",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">'
                       '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>'
                       '<Default Extension="xml" ContentType="application/xml"/>'
                       '<Default Extension="png" ContentType="image/png"/>'
                       '<Override PartName="/word/document.xml" '
                       'ContentType="application/vnd.openxmlformats-officedocument.wordprocessingml.document.main+xml"/>'
                       '</Types>')
            z.writestr("_rels/.rels",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">'
                       '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" '
                       'Target="word/document.xml"/></Relationships>')
            z.writestr("word/_rels/document.xml.rels",
                       '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>'
                       '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">%s'
                       '</Relationships>' % "".join(rels))
            z.writestr("word/document.xml", doc)
            for k, png in enumerate(self.media, 1):
                z.writestr("word/media/image%d.png" % k, png)
        return buf.getvalue()


# ---------------------------------------------------------------------------
# Pathway figure kit (BioRender-inspired, matplotlib only).
# Muted palette, rounded compartments, lipid-bilayer membranes, short labels,
# arrows = conversion/activation, T-bars = inhibition, dashed = indirect,
# hexagon badges = drugs. Coordinates are in a 0-100 x 0-60 canvas.
# ---------------------------------------------------------------------------
import math
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, RegularPolygon, FancyArrowPatch, Ellipse, Rectangle
from matplotlib.lines import Line2D

PW = dict(teal="#2a9d8f", blue="#3a6ea5", coral="#e76f51", gold="#e9b44c", green="#5a9e6f",
          purple="#7b6fb0", grey="#6b7280", ink="#1f2937", paper="#ffffff",
          cyto="#f3f8fb", mito="#fdf3ea", nucleus="#f1eef8", lumen="#eef7f1", er="#fbf1f4",
          blood="#fdeeee", extracell="#f7f7f2")
KIND = {  # node fill, edge, text colour, shape
    "metabolite": (PW["paper"], PW["blue"], PW["ink"], "round"),
    "enzyme": ("#e3f1ef", PW["teal"], PW["ink"], "pill"),
    "transporter": ("#fbe9e3", PW["coral"], PW["ink"], "pill"),
    "receptor": ("#efeaf7", PW["purple"], PW["ink"], "pill"),
    "gene": ("#fdf5e1", PW["gold"], PW["ink"], "square"),
    "cofactor": (PW["paper"], PW["grey"], PW["grey"], "round"),
    "hormone": ("#e8f3ea", PW["green"], PW["ink"], "round"),
}


class Pathway:
    def __init__(self, title="", w=10.5, h=6.0, xlim=(0, 100), ylim=(0, 60)):
        self.fig, self.ax = plt.subplots(figsize=(w, h))
        self.ax.set_xlim(*xlim); self.ax.set_ylim(*ylim); self.ax.set_aspect("equal"); self.ax.axis("off")
        self.fig.patch.set_facecolor("white")
        if title:
            self.ax.set_title(title, fontsize=13, color=PW["ink"], loc="left", fontweight="bold", pad=10)
        self.nodes = {}
        self._legend = set()

    # ---- containers
    def compartment(self, x, y, w, h, label="", kind="cyto", membrane=True, label_pos="tl"):
        fill = PW.get(kind, kind)
        self.ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=%.1f" % min(4, h / 4),
                                         fc=fill, ec="#c9d3dd" if not membrane else "#d9a441", lw=0.8 if not membrane else 3.2,
                                         zorder=0))
        if membrane:   # inner line gives a bilayer look
            self.ax.add_patch(FancyBboxPatch((x + 0.6, y + 0.6), w - 1.2, h - 1.2,
                                             boxstyle="round,pad=0,rounding_size=%.1f" % max(0.5, min(4, h / 4) - 0.6),
                                             fc="none", ec="#f2d48f", lw=1.4, zorder=0))
        if label:
            tx, ha = (x + 1.6, "left") if "l" in label_pos else (x + w - 1.6, "right")
            ty, va = (y + h - 1.4, "top") if "t" in label_pos else (y + 1.4, "bottom")
            self.ax.text(tx, ty, label, ha=ha, va=va, fontsize=9, color=PW["grey"], style="italic", zorder=1)

    def membrane(self, x0, y0, x1, y1, label=""):
        """Straight lipid bilayer (two rows of heads) between two points."""
        n = max(4, int(math.hypot(x1 - x0, y1 - y0) / 1.3))
        dx, dy = (x1 - x0), (y1 - y0)
        L = math.hypot(dx, dy); nx, ny = -dy / L, dx / L
        for k in range(n + 1):
            px, py = x0 + dx * k / n, y0 + dy * k / n
            for s in (-1, 1):
                self.ax.add_patch(Circle((px + s * 0.9 * nx, py + s * 0.9 * ny), 0.42, fc="#e9b44c", ec="#c9902a", lw=0.4, zorder=1))
            self.ax.plot([px - 0.6 * nx, px + 0.6 * nx], [py - 0.6 * ny, py + 0.6 * ny], color="#d8c08a", lw=0.6, zorder=0.9)
        if label:
            self.ax.text(x1, y1 + 1.2, label, fontsize=8, color=PW["grey"], ha="right", style="italic")

    # ---- nodes
    def node(self, key, x, y, label, kind="metabolite", w=None, fs=9.5):
        fc, ec, tc, shape = KIND[kind]
        w = w or max(7.0, 1.15 * len(max(label.split("\n"), key=len)) + 2.6)
        h = 3.6 + 2.2 * label.count("\n")
        rs = {"round": h / 2, "pill": h / 2, "square": 0.6}[shape]
        self.ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0,rounding_size=%.2f" % rs,
                                         fc=fc, ec=ec, lw=1.6 if kind != "metabolite" else 1.2, zorder=3))
        self.ax.text(x, y, label, ha="center", va="center", fontsize=fs, color=tc, zorder=4,
                     fontweight="bold" if kind in ("enzyme", "transporter", "receptor") else "normal",
                     style="italic" if kind == "gene" else "normal")
        self.nodes[key] = (x, y, w, h)
        self._legend.add(kind)
        return key

    def drug(self, key, x, y, label, r=2.6):
        self.ax.add_patch(RegularPolygon((x, y), 6, radius=r, orientation=math.pi / 6, fc=PW["coral"], ec="#a8432b",
                                         lw=1.2, zorder=5))
        self.ax.text(x, y - r - 0.9, label, ha="center", va="top", fontsize=8.6, color="#a8432b", fontweight="bold", zorder=5)
        self.ax.text(x, y, "Rx", ha="center", va="center", fontsize=7.5, color="white", fontweight="bold", zorder=6)
        self.nodes[key] = (x, y, 2 * r, 2 * r)
        self._legend.add("drug")
        return key

    def _edge_pt(self, key, toward):
        x, y, w, h = self.nodes[key]
        tx, ty = toward
        dx, dy = tx - x, ty - y
        if dx == 0 and dy == 0:
            return x, y
        sx = (w / 2 + 0.5) / abs(dx) if dx else 1e9
        sy = (h / 2 + 0.5) / abs(dy) if dy else 1e9
        s = min(sx, sy)
        return x + dx * s, y + dy * s

    # ---- edges
    def arrow(self, a, b, kind="convert", label="", rad=0.0, color=None, lw=1.7, label_side=1):
        pa = self.nodes[a][:2] if isinstance(a, str) else a
        pb = self.nodes[b][:2] if isinstance(b, str) else b
        p0 = self._edge_pt(a, pb) if isinstance(a, str) else pa
        p1 = self._edge_pt(b, pa) if isinstance(b, str) else pb
        col = color or {"convert": PW["ink"], "activate": PW["green"], "inhibit": PW["coral"],
                        "indirect": PW["grey"], "transport": PW["blue"]}[kind]
        ls = "--" if kind == "indirect" else "-"
        style = "-[" if kind == "inhibit" else "-|>"
        self.ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle=style if kind != "inhibit" else "-",
                                          mutation_scale=13, color=col, lw=lw, ls=ls,
                                          connectionstyle="arc3,rad=%.2f" % rad, zorder=2, shrinkA=0, shrinkB=0))
        if kind == "inhibit":   # T-bar
            dx, dy = p1[0] - p0[0], p1[1] - p0[1]
            L = math.hypot(dx, dy) or 1
            nx, ny = -dy / L * 1.5, dx / L * 1.5
            self.ax.plot([p1[0] - nx, p1[0] + nx], [p1[1] - ny, p1[1] + ny], color=col, lw=2.6, zorder=2,
                         solid_capstyle="round")
        if label:
            mx, my = (p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2
            dx, dy = p1[0] - p0[0], p1[1] - p0[1]
            L = math.hypot(dx, dy) or 1
            ox, oy = -dy / L * 1.6 * label_side, dx / L * 1.6 * label_side
            if rad:
                ox += -dy * rad * 0.5; oy += dx * rad * 0.5
            self.ax.text(mx + ox, my + oy, label, fontsize=8.2, color=col, ha="center", va="center", zorder=6,
                         bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
        self._legend.add(kind)

    def catalyze(self, enzyme, a, b, label_side=1):
        """Metabolite a -> b with enzyme drawn beside the arrow and a short dashed tie to its midpoint."""
        self.arrow(a, b, "convert")
        xa, ya = self.nodes[a][:2]; xb, yb = self.nodes[b][:2]
        xe, ye = self.nodes[enzyme][:2]
        self.ax.add_patch(FancyArrowPatch((xe, ye), ((xa + xb) / 2, (ya + yb) / 2), arrowstyle="-", color=PW["teal"],
                                          lw=1.0, ls=(0, (2, 2)), zorder=1, shrinkA=10, shrinkB=2))

    def note(self, x, y, text, fs=8.4, color=None, ha="left"):
        import textwrap
        xmax = self.ax.get_xlim()[1]
        room = (xmax - x) if ha == "left" else (x if ha == "right" else 2 * min(x, xmax - x))
        width = max(12, int(room * 1.55 * 8.4 / fs))
        text = "\n".join(textwrap.fill(line, width) for line in text.split("\n"))
        self.ax.text(x, y, text, fontsize=fs, color=color or PW["grey"], ha=ha, va="top", zorder=6)

    def legend(self, loc="below"):
        items = []
        names = {"convert": ("conversion", PW["ink"], "-"), "activate": ("activates", PW["green"], "-"),
                 "inhibit": ("inhibits (T-bar)", PW["coral"], "-"), "indirect": ("indirect / multistep", PW["grey"], "--"),
                 "transport": ("transport", PW["blue"], "-")}
        for k in ("convert", "activate", "inhibit", "indirect", "transport"):
            if k in self._legend:
                n, c, ls = names[k]
                items.append(Line2D([0], [0], color=c, lw=2, ls=ls, label=n))
        for k in ("enzyme", "transporter", "receptor", "gene", "hormone"):
            if k in self._legend:
                fc, ec, _, _ = KIND[k]
                items.append(Rectangle((0, 0), 1, 1, fc=fc, ec=ec, lw=1.4, label=k))
        if "drug" in self._legend:
            items.append(Line2D([0], [0], marker="h", color="w", markerfacecolor=PW["coral"], markersize=11, label="drug"))
        if items and loc == "below":
            self.ax.legend(handles=items, loc="upper center", bbox_to_anchor=(0.5, -0.01), fontsize=7.8,
                           frameon=False, ncol=min(6, len(items)))
        elif items:
            self.ax.legend(handles=items, loc=loc, fontsize=7.8, frameon=True, framealpha=0.95, edgecolor="#d1d5db",
                           ncol=2 if len(items) > 4 else 1)

    def show(self, report=None):
        self.fig.tight_layout()
        plt.show()
        return self.fig


def demo_orlistat():
    """Example: dietary TAG digestion, absorption and the orlistat block."""
    P = Pathway("Fat digestion in the small-intestine lumen and the orlistat block", 10.5, 6.2)
    P.compartment(2, 30, 62, 27, "intestinal lumen", "lumen", membrane=False)
    P.compartment(66, 4, 32, 53, "enterocyte", "cyto")
    P.compartment(2, 3, 62, 24, "pancreas → duodenum", "extracell", membrane=False, label_pos="bl")
    P.node("tag", 10, 48, "TAG", "metabolite")
    P.node("bile", 10, 38, "bile salts", "cofactor")
    P.node("mic", 32, 48, "emulsion\ndroplet", "metabolite")
    P.node("mag", 48, 48, "2-MAG +\n2 FFA", "metabolite")
    P.node("pl", 32, 14, "pancreatic\nlipase", "enzyme")
    P.node("col", 50, 14, "colipase", "enzyme")
    P.node("cd36", 66, 36, "FATP4 /\nCD36", "transporter")
    P.node("rtag", 82, 34, "TAG", "metabolite")
    P.node("cm", 82, 16, "chylomicron", "metabolite")
    P.drug("orl", 18, 22, "orlistat")
    P.arrow("tag", "mic", "convert")
    P.arrow("bile", "mic", "activate", "emulsifies", rad=-0.2, label_side=-1)
    P.arrow("mic", "mag", "convert", "lipolysis")
    P.arrow("pl", "mic", "activate", rad=0.15)
    P.arrow("col", "pl", "activate", "anchors", label_side=-1)
    P.arrow("orl", "pl", "inhibit", "covalent Ser", label_side=1)
    P.arrow("mag", "cd36", "transport")
    P.arrow("cd36", "rtag", "indirect", "re-esterified", label_side=-1)
    P.arrow("rtag", "cm", "convert", "+ apoB-48")
    P.note(3, 2.2, "Unabsorbed TAG passes to stool (steatorrhea): measured as fecal fat.", fs=8)
    P.legend()
    return P.show()


import json as _json
CALC_LAYOUTS = _json.loads('{"HP-15C": {"sources": ["https://finseth.com/hpdata/hp15c.php", "https://www.manualslib.com/manual/558055/Hp-Hp-15c.html", "https://manualshive.com/en/manual/hp/15c/156479?page=10"], "shift_keys": {"f": {"color": "#E3A63A", "label_position": "above the key, printed on the faceplate (gold)"}, "g": {"color": "#5AA7D6", "label_position": "on the slanted front face of the key, below the main legend (blue)"}}, "grid": {"rows": 4, "cols": 10}, "keys": [{"row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "√x", "shift1": "A", "shift2": "x²", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "eˣ", "shift1": "B", "shift2": "LN", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "10ˣ", "shift1": "C", "shift2": "LOG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "yˣ", "shift1": "D", "shift2": "%", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "1/x", "shift1": "E", "shift2": "Δ%", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue legend is Δ% (percent change); Finseth renders it \'%CHG\'."}, {"row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "CHS", "shift1": "MATRIX", "shift2": "ABS", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 6, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "FIX", "shift2": "DEG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 7, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "SCI", "shift2": "RAD", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 8, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "ENG", "shift2": "GRD", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 0, "col": 9, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "SOLVE", "shift2": "x≤y", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "SST", "shift1": "LBL", "shift2": "BST", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "GTO", "shift1": "HYP", "shift2": "HYP⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue legend is HYP⁻¹ (inverse hyperbolic); Finseth\'s text shows it as a second \'HYP\' because the superscript is lost."}, {"row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "SIN", "shift1": "DIM", "shift2": "SIN⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "COS", "shift1": "(i)", "shift2": "COS⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "TAN", "shift1": "I", "shift2": "TAN⁻¹", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "EEX", "shift1": "RESULT", "shift2": "π", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 6, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "x≷", "shift2": "SF", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend is x≷ (exchange X with a register), often written x<>."}, {"row": 1, "col": 7, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "DSE", "shift2": "CF", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 8, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "ISG", "shift2": "F?", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 1, "col": 9, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "∫ᵧˣ", "shift2": "x=0", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend is the integral sign ∫ with limits y and x (∫ᵧˣ)."}, {"row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "R/S", "shift1": "PSE", "shift2": "P/R", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "GSB", "shift1": "Σ", "shift2": "RTN", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR Σ."}, {"row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "R↓", "shift1": "PRGM", "shift2": "R↑", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR PRGM."}, {"row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "x≷y", "shift1": "REG", "shift2": "RND", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR REG."}, {"row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "←", "shift1": "PREFIX", "shift2": "CLx", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Gold legend sits under the gold CLEAR bracket, so the full f-function is CLEAR PREFIX."}, {"row": 2, "col": 5, "rowspan": 2, "colspan": 1, "main": "ENTER", "shift1": "RAN #", "shift2": "LSTx", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "Tall ENTER: occupies column 5 of rows 2 and 3."}, {"row": 2, "col": 6, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "→R", "shift2": "→P", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 7, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "→H.MS", "shift2": "→H", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 8, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "→RAD", "shift2": "→DEG", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 2, "col": 9, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "Re≷Im", "shift2": "TEST", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "ON", "shift1": "", "shift2": "", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": "No shifted functions. Held with other keys at power-on for self-tests / decimal-point swap."}, {"row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "f", "shift1": "", "shift2": "", "alpha": "", "key_color": "#E3A63A", "text_color": "#1E1E1E", "uncertain": false, "note": "Gold prefix key; activates the gold legends printed above the keys."}, {"row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "g", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5AA7D6", "text_color": "#1E1E1E", "uncertain": false, "note": "Blue prefix key; activates the blue legends printed on the slanted front face of the keys."}, {"row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "STO", "shift1": "FRAC", "shift2": "INT", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "RCL", "shift1": "USER", "shift2": "MEM", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 6, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "x!", "shift2": "x̄", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 7, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "ŷ,r", "shift2": "s", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 8, "rowspan": 1, "colspan": 1, "main": "Σ+", "shift1": "L.R.", "shift2": "Σ−", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"row": 3, "col": 9, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "Py,x", "shift2": "Cy,x", "alpha": "", "key_color": "#1E1E1E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}], "notes": "Landscape (Voyager) layout: 4 rows x 10 columns, 39 physical keys. ENTER is the tall key in column 5 spanning rows 2-3, so row 3 has no separate key at column 5. Bottom-left of row 3 holds ON, f (gold key) and g (blue key). Main legends are white on black keys; f-legends are gold on the faceplate above each key; g-legends are blue on the key\'s front face. A gold \'CLEAR\' bracket spans above GSB, R↓, x≷y and ← (CLEAR Σ, CLEAR PRGM, CLEAR REG, CLEAR PREFIX). The HP 15C Collector\'s Edition (2023) keeps the identical layout and legends. Hex colours are approximations of the printed/moulded colours, not official values."}, "HP 35s": {"sources": ["https://finseth.com/hpdata/hp35s.php", "https://stefanv.com/calculators/hp35s.html", "https://hpmuseum.org/forum/post-7043.html", "https://www.hpmuseum.org/forum/post-39775.html", "https://literature.hpcalc.org/official/hp35s-qsg-en.pdf", "https://www.manualslib.com/manual/257003/Hp-35s.html?page=18"], "shift_keys": {"left_shift": {"color": "#F2B33D", "label_position": "above the key, printed on the faceplate (yellow)"}, "right_shift": {"color": "#3D8BD4", "label_position": "on the bottom/front face of the key (blue)"}, "alpha": {"color": "#E0457B", "label_position": "small red/pink letter at the bottom-right corner of the key (used after STO/RCL/XEQ/GTO, no ALPHA key)"}}, "sections": [{"name": "top", "rows": 2, "cols": 4, "placement": "upper-left block"}, {"name": "cursor", "rows": 3, "cols": 3, "placement": "right of \'top\', 4 keys in a diamond (only (0,1),(1,0),(1,2),(2,1) used)"}, {"name": "upper", "rows": 2, "cols": 6, "placement": "full width below top/cursor"}, {"name": "lower", "rows": 4, "cols": 5, "placement": "full width, bottom"}], "grid": {"rows": 10, "cols": 6, "note": "overall; use \'sections\' for exact positions"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "R/S", "shift1": "FN=", "shift2": "PRGM", "alpha": "A", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "GTO", "shift1": "ISG", "shift2": "DSE", "alpha": "B", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "XEQ", "shift1": "RTN", "shift2": "LBL", "alpha": "C", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "MODE", "shift1": "x?y", "shift2": "x?0", "alpha": "D", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source). MODE is a menu key."}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "RCL", "shift1": "x≷", "shift2": "STO", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "STO is the blue (right-shifted) function of RCL - confirmed by Stefan Vorkoetter\'s review."}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "R↓", "shift1": "VIEW", "shift2": "R↑", "alpha": "E", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "x≷y", "shift1": "INPUT", "shift2": "PSE", "alpha": "F", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "top", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "i", "shift1": "ARG", "shift2": "θ", "alpha": "G", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Alpha letter from Finseth\'s alpha block only (single source)."}, {"section": "cursor", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "FLAGS", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "DISPLAY", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "CONST", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "cursor", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "MEM", "shift2": "", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Cursor key; only a yellow (left-shift) function. Cursor pad sits to the right of the \'top\' section, centred vertically on its two rows."}, {"section": "upper", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "SIN", "shift1": "HYP", "shift2": "ASIN", "alpha": "H", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "COS", "shift1": "π", "shift2": "ACOS", "alpha": "I", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "TAN", "shift1": "INTG", "shift2": "ATAN", "alpha": "J", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "√x", "shift1": "ˣ√y", "shift2": "x²", "alpha": "K", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "yˣ", "shift1": "LOG", "shift2": "LN", "alpha": "L", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "1/x", "shift1": "10ˣ", "shift2": "eˣ", "alpha": "M", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 1, "col": 0, "rowspan": 1, "colspan": 2, "main": "ENTER", "shift1": "SHOW", "shift2": "LASTx", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Double-width ENTER spanning columns 0-1."}, {"section": "upper", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "+/−", "shift1": "=", "shift2": "ABS", "alpha": "N", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "E", "shift1": "←ENG", "shift2": "RND", "alpha": "O", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "E = EEX (enter exponent)."}, {"section": "upper", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "( )", "shift1": "ENG→", "shift2": "[ ]", "alpha": "P", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Parentheses key; blue shift gives vector brackets [ ] (Dieter, hpmuseum post-7043; Vorkoetter review)."}, {"section": "upper", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "←", "shift1": "UNDO", "shift2": "CLEAR", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Backspace. CLEAR is a menu."}, {"section": "lower", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "EQN", "shift1": "∫FN", "shift2": "SOLVE", "alpha": "Q", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Finseth encodes the yellow legend as \'\\\\GI\'; read as the integrate function ∫FN (pairs with FN= on R/S). Verify against a photo."}, {"section": "lower", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "→°F", "shift2": "→°C", "alpha": "R", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "HMS→", "shift2": "→HMS", "alpha": "S", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "→RAD", "shift2": "→DEG", "alpha": "T", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "%CHG", "shift2": "%", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀ (left shift)", "shift1": "", "shift2": "", "alpha": "", "key_color": "#F2B33D", "text_color": "#1E1E1E", "uncertain": false, "note": "Yellow/gold left-shift key (key cap printed with a bent arrow)."}, {"section": "lower", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "→lb", "shift2": "→kg", "alpha": "U", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "→MILE", "shift2": "→KM", "alpha": "V", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "→in", "shift2": "→cm", "alpha": "W", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "nCr", "shift2": "nPr", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "▶ (right shift)", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3D8BD4", "text_color": "#1E1E1E", "uncertain": false, "note": "Blue right-shift key (key cap printed with a bent arrow)."}, {"section": "lower", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "LOGIC", "shift2": "BASE", "alpha": "X", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "→gal", "shift2": "→l", "alpha": "Y", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "SEED", "shift2": "RAND", "alpha": "Z", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "L.R.", "shift2": "SUMS", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "lower", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "C", "shift1": "OFF", "shift2": "ON", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Clear/cancel key; turns calculator on. \'ON\' is printed in blue on the bottom (front) edge; OFF in yellow above."}, {"section": "lower", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": ",", "shift2": "SPACE", "alpha": "(I)", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Yellow \',\' confirmed (hpmuseum post-7043: comma on yellow-shifted zero). Red \'(I)\' read from Finseth alpha line \'(i)\'; manual says indirect registers (I)/(J) are on specific keys but key glyphs are lost in text."}, {"section": "lower", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "/c", "shift2": "FDISP", "alpha": "(J)", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": true, "note": "Yellow \'/c\' (max fraction denominator) confirmed by hpmuseum post-39775 (\'left shift /c\'). \'a b/c\' reminder printed in red below the key. Red \'(J)\' read from Finseth alpha line; exact red legend arrangement uncertain."}, {"section": "lower", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "Σ+", "shift1": "Σ−", "shift2": "!", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue \'!\' is factorial (x!)."}, {"section": "lower", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "x̄,ȳ", "shift2": "s,σ", "alpha": "", "key_color": "#232323", "text_color": "#FFFFFF", "uncertain": false, "note": "Both shifted legends are menus (x̄,ȳ means; s,σ standard deviations)."}], "notes": "43 keys counted here (8 top + 4 cursor + 11 upper + 20 lower). shift1 = yellow left-shift, shift2 = blue right-shift, alpha = red letter. The yellow shift key is in the first column of the 2nd row of the lower block (left of 4) and the blue shift key below it (left of 1); the C/ON key is bottom-left. R/S is top-left (not bottom-right as on classic HPs). ENTER is double width. 16 keys open menus. Finseth\'s text map is the primary positional source; HP\'s own QSG/manual confirm the colour scheme (yellow above, blue on key bottom, letters bottom-right, cursor keys yellow-only) but their key glyphs do not survive text extraction. Colours are approximations."}, "TI-30XS MultiView": {"sources": ["https://education.ti.com/html/eguides/scientifics/PDFs/EN/TI30XS-MultiView-Guide_EN.pdf", "https://education.ti.com/html/eguides/scientifics/TI-30XS-MultiView/EN/Content/EG_SmartView/M_KeyBoard/M_KeyBoard.HTML", "https://cdn.geogebra.org/resource/dckja6sx/Hxe3aChHaf42lER1/material-dckja6sx.pdf", "https://tech.aph.org/Orion%20TI-30XS%20Documents/xs_doc.htm", "https://www.rskey.org/~mwsebastian/reviews/ti30xsmv.htm", "https://education.ti.com/en/customer-support/knowledge-base/scientific-elem-calculators/product-usage/20309"], "shift_keys": {"2nd": {"color": "#3BAA4B", "label_position": "above the key (green text)"}}, "sections": [{"name": "top", "rows": 2, "cols": 3, "placement": "upper-left, under the display"}, {"name": "arrows", "rows": 3, "cols": 3, "placement": "4-way rocker in the top-right corner beside \'top\' (occupies the space of columns 3-4 of a 5-column grid)"}, {"name": "main", "rows": 7, "cols": 5, "placement": "full width below"}], "grid": {"rows": 9, "cols": 5, "note": "overall; \'top\' + \'arrows\' form the first two rows"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "2nd", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3BAA4B", "text_color": "#FFFFFF", "uncertain": false, "note": "Green 2nd key; 2nd legends are printed in green above keys."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "mode", "shift1": "quit", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "delete", "shift1": "insert", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "log", "shift1": "10ˣ", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "prb", "shift1": "angle", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Key position derived (see notes). 2nd legend (angle) from memory; guidebook confirms 2nd[angle] and 2nd[stat] exist but text extraction loses which key."}, {"section": "top", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "data", "shift1": "stat", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Key position derived (see notes). 2nd legend (stat) from memory; guidebook confirms 2nd[angle] and 2nd[stat] exist but text extraction loses which key."}, {"section": "arrows", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker."}, {"section": "arrows", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker. 2nd ◀ jumps to start of entry (no printed legend)"}, {"section": "arrows", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker. 2nd ▶ jumps to end of entry (no printed legend)"}, {"section": "arrows", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#9B9B9B", "text_color": "#1E1E1E", "uncertain": false, "note": "Part of 4-way rocker."}, {"section": "main", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "eˣ", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "n/d", "shift1": "U n/d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "×10ⁿ", "shift1": "n/d◀▶U n/d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": "2nd n/d◀▶U n/d confirmed by GED reference sheet (2nd + this key toggles fraction/mixed)."}, {"section": "main", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "table", "shift1": "f◀▶d", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": "2nd f◀▶d confirmed by GED reference sheet."}, {"section": "main", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "clear", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "π", "shift1": "hyp", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "K", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend K (constant) placed from memory; guidebook confirms 2nd[K] exists."}, {"section": "main", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "^", "shift1": "ˣ√", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Position inferred from TI-SmartView key-letter sequence (only remaining key for slot). 2nd legend unknown."}, {"section": "main", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "%", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend (%) placed from memory; TI KB 20309 confirms 2nd[%] and 2nd[▶%] exist but not which key."}, {"section": "main", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": "▶%", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend (▶%) placed from memory; TI KB 20309 confirms 2nd[%] and 2nd[▶%] exist but not which key."}, {"section": "main", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "√", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 0, "rowspan": 1, "colspan": 1, "main": "x y z t a b c", "shift1": "clear var", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 4, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 0, "rowspan": 1, "colspan": 1, "main": "sto→", "shift1": "recall", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 5, "col": 4, "rowspan": 1, "colspan": 1, "main": "◀▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "Answer-toggle key (switches fraction/decimal/π display). Position directly above enter derived from SmartView key-letter sequence; 2nd legend unknown."}, {"section": "main", "row": 6, "col": 0, "rowspan": 1, "colspan": 1, "main": "on", "shift1": "off", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "reset", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": false, "note": "2nd reset confirmed by GED sheet (\'2nd 0 2\' = reset all)."}, {"section": "main", "row": 6, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}, {"section": "main", "row": 6, "col": 3, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "Ans", "shift2": "", "alpha": "", "key_color": "#ECECEC", "text_color": "#1E1E1E", "uncertain": true, "note": "2nd Ans from memory; Ans is a confirmed 2nd function."}, {"section": "main", "row": 6, "col": 4, "rowspan": 1, "colspan": 1, "main": "enter", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2E2E2E", "text_color": "#FFFFFF", "uncertain": true, "note": "2nd legend unknown/unverified (left blank rather than guessed)."}], "notes": "45 keys including the 4 rocker positions (rskey.org review; APH guide: 41 keys in a 9x5 arrangement plus a 4-key directional pad top-right; ON bottom-left, 2nd top-left). No ALPHA key; variables come from the \'x y z t a b c\' key. POSITION METHOD: TI\'s own guidebooks render keys in a symbol font that does not survive text extraction, and the keypad figures are images. Positions were reconstructed from the TI-SmartView computer-key shortcuts (mode=[W] stated by TI; a GED reference sheet using the same shortcut letters gives q=2nd, w=mode, e=delete, r/u=log/ln, t=prb, y=data, i=n/d, p=×10ⁿ, a=table, s=clear, d=π, f/g/h=sin/cos/tan, j=^, l=x², c=toggle, n=enter). These letters run in keyboard order (q w e r t y u i [o skipped] p a s d f g h j k l z x c v b n) across the non-digit/non-operator keys row by row, which pins every key into a consistent 9x5 layout. Treat as strong inference, not a direct reading of a keypad image. Colours approximate (dark function keys, light number keys, green 2nd key; varies by case colour)."}, "TI-84 Plus CE": {"sources": ["https://education.ti.com/html/eguides/graphing/84CE-Online-Calculator/EN/Content/EG_GSGuide/M_Map/KB_Key_Map.HTML", "https://learn.cemetech.net/index.php/TI-BASIC:Key_Codes", "https://learn.cemetech.net/index.php/TI-BASIC:Getkey", "https://education.ti.com/en/product-resources/guidebooks/ti-84-plus-ce-python"], "shift_keys": {"2nd": {"color": "#3FA4DC", "label_position": "above the key, left side (blue)"}, "alpha": {"color": "#62B446", "label_position": "above the key, right side (green)"}}, "sections": [{"name": "top_row", "rows": 1, "cols": 5, "placement": "directly under the screen"}, {"name": "upper_left", "rows": 2, "cols": 3, "placement": "left 3 columns under top_row"}, {"name": "arrows", "rows": 3, "cols": 3, "placement": "right of upper_left (diamond; only (0,1),(1,0),(1,2),(2,1) used)"}, {"name": "main", "rows": 7, "cols": 5, "placement": "full width below"}], "grid": {"rows": 10, "cols": 5, "note": "getKey rows 1-10; use \'sections\' for exact positions"}, "keys": [{"section": "top_row", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "y=", "shift1": "stat plot", "shift2": "", "alpha": "f1", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": "Graph keys; 2nd legends blue, alpha (f1-f5 menu softkeys) green."}, {"section": "top_row", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "window", "shift1": "tblset", "shift2": "", "alpha": "f2", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "zoom", "shift1": "format", "shift2": "", "alpha": "f3", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "trace", "shift1": "calc", "shift2": "", "alpha": "f4", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "top_row", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "graph", "shift1": "table", "shift2": "", "alpha": "f5", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "2nd", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3FA4DC", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue 2nd key (getKey 21)."}, {"section": "upper_left", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "mode", "shift1": "quit", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "del", "shift1": "ins", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "alpha", "shift1": "A-lock", "shift2": "", "alpha": "", "key_color": "#62B446", "text_color": "#FFFFFF", "uncertain": false, "note": "Green alpha key (getKey 31)."}, {"section": "upper_left", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "X,T,θ,n", "shift1": "link", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "upper_left", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "stat", "shift1": "list", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "arrows", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 25). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 24). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 26). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "arrows", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#A6A6A6", "text_color": "#1E1E1E", "uncertain": false, "note": "Arrow pad to the right of \'upper_left\' (getKey 34). 2nd ▲/▼ adjust screen brightness on the CE (no printed legend)."}, {"section": "main", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "math", "shift1": "test", "shift2": "", "alpha": "A", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "apps", "shift1": "angle", "shift2": "", "alpha": "B", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "prgm", "shift1": "draw", "shift2": "", "alpha": "C", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "vars", "shift1": "distr", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "clear", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "matrix", "shift2": "", "alpha": "D", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "E", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "F", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "G", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "^", "shift1": "π", "shift2": "", "alpha": "H", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "√", "shift2": "", "alpha": "I", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": ",", "shift1": "EE", "shift2": "", "alpha": "J", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "{", "shift2": "", "alpha": "K", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": "}", "shift2": "", "alpha": "L", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "e", "shift2": "", "alpha": "M", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "log", "shift1": "10ˣ", "shift2": "", "alpha": "N", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "u", "shift2": "", "alpha": "O", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "v", "shift2": "", "alpha": "P", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "w", "shift2": "", "alpha": "Q", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "[", "shift2": "", "alpha": "R", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 0, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "eˣ", "shift2": "", "alpha": "S", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 1, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "L4", "shift2": "", "alpha": "T", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 2, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "L5", "shift2": "", "alpha": "U", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 3, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "L6", "shift2": "", "alpha": "V", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 4, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "]", "shift2": "", "alpha": "W", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 0, "rowspan": 1, "colspan": 1, "main": "sto→", "shift1": "rcl", "shift2": "", "alpha": "X", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 1, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "L1", "shift2": "", "alpha": "Y", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 2, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "L2", "shift2": "", "alpha": "Z", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 3, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "L3", "shift2": "", "alpha": "θ", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 5, "col": 4, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "mem", "shift2": "", "alpha": "\\"", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 0, "rowspan": 1, "colspan": 1, "main": "on", "shift1": "off", "shift2": "", "alpha": "", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": "ON has no getKey code."}, {"section": "main", "row": 6, "col": 1, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "catalog", "shift2": "", "alpha": "␣", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": "Alpha legend is a space character (␣)."}, {"section": "main", "row": 6, "col": 2, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "i", "shift2": "", "alpha": ":", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 3, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "ans", "shift2": "", "alpha": "?", "key_color": "#E9E9E9", "text_color": "#1E1E1E", "uncertain": false, "note": ""}, {"section": "main", "row": 6, "col": 4, "rowspan": 1, "colspan": 1, "main": "enter", "shift1": "entry", "shift2": "", "alpha": "solve", "key_color": "#2B2B2B", "text_color": "#FFFFFF", "uncertain": false, "note": ""}], "notes": "50 keys (5 + 6 + 4 arrows + 35). Positions cross-checked two ways: TI\'s online-calculator key map lists keys in row order, and the getKey code grid (row*10+col: 11-15, 21-23 + arrows 24/25/26/34, 31-33, 41-45 ... 101-105) fixes rows/columns. Legend text from TI\'s standard keyboard (identical to TI-84 Plus except CE uses f1-f5 alpha legends on the top row). Key/body colours vary by case colour; hex values are approximate (dark function keys, light number keys, blue 2nd, green alpha, grey arrows)."}, "Casio fx-991EX": {"sources": ["https://www.casio.com/content/dam/casio/global/support/manuals/calculators/pdf/004-en/f/fx-570_991EX_EN.pdf", "https://de.ifixit.com/Document/DynfKy4jK4lucArw/QUICK-START-GUIDE-fx-991EX.pdf", "https://www.manualslib.com/manual/1260605/Casio-Fx-991ex.html", "https://pundit.pratt.duke.edu/wiki/Calculator_Tips/Classwiz", "https://hpmuseum.org/forum/thread-4442-page-3.html", "https://www.casio.com/content/dam/casio/global/support/manuals/calculators/pdf/004-en/f/fx-115ESPLUS_991ESPLUSC_EN.pdf", "https://support.casio.com/global/en/calc/manual/fx-570ESPLUS_991ESPLUS_en/before_using_calculator/getting_started.html"], "shift_keys": {"SHIFT": {"color": "#F2C230", "label_position": "above the key, left (yellow)"}, "ALPHA": {"color": "#E0453A", "label_position": "above the key, right (red)"}, "complex_mode": {"color": "#9B59B6", "label_position": "purple legends/brackets (Complex mode)"}, "base_n": {"color": "#3D7CC9", "label_position": "blue legends/brackets (Base-N: DEC HEX BIN OCT)"}}, "sections": [{"name": "top", "rows": 2, "cols": 6, "placement": "under the display; columns 2-3 are occupied by the cursor pad"}, {"name": "cursor", "rows": 3, "cols": 3, "placement": "round pad centred in \'top\' columns 2-3"}, {"name": "function", "rows": 3, "cols": 6, "placement": "6-column scientific block"}, {"name": "numeric", "rows": 4, "cols": 5, "placement": "5-column number block (larger keys)"}], "grid": {"rows": 9, "cols": 6, "note": "overall; use \'sections\'"}, "keys": [{"section": "top", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "SHIFT", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#F2C230", "uncertain": false, "note": "SHIFT key: dark key with yellow \'SHIFT\' text."}, {"section": "top", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "ALPHA", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#E0453A", "uncertain": false, "note": "ALPHA key: dark key with red \'ALPHA\' text."}, {"section": "top", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "MENU", "shift1": "SETUP", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT MENU = SETUP (confirmed in manual and Duke ClassWiz notes)."}, {"section": "top", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "ON", "shift1": "", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "ON is top-right on ClassWiz."}, {"section": "top", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "OPTN", "shift1": "QR", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT OPTN = QR (QR-code) per quick-start-guide keyboard OCR."}, {"section": "top", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "CALC", "shift1": "SOLVE", "shift2": "", "alpha": "=", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "ALPHA CALC gives the \'=\' used in equations."}, {"section": "top", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "∫▫□", "shift1": "d/dx▫", "shift2": "", "alpha": ":", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "ALPHA \':\' (multi-statement) assumed from ES PLUS, not confirmed for EX."}, {"section": "top", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "x", "shift1": "∑▫□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "Dedicated variable-x key. SHIFT legend Σ placed from memory; manual confirms Σ( exists but not its key."}, {"section": "cursor", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "▲", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "◀", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "▶", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "cursor", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "▼", "shift1": "", "shift2": "", "alpha": "", "key_color": "#3A3E45", "text_color": "#FFFFFF", "uncertain": false, "note": "Round 4-way cursor pad in the centre of the \'top\' block (occupies top-block columns 2-3, both rows)."}, {"section": "function", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "■/□", "shift1": "■■/□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT = mixed-fraction template."}, {"section": "function", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "√■", "shift1": "∛■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "No ALPHA legend (hpmuseum forum)."}, {"section": "function", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "x²", "shift1": "x³", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'DEC\' printed beside x³ (OCR of QSG keyboard)."}, {"section": "function", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "x^■", "shift1": "■√□", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'HEX\'."}, {"section": "function", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "log▫□", "shift1": "10^■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'BIN\'."}, {"section": "function", "row": 0, "col": 5, "rowspan": 1, "colspan": 1, "main": "ln", "shift1": "e^■", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "Blue Base-N legend \'OCT\'."}, {"section": "function", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "(−)", "shift1": "log", "shift2": "", "alpha": "A", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT (−) = log( (log with base argument), from QSG keyboard OCR (\'log A\') and manual \'(log)\'. Differs from ES PLUS, where SHIFT (−) = ∠."}, {"section": "function", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "°\'\\"", "shift1": "FACT", "shift2": "", "alpha": "B", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "x⁻¹", "shift1": "x!", "shift2": "", "alpha": "C", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "sin", "shift1": "sin⁻¹", "shift2": "", "alpha": "D", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "cos", "shift1": "cos⁻¹", "shift2": "", "alpha": "E", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 1, "col": 5, "rowspan": 1, "colspan": 1, "main": "tan", "shift1": "tan⁻¹", "shift2": "", "alpha": "F", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "STO", "shift1": "RECALL", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "function", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "ENG", "shift1": "∠", "shift2": "", "alpha": "i", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ENG = ∠ per Duke ClassWiz notes (\'angle symbol (SHIFT ENG)\'). Imaginary unit i is printed in purple (Complex mode) on/above this key - exact placement (ALPHA vs purple bracket) uncertain."}, {"section": "function", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "(", "shift1": "Abs", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ( = Abs read from QSG keyboard OCR order \'RECALL Abs ...\'; single source."}, {"section": "function", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": ")", "shift1": ",", "shift2": "", "alpha": "x", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT ) = \',\' confirmed (Casio UK QSG \'SHIFT ) (,)\'). ALPHA legend x assumed from ES PLUS."}, {"section": "function", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "S⇔D", "shift1": "a b/c⇔d/c", "shift2": "", "alpha": "y", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT legend assumed from ES PLUS; ALPHA y assumed."}, {"section": "function", "row": 2, "col": 5, "rowspan": 1, "colspan": 1, "main": "M+", "shift1": "M−", "shift2": "", "alpha": "M", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT M+ = M−, ALPHA M (QSG OCR \'M- M\')."}, {"section": "numeric", "row": 0, "col": 0, "rowspan": 1, "colspan": 1, "main": "7", "shift1": "CONST", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 1, "rowspan": 1, "colspan": 1, "main": "8", "shift1": "CONV", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 2, "rowspan": 1, "colspan": 1, "main": "9", "shift1": "RESET", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 0, "col": 3, "rowspan": 1, "colspan": 1, "main": "DEL", "shift1": "INS", "shift2": "", "alpha": "UNDO", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT DEL = INS; ALPHA DEL = UNDO read from OCR \'INS UNDO OFF\'."}, {"section": "numeric", "row": 0, "col": 4, "rowspan": 1, "colspan": 1, "main": "AC", "shift1": "OFF", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 0, "rowspan": 1, "colspan": 1, "main": "4", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 1, "rowspan": 1, "colspan": 1, "main": "5", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 2, "rowspan": 1, "colspan": 1, "main": "6", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 1, "col": 3, "rowspan": 1, "colspan": 1, "main": "×", "shift1": "nPr", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "nPr/nCr exist (manual) and are SHIFT ×/÷ on ES PLUS, but the EX quick-start OCR shows no legend line above this row."}, {"section": "numeric", "row": 1, "col": 4, "rowspan": 1, "colspan": 1, "main": "÷", "shift1": "nCr", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "See ×."}, {"section": "numeric", "row": 2, "col": 0, "rowspan": 1, "colspan": 1, "main": "1", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 1, "rowspan": 1, "colspan": 1, "main": "2", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 2, "rowspan": 1, "colspan": 1, "main": "3", "shift1": "", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 3, "rowspan": 1, "colspan": 1, "main": "+", "shift1": "Pol", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 2, "col": 4, "rowspan": 1, "colspan": 1, "main": "−", "shift1": "Rec", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 0, "rowspan": 1, "colspan": 1, "main": "0", "shift1": "Rnd", "shift2": "", "alpha": "", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 1, "rowspan": 1, "colspan": 1, "main": ".", "shift1": "Ran#", "shift2": "", "alpha": "RanInt", "key_color": "#5B6068", "text_color": "#FFFFFF", "uncertain": false, "note": ""}, {"section": "numeric", "row": 3, "col": 2, "rowspan": 1, "colspan": 1, "main": "×10ˣ", "shift1": "π", "shift2": "", "alpha": "e", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT ×10ˣ = π confirmed (Casio UK QSG). ALPHA = e."}, {"section": "numeric", "row": 3, "col": 3, "rowspan": 1, "colspan": 1, "main": "Ans", "shift1": "%", "shift2": "", "alpha": "PreAns", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": true, "note": "SHIFT Ans = % read from QSG OCR (π % ≈ sequence). ALPHA legend PreAns from memory; not found in manual."}, {"section": "numeric", "row": 3, "col": 4, "rowspan": 1, "colspan": 1, "main": "=", "shift1": "≈", "shift2": "", "alpha": "", "key_color": "#2A2D33", "text_color": "#FFFFFF", "uncertain": false, "note": "SHIFT = gives decimal result (≈)."}], "notes": "50 keys counted (8 top + 4 cursor directions + 18 function + 20 numeric). Casio manuals and quick-start guides draw keys as font glyphs/images, so legends were assembled from: the EN user\'s guide list of alternate-function names, an OCR of the quick-start-guide keyboard figure (iFixit copy), Casio UK QSG text, and Duke\'s ClassWiz notes; remaining items carried over from the ES PLUS layout are flagged uncertain. Digit keys 1-6 have no shifted legends on the EX (Matrix/Vector/Stat etc. moved to MENU apps and OPTN). Colours approximate.", "fx-115ES_PLUS_differences": {"uncertain": true, "summary": "The US fx-115ES PLUS is the previous (ES PLUS) generation and its keyboard differs materially: MODE (SHIFT=SETUP) instead of MENU; no OPTN key and no dedicated x key; row under SHIFT/ALPHA is CALC (SHIFT SOLVE, ALPHA =), ∫ (SHIFT d/dx, ALPHA :) on the left and x⁻¹ (SHIFT x!), log▫□ (SHIFT Σ) on the right of the cursor pad; next row ■/□, √■, x², x^■, log, ln; then (−) (SHIFT ∠, ALPHA A), °\'\\" (SHIFT ←, ALPHA B), hyp (SHIFT Abs, ALPHA C), sin/cos/tan (ALPHA D/E/F); then RCL (SHIFT STO), ENG (SHIFT ←, purple i), ( (SHIFT %), ) (SHIFT \',\', ALPHA X), S⇔D (ALPHA Y), M+ (SHIFT M−, ALPHA M). Number block: SHIFT 1 STAT, 2 CMPLX, 3 DISTR, 4 MATRIX, 5 VECTOR, 7 CONST, 8 CONV, 9 CLR, × nPr, ÷ nCr, + Pol, − Rec, 0 Rnd, . Ran# (ALPHA RanInt#), ×10ˣ π (ALPHA e), Ans DRG▸, DEL INS, AC OFF. Manual confirms (STO), (CLR), (DRG▸), (←), (Abs), (FACT), (∠) as shifted functions; exact key placement is from memory of the ES PLUS keyboard and should be checked against a photo before use."}}}')

# ---------------------------------------------------------------------------
# Calculator faceplates. Each model's keys sit at the same row/column as on
# the physical device, with the shifted legends printed where the device
# prints them. Pressing a key sends the legend that is active (main, first
# shift, second shift or alpha) to the simulated engine; legends that the
# engine does not cover (programming, graphing menus, cursor editing) show
# "not simulated" instead of a wrong result.
# ---------------------------------------------------------------------------
import re as _re_ui

# legend (normalised) -> engine key.  Normalisation: strip spaces, unify minus
# signs and superscripts.  RPN and algebraic legends share most names.
_LEG = {
    # digits / entry
    **{d: d for d in "0123456789"}, ".": ".", "·": ".",
    "enter": "ENTER", "=": "EQ", "exe": "EXE", "enter↑": "ENTER", "e": "E",
    "chs": "CHS", "+/-": "CHS", "(-)": "NEG", "eex": "EEX", "×10ˣ": "EEX", "x10ˣ": "EEX", "ee": "EEX",
    "×10^x": "EEX", "x10^x": "EEX", "×10x": "EEX",
    "←": "BSP", "bsp": "BSP", "del": "DEL", "clx": "CLX", "clear": "CLEAR", "ac": "AC", "c": "CLEAR",
    "clstk": "CLSTK", "clrstk": "CLSTK", "clear stk": "CLSTK",
    # arithmetic
    "+": "ADD", "-": "SUB", "×": "MUL", "*": "MUL", "÷": "DIV", "/": "DIV",
    "(": "LP", ")": "RP", ",": "COMMA",
    # powers and roots
    "yˣ": "YX", "y^x": "YX", "^": "YX", "xʸ": "YX", "x^y": "YX", "x■": "YX", "x^□": "YX", "□^□": "YX",
    "√x": "SQRT", "√": "SQRT", "√(": "SQRT", "√■": "SQRT", "√□": "SQRT",
    "x²": "SQ", "x^2": "SQ", "x■²": "SQ", "□²": "SQ",
    "1/x": "INV", "x⁻¹": "INV", "x^-1": "INV", "□⁻¹": "INV",
    "ˣ√y": "XROOTY", "x√y": "XROOTY", "ˣ√": "XROOTY", "x√": "XROOTY", "ˣ√■": "XROOTY", "■√□": "XROOTY",
    "□√□": "XROOTY", "ˣ√(": "XROOTY", "∛": "CBRT", "∛■": "CBRT", "∛□": "CBRT", "3√": "CBRT",
    # exp / log
    "eˣ": "EXP", "e^x": "EXP", "e■": "EXP", "e^□": "EXP", "ln": "LN", "10ˣ": "10X", "10^x": "10X",
    "10■": "10X", "10^□": "10X", "log": "LOG",
    # trig
    "sin": "SIN", "cos": "COS", "tan": "TAN", "sin⁻¹": "ASIN", "cos⁻¹": "ACOS", "tan⁻¹": "ATAN",
    "sin^-1": "ASIN", "cos^-1": "ACOS", "tan^-1": "ATAN", "asin": "ASIN", "acos": "ACOS", "atan": "ATAN",
    "hyp": "HYP", "hyp⁻¹": "AHYP", "hyp^-1": "AHYP",
    "deg": "DEG", "rad": "RAD", "grd": "GRD", "→rad": "TORAD", "→deg": "TODEG", "→r": "TOR", "→p": "TOP",
    "→h.ms": "HMS", "→h": "HR", "→hms": "HMS", "→hr": "HR",
    # misc functions
    "%": "PCT", "δ%": "DPCT", "∆%": "DPCT", "abs": "ABS", "|x|": "ABS", "abs(": "ABS", "int": "INT", "ip": "INT",
    "frac": "FRAC", "fp": "FRAC", "x!": "FACT", "!": "FACT", "rnd": "RND",
    "py,x": "PERM", "cy,x": "COMB", "npr": "PERM", "ncr": "COMB",
    "π": "PI", "pi": "PI", "ans": "ANS",
    # stack
    "x≷y": "SWAP", "x<>y": "SWAP", "x⇄y": "SWAP", "r↓": "RDN", "r↑": "RUP", "lstx": "LASTX", "lastx": "LASTX",
    # statistics
    "σ+": "SIGMAP", "σ-": "SIGMAM", "x̄": "MEAN", "s": "SDEV", "l.r.": "LR", "ŷ,r": "YHATR", "clσ": "CLSIGMA",
    "clear σ": "CLSIGMA",
    # display / memory
    "fix": "FIX", "sci": "SCI", "eng": "ENG", "all": "ALL", "sto": "STO", "rcl": "RCL", "sto→": "STO",
    "sto▸": "STO", "sto►": "STO",
    # model-specific spellings (from calc_layouts.json)
    "E": "EEX", "×10ⁿ": "EEX", "x10ⁿ": "EEX", "%chg": "DPCT", "x̄,ȳ": "MEAN", "s,σ": "SDEV", "hms→": "HR", "→hms": "HMS",
    "→°f": "TOF", "→°c": "TOC", "→lb": "TOLB", "→kg": "TOKG", "→mile": "TOMILE", "→km": "TOKM", "→in": "TOIN",
    "→cm": "TOCM", "→gal": "TOGAL", "→l": "TOL", "x^■": "YX", "10^■": "10X", "e^■": "EXP", "x³": "CUBE",
    "■/□": "DIV", "entry": "ENTRY", "delete": "DEL", "ans": "ANS", "√": "SQRT", "ip": "INT", "fp": "FRAC",
}
_SHIFT_NAMES = {"◀(leftshift)": "shift1", "▶(rightshift)": "shift2","f": "shift1", "g": "shift2", "2nd": "shift1", "shift": "shift1", "alpha": "alpha",
                "◢": "shift1", "◣": "shift2", "left-shift": "shift1", "right-shift": "shift2",
                "leftshift": "shift1", "rightshift": "shift2", "⬏": "shift1", "⬎": "shift2"}
_ALG_VARS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZθxyzt")


def _norm(s):
    s = (s or "").strip()
    s = s.replace("−", "-").replace("–", "-").replace("Σ", "σ").replace("x̅", "x̄")
    s = _re_ui.sub(r"\s+", "", s)
    return s.lower() if len(s) > 1 else s


def legend_to_key(legend, rpn):
    if not legend:
        return None
    n = _norm(legend)
    if n in _LEG:
        k = _LEG[n]
        if rpn and k == "NEG":
            return "CHS"
        if not rpn and k == "CHS":
            return "NEG"
        if not rpn and k == "CLX":
            return "CLEAR"
        if rpn and k in ("CLEAR", "AC"):
            return "CLX"
        if rpn and k in ("CUBE", "ENTRY", "DEL", "ANS"):
            return {"DEL": "BSP"}.get(k)
        if not rpn and k.startswith("TO") and k not in ("TOP", "TOR"):
            return None
        if rpn and k == "EQ":
            return None
        if rpn and k == "LP":
            return None
        return k
    if not rpn and len(legend.strip()) == 1 and legend.strip() in _ALG_VARS:
        return "VAR_" + legend.strip()
    return None


class Faceplate:
    """UI-independent model of one calculator: key grid + shift state + engine."""

    RPN_MODELS = ("HP-15C", "HP 35s")

    def __init__(self, model, layout):
        self.model, self.layout = model, layout
        self.rpn = model in self.RPN_MODELS
        self.engine = RPNEngine(model) if self.rpn else AlgEngine(model, implicit_mult=True)
        if model.startswith("Casio"):
            self.engine.disp_mode = "ALL"
        self.state = None            # None | 'shift1' | 'shift2' | 'alpha'
        self.state_used = None
        self.log = []
        self.keys = layout["keys"]

    def shift_of(self, k):
        for lab in (k.get("main"), k.get("note")):
            n = _norm(lab)
            if n in _SHIFT_NAMES:
                return _SHIFT_NAMES[n]
        return None

    def active_legend(self, k):
        if self.state and k.get(self.state):
            return k[self.state]
        return k.get("main", "")

    def press(self, k):
        """Press key dict k. Returns the legend that acted."""
        sh = self.shift_of(k)
        if sh:
            self.state = None if self.state == sh else sh
            self.log.append(k.get("main"))
            return k.get("main")
        legend = self.active_legend(k)
        self.state_used = self.state if (self.state and k.get(self.state)) else None
        self.log.append(("%s " % {"shift1": self.layout.get("_s1", "2nd"), "shift2": self.layout.get("_s2", "g"),
                                  "alpha": "ALPHA"}[self.state] if self.state else "") + (k.get("main") or ""))
        self.state = None
        n = _norm(legend)
        if self.state_used == "alpha" and not self.rpn and len(legend.strip()) == 1 and legend.strip().isalpha():
            self.engine.press("VAR_" + legend.strip()); return legend
        if n in ("on", "on/c", "on/off", "c/on"):
            self.engine.__init__(self.model) if self.rpn else self.engine.__init__(self.model, True)
            return legend
        key = legend_to_key(legend, self.rpn)
        if key is None:
            self.engine.message = "%s: not simulated" % legend
            return legend
        self.engine.press(key)
        return legend

    def press_legend(self, legend, state=None):
        """Find the key whose legend (in the given state) matches and press it (used by tests)."""
        for k in self.keys:
            if (k.get(state) if state else k.get("main")) == legend:
                if state:
                    sk = next(x for x in self.keys if self.shift_of(x) == state)
                    self.press(sk)
                return self.press(k)
        raise KeyError("%s has no key %r (%s)" % (self.model, legend, state or "main"))

    def screen(self):
        e = self.engine
        if self.rpn:
            st = e.stack()
            return dict(main=e.display(), stack=[("T", st["T"]), ("Z", st["Z"]), ("Y", st["Y"])],
                        ann=" ".join(x for x in (self.state and {"shift1": "f", "shift2": "g"}[self.state],
                                                 e.angle if e.angle != "DEG" else "") if x))
        return dict(main=e.display(), entry=e.entry_text(), ann=" ".join(
            x for x in ({"shift1": "2nd", "shift2": "", "alpha": "ALPHA"}.get(self.state or "", ""), e.angle) if x))

    def coverage(self):
        """(mapped, total, unmapped legends) over every legend on the faceplate."""
        tot, ok, miss = 0, 0, []
        for k in self.keys:
            if self.shift_of(k):
                continue
            for st in ("main", "shift1", "shift2", "alpha"):
                lab = k.get(st)
                if not lab or (st == "alpha" and not self.rpn and False):
                    continue
                tot += 1
                if legend_to_key(lab, self.rpn) or _norm(lab) in ("on", "on/c", "on/off", "c/on"):
                    ok += 1
                else:
                    miss.append(lab)
        return ok, tot, miss


# ---------------------------------------------------------------------------
# How the JSON sections sit relative to each other on the physical device:
# each block is one grid; a section is placed at (row offset, col offset, col remap).
ARRANGE = {
    "HP 35s": [[("top", 0, 0), ("cursor", 0, 4)], [("upper", 0, 0)], [("lower", 0, 0)]],
    "TI-30XS MultiView": [[("top", 0, 0), ("arrows", 0, 3)], [("main", 0, 0)]],
    "TI-84 Plus CE": [[("top_row", 0, 0)], [("upper_left", 0, 0), ("arrows", 0, 3)], [("main", 0, 0)]],
    "Casio fx-991EX": [[("top", 0, 0, {4: 5, 5: 6}), ("cursor", 0, 2)], [("function", 0, 0)], [("numeric", 0, 0)]],
}


def blocks(model, layout):
    """[(rows, cols, [(key, row, col)])] — one grid per block, keys at device positions."""
    secs = {s["name"]: s for s in layout.get("sections", [])}
    if not secs:
        g = layout["grid"]
        return [(g["rows"], g["cols"], [(k, k["row"], k["col"]) for k in layout["keys"]])]
    arr = ARRANGE.get(model) or [[(n, 0, 0)] for n in secs]
    out = []
    for blk in arr:
        items, R, Cn = [], 0, 0
        for ent in blk:
            name, ro, co = ent[:3]
            remap = ent[3] if len(ent) > 3 else {}
            for k in layout["keys"]:
                if k.get("section") == name:
                    c = remap.get(k["col"], k["col"]) + co
                    r = k["row"] + ro
                    items.append((k, r, c))
                    R = max(R, r + k.get("rowspan", 1)); Cn = max(Cn, c + k.get("colspan", 1))
        out.append((R, Cn, items))
    return out


CALC_LAYOUTS = globals().get("CALC_LAYOUTS", {})
_PLATES = {}


def _key_html(k, model, layout):
    sk = layout.get("shift_keys", {})
    pick = lambda names, d: next((sk[n]["color"] for n in names if n in sk), d)
    s1c = pick(("f", "left_shift", "2nd", "SHIFT"), "#d4a017")
    s2c = pick(("g", "right_shift"), "#3b82c4")
    ac = pick(("alpha", "ALPHA"), "#16a34a")
    top = []
    if k.get("shift1"):
        top.append('<span style="color:%s">%s</span>' % (s1c, k["shift1"]))
    if k.get("alpha"):
        top.append('<span style="color:%s;float:right">%s</span>' % (ac, k["alpha"]))
    bottom = ('<span style="color:%s">%s</span>' % (s2c, k["shift2"])) if k.get("shift2") else ""
    return ("<div style='font:9px/11px sans-serif;height:11px;white-space:nowrap;overflow:hidden'>%s</div>" % " ".join(top),
            "<div style='font:9px/11px sans-serif;height:11px;white-space:nowrap;overflow:hidden;text-align:center'>%s</div>" % bottom)


def calculator_widget(model):
    import ipywidgets as W
    layout = CALC_LAYOUTS[model]
    fp = _PLATES.get(model) or Faceplate(model, layout)
    _PLATES[model] = fp
    lcd = W.HTML()
    keylog = W.HTML()

    def refresh():
        s = fp.screen()
        if fp.rpn:
            stack = "".join("<div style='color:#6b7280;font-size:11px'>%s: %s</div>" % (n, fmt_number(v, "ALL", 10))
                            for n, v in s["stack"])
            body = "%s<div style='font-size:22px;text-align:right'>%s</div>" % (stack, s["main"])
        else:
            body = ("<div style='font-size:15px;min-height:20px;word-break:break-all'>%s</div>"
                    "<div style='font-size:22px;text-align:right'>%s</div>" % (s["entry"] or "&nbsp;", s["main"] or "&nbsp;"))
        lcd.value = ("<div style='background:#c9d6b8;color:#111;font-family:monospace;padding:6px 10px;border-radius:6px;"
                     "width:300px;border:2px solid #555'><div style='font-size:10px;height:12px'>%s</div>%s</div>"
                     % (s["ann"], body))
        keylog.value = "<span style='font:11px monospace;color:#555'>Keys: %s</span>" % " ".join(
            str(x) for x in fp.log[-24:])

    boxes = []
    for R, Cn, items in blocks(model, layout):
        children = []
        for k, r, c in items:
            top, bot = _key_html(k, model, layout)
            b = W.Button(description=k.get("main", ""), layout=W.Layout(width="auto", height="30px", padding="0"),
                         tooltip="; ".join("%s: %s" % (a, k[a]) for a in ("shift1", "shift2", "alpha") if k.get(a)))
            b.style.button_color = k.get("key_color", "#2b2b2b")
            b.style.text_color = k.get("text_color", "#ffffff")
            b.style.font_weight = "bold"
            if k.get("uncertain"):
                b.layout.border = "2px dashed #f59e0b"
                b.tooltip = (b.tooltip + "; " if b.tooltip else "") + "position/legend not yet confirmed: " + (k.get("note") or "")

            def on(_b, k=k):
                fp.press(k)
                refresh()
            b.on_click(on)
            cell = W.VBox([W.HTML(top), b, W.HTML(bot)], layout=W.Layout(
                grid_row="%d / span %d" % (r + 1, k.get("rowspan", 1)),
                grid_column="%d / span %d" % (c + 1, k.get("colspan", 1)), margin="0"))
            children.append(cell)
        boxes.append(W.GridBox(children, layout=W.Layout(
            grid_template_columns="repeat(%d, 58px)" % Cn, grid_gap="2px 4px", width="max-content", margin="0 0 8px 0")))
    refresh()
    body = layout.get("body_color", "#3a3a3a")
    return W.VBox([lcd] + boxes + [keylog], layout=W.Layout(
        padding="10px", border="1px solid #999", width="max-content")), fp


def open_calculator(model=None):
    """Model selector + faceplate. Engines keep their state between questions."""
    import ipywidgets as W
    from IPython.display import display
    models = [m for m in ("HP-15C", "HP 35s", "TI-30XS MultiView", "TI-84 Plus CE", "Casio fx-991EX") if m in CALC_LAYOUTS]
    pick = W.Dropdown(options=models, value=model or globals().get("_LAST_CALC", models[0]), description="Calculator")
    holder = W.Output()
    help_ = W.HTML("<small>Shifted functions: press the coloured shift key, then the key under the legend. "
                   "Grey legends that are not simulated (programming, graphing, cursor editing) say so instead of "
                   "giving a wrong answer.</small>")

    def show(_=None):
        globals()["_LAST_CALC"] = pick.value
        holder.clear_output()
        with holder:
            w, _ = calculator_widget(pick.value)
            display(w)
    pick.observe(show, names="value")
    display(W.VBox([pick, help_, holder]))
    show()


# ---------------------------------------------------------------------------
# Notebook glue: lab-report hooks, per-cell section labels, calculation cards
# and the end-of-notebook report panel.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as _plt_glue

REPORT = None


def _start_report(course, session, title):
    global REPORT
    REPORT = LabReport(course, session, title)
    _orig_show = _plt_glue.show

    def _show(*a, **k):                      # every figure shown is copied into the report
        try:
            for num in _plt_glue.get_fignums():
                f = _plt_glue.figure(num)
                if not getattr(f, "_in_report", False):
                    REPORT.add_figure(f)
                    f._in_report = True
        except Exception:
            pass
        return _orig_show(*a, **k)
    _plt_glue.show = _show
    return REPORT


def _cell(label):
    """Called at the top of each hidden code cell: sets the report section."""
    if REPORT is not None:
        REPORT.set_section(label)


def card(card_id, calculator=True):
    """Interactive solver for one calculation card (+ the calculator panel)."""
    c = CARDS[card_id]
    opener = None
    if calculator:
        try:
            opener = lambda: open_calculator()      # defined by calc_ui
        except NameError:
            opener = None
    return solver_widget(c, REPORT, opener)


def report_panel():
    """Student name, reflection notes and download buttons for the lab report."""
    try:
        import ipywidgets as W
        from IPython.display import display, Markdown, clear_output
    except Exception:
        print(REPORT.counts() if REPORT else "no report")
        return
    name = W.Text(description="Your name", placeholder="as in Canvas", layout=W.Layout(width="420px"))
    notes = W.Textarea(description="Discussion", placeholder="What do your calculations show? What would you check next?",
                       layout=W.Layout(width="95%", height="120px"))
    btn = W.Button(description="Build and download report (.docx + .html)", button_style="primary", icon="download",
                   layout=W.Layout(width="360px"))
    out = W.Output()

    def go(_):
        with out:
            clear_output()
            REPORT.student = name.value.strip()
            REPORT.entries = [e for e in REPORT.entries if not (e["kind"] == "note" and e["title"] == "Discussion")]
            if notes.value.strip():
                REPORT.section = "Discussion"
                REPORT.add_note("Discussion", notes.value.strip())
            c = REPORT.counts()
            display(Markdown("Collected **%d** figures, **%d** calculations and **%d** self-check answers." % (
                c.get("figure", 0), c.get("calc", 0), c.get("answer", 0))))
            print(REPORT.download())
    btn.on_click(go)
    display(W.VBox([name, notes, btn, out]))


def _log_answers(gb):
    """Wrap the gradebook's answer() so first answers (not the readiness check) go into the report."""
    orig = gb.answer

    def _resp():
        r = getattr(gb, "resp", None)
        return r if r is not None else getattr(gb, "responses", {})

    def answer(qid, choice, *a, **k):
        first = qid not in _resp()
        r = orig(qid, choice, *a, **k)
        try:
            if first and qid in _resp() and REPORT is not None:
                it = gb.items[qid]
                if getattr(it, "section", "") != "precheck":
                    ch = _resp()[qid]
                    REPORT.add_answer(qid, getattr(it, "stem", qid), ch, ch == it.key)
        except Exception:
            pass
        return r
    gb.answer = answer



# ---------------------------------------------------------------------------
# PHS 312 L32 (Prodrugs) calculation cards: by-products released by a
# tripartite prodrug, log D of an acid vs its ester, the salt/ester mass
# factor of a prodrug, first-order activation in two outlier patients, and
# conversion half-life with minutes <-> hours. No doses or dosing intervals.
# ---------------------------------------------------------------------------


def _c32_pos(d, *keys):
    for k in keys:
        if d[k] <= 0:
            raise ValueError("%s must be > 0" % k)


def _c32_p(x):
    return r"\left(%s\right)" % fmt(x) if x < 0 else fmt(x)


# ---- 1. What a tripartite prodrug releases (new practice, key-ideas part) -----
def _solve_bp(card, d):
    S = Solution(card, d); V = card.V
    _c32_pos(d, "m", "M_pro")
    m_mg = d["m"] * 1e3
    n = m_mg / d["M_pro"]                       # mmol (mg / (g/mol) = mmol)
    narm = d["arms"] * n
    mt, mf, mi, mc, mw = n * d["M_d"], narm * d["M_f"], narm * d["M_i"], narm * d["M_c"], narm * d["M_w"]
    S.general().given().unit_check(
        "Write the sample in milligrams: 1 g = 10³ mg. Then mg ÷ (g/mol) = 10⁻³ g ÷ (g/mol) = 10⁻³ mol = mmol, and "
        "mmol × g/mol = mg. Mol per mol stoichiometry factors carry the milli prefix through unchanged.")
    S.step("Sample in mg", r"%s = %s\,\mathrm{g} \times 10^{3}\,\mathrm{mg/g} = %s\,\mathrm{mg}" % (V["m"].tex(), fmt(d["m"]), fmt(m_mg)))
    S.step("Moles of prodrug", r"n = \frac{%s}{%s} = \frac{%s\,\mathrm{mg}}{%s\,\mathrm{g/mol}} = %s\,\mathrm{mmol}" % (
        V["m"].tex(), V["M_pro"].tex(), fmt(m_mg), fmt(d["M_pro"]), fmt(n)))
    S.step("Moles from the two arms", r"n_{arm} = %s \times n = %s \times %s = %s\,\mathrm{mmol}" % (V["arms"].tex(), fmt(d["arms"]), fmt(n), fmt(narm)))
    S.step("Tenofovir released", r"%s \times %s = %s\,\mathrm{mmol} \times %s\,\mathrm{g/mol} = %s\,\mathrm{mg}" % (
        "n", V["M_d"].tex(), fmt(n), fmt(d["M_d"]), fmt(mt)))
    S.step("Formaldehyde released", r"%s = n_{arm} \times %s = %s \times %s = %s\,\mathrm{mg}" % (
        V["m_f"].tex(), V["M_f"].tex(), fmt(narm), fmt(d["M_f"]), fmt(mf)))
    S.step("Isopropanol released", r"n_{arm} \times %s = %s \times %s = %s\,\mathrm{mg}" % (V["M_i"].tex(), fmt(narm), fmt(d["M_i"]), fmt(mi)))
    S.step("CO₂ released", r"n_{arm} \times %s = %s \times %s = %s\,\mathrm{mg}" % (V["M_c"].tex(), fmt(narm), fmt(d["M_c"]), fmt(mc)))
    S.step("Mass balance check (water is consumed)", r"%s + %s + %s + %s = %s\,\mathrm{mg} = %s + %s\,\mathrm{mg\ H_2O}" % (
        fmt(mt), fmt(mf), fmt(mi), fmt(mc), fmt(mt + mf + mi + mc), fmt(m_mg), fmt(mw)))
    S.answer(mf, "mg", "Complete hydrolysis of %s g of tenofovir disoproxil gives %s mg of tenofovir (%s%% of the mass) plus "
             "%s mg formaldehyde, %s mg isopropanol and %s mg CO₂ from the two promoiety arms. The promoiety must be judged "
             "at the amount released, which is why formaldehyde-releasing linkers are scrutinized." % (
                 fmt(d["m"]), fmt(mt), fmt(100 * mt / m_mg), fmt(mf), fmt(mi), fmt(mc)))
    return S


_CARD32_BP = Card("l32_byproducts", "By-products released by a tripartite prodrug (practice: tenofovir disoproxil)",
    r"n = \frac{{{m}}}{{{M_pro}}};\quad {{m_f}} = {{arms}} \times n \times {{M_f}}",
    [Var("m", r"m_{pro}", "tenofovir disoproxil (free base) hydrolysed", "g", "input", "in-vitro study (case)", 1.000),
     Var("M_pro", r"M_{pro}", "molar mass of tenofovir disoproxil, C₁₉H₃₀N₅O₁₀P", "g/mol", "input", "formula, IUPAC atomic weights", 519.44),
     Var("M_d", r"M_{drug}", "molar mass of tenofovir, C₉H₁₄N₅O₄P", "g/mol", "input", "formula, IUPAC atomic weights", 287.21),
     Var("m_f", r"m_{HCHO}", "formaldehyde released", "mg", "result", "calculated")],
    "A hospital drug-information pharmacist is asked how much formaldehyde the 'disoproxil' promoiety releases. In an in-vitro "
    "esterase study, [[m]] of tenofovir disoproxil (molar mass [[M_pro]]) is hydrolysed completely. Each of its [[arms]] "
    "–O–CH₂–O–C(=O)–O–iPr arms releases one formaldehyde, one CO₂ and one isopropanol, leaving tenofovir ([[M_d]]). "
    "How many mg of each product form?",
    _solve_bp,
    constants=[Var("arms", r"n_{arms}", "promoiety arms per molecule", "", "constant", "structure (two phosphonate esters)", 2, const=True),
               Var("M_f", r"M_{HCHO}", "molar mass of formaldehyde, CH₂O", "g/mol", "constant", "IUPAC atomic weights", 30.026, const=True),
               Var("M_i", r"M_{iPrOH}", "molar mass of isopropanol, C₃H₈O", "g/mol", "constant", "IUPAC atomic weights", 60.096, const=True),
               Var("M_c", r"M_{CO_2}", "molar mass of CO₂", "g/mol", "constant", "IUPAC atomic weights", 44.009, const=True),
               Var("M_w", r"M_{H_2O}", "molar mass of water (consumed, one per arm)", "g/mol", "constant", "IUPAC atomic weights", 18.015, const=True)],
    conversions=["1 g = 1000 mg", "mg ÷ (g/mol) = mmol", "1 mmol = 10⁻³ mol = 1000 µmol"],
    result_name=r"m_{HCHO}", source="activation chemistry of acyloxymethyl/carbonate-acetal prodrugs (L32 notes)")


# ---- 2. log D of an acid and its ethyl ester (worked example) ------------------
def _solve_lde(card, d):
    S = Solution(card, d); V = card.V
    e = d["pH"] - d["pKa"]
    r = 10 ** e
    ion = 100 * r / (1 + r)
    corr = math.log10(1 + r)
    la = d["lPa"] - corr
    le = d["lPe"]
    D = le - la
    S.general().given().unit_check("log P, log D, pH and pKa are logarithms (no units); the fold change is a ratio of two "
                                   "octanol/water distribution coefficients, also unit-free.")
    S.step("Acid: exponent", r"pH - %s = %s - %s = %s" % (V["pKa"].tex(), fmt(d["pH"]), fmt(d["pKa"]), fmt(e)))
    S.step("Acid: ionized/neutral ratio", r"\frac{[A^{-}]}{[HA]} = 10^{%s} = %s" % (fmt(e), fmt(r)))
    S.step("Acid: percent ionized", r"\frac{100 \times %s}{1 + %s} = %s\%%" % (fmt(r), fmt(r), fmt(ion)))
    S.step("Acid: log term", r"\log_{10}(1 + %s) = \log_{10}(%s) = %s" % (fmt(r), fmt(1 + r), fmt(corr)))
    S.step("Acid: log D", r"%s = %s - %s = %s - %s = %s" % (V["lDa"].tex(), V["lPa"].tex(), fmt(corr), fmt(d["lPa"]), fmt(corr), fmt(la)))
    S.step("Ester: no ionizable group", r"%s = %s = %s" % (V["lDe"].tex(), V["lPe"].tex(), fmt(le)))
    S.step("Difference", r"%s - %s = %s - %s = %s" % (V["lDe"].tex(), V["lDa"].tex(), fmt(le), _c32_p(la), fmt(D)))
    S.step("Fold change", r"10^{%s} = %s" % (fmt(D), fmt(10 ** D)))
    S.step("Split of the gain", r"\text{extra carbons: } %s - %s = %s;\ \text{removing the charge: } %s" % (
        fmt(d["lPe"]), fmt(d["lPa"]), fmt(d["lPe"] - d["lPa"]), fmt(corr)))
    S.answer(D, "", "At pH %s the acid is %s%% ionized (log D %s); the ester stays neutral (log D %s). The ester partitions "
             "%s log units, about %s-fold, more into octanol; %s of that comes from removing the charge, only %s from the "
             "extra carbons." % (fmt(d["pH"]), fmt(ion), fmt(la), fmt(le), fmt(D), fmt(10 ** D), fmt(corr), fmt(d["lPe"] - d["lPa"])))
    return S


_CARD32_LDE = Card("l32_logd_ester", "log D of a carboxylic-acid drug and its ethyl ester prodrug (worked example: how much does an ester raise log D?)",
    r"{{lDa}} = {{lPa}} - \log_{10}\left(1 + 10^{{{pH}} - {{pKa}}}\right);\quad {{lDe}} = {{lPe}}",
    [Var("lPa", r"\log P_{acid}", "log P of the neutral acid", "", "input", "case (illustrative)", 1.0),
     Var("pKa", r"pK_a", "pKa of the acid", "", "input", "case (illustrative)", 4.0),
     Var("lPe", r"\log P_{ester}", "log P of the ethyl ester (no ionizable group)", "", "input", "case (illustrative)", 2.3),
     Var("pH", r"pH", "pH of the upper small intestine", "", "input", "physiological range 6–7", 6.5),
     Var("lDa", r"\log D_{acid}", "log D of the acid", "", "intermediate", "calculated"),
     Var("lDe", r"\log D_{ester}", "log D of the ester", "", "intermediate", "calculated")],
    "A carboxylic-acid drug has log P [[lPa]] and pKa [[pKa]]. Its ethyl ester prodrug has log P [[lPe]] and no ionizable "
    "group. Calculate log D for each at pH [[pH]] (upper small intestine) and the fold change in octanol/water distribution.",
    _solve_lde,
    conversions=["D = 10^(log D)", "% neutral = 100 − % ionized"],
    result_name=r"\Delta\log D", source="log D of a monoprotic acid (L32 notes, masking a charge)")


# ---- 3. Salt and ester mass factor (new practice, ester part) -----------------
def _solve_mw(card, d):
    S = Solution(card, d); V = card.V
    _c32_pos(d, "M_s", "M_e", "M_a", "m_eq")
    fe = d["M_e"] / d["M_a"]
    fs = d["M_s"] / d["M_e"]
    ft = d["M_s"] / d["M_a"]
    w = d["m_eq"] * ft
    S.general().given().unit_check("g/mol divided by g/mol cancels, so each factor is unit-free; the mass to weigh keeps the "
                                   "unit of the target (mg). The milli prefix carries through and is never converted.")
    S.step("Ester factor", r"\frac{%s}{%s} = \frac{%s}{%s} = %s" % (V["M_e"].tex(), V["M_a"].tex(), fmt(d["M_e"]), fmt(d["M_a"]), fmt(fe)))
    S.step("Salt factor", r"\frac{%s}{%s} = \frac{%s}{%s} = %s" % (V["M_s"].tex(), V["M_e"].tex(), fmt(d["M_s"]), fmt(d["M_e"]), fmt(fs)))
    S.step("Overall factor", r"%s = %s \times %s = %s" % (V["f"].tex(), fmt(fe), fmt(fs), fmt(ft)))
    S.step("Check directly", r"\frac{%s}{%s} = \frac{%s}{%s} = %s" % (V["M_s"].tex(), V["M_a"].tex(), fmt(d["M_s"]), fmt(d["M_a"]), fmt(ft)))
    S.step("Fraction of the salt that is active drug", r"\frac{1}{%s} = %s = %s\%%" % (fmt(ft), fmt(1 / ft), fmt(100 / ft)))
    S.step("Mass of salt to weigh", r"%s \times %s = %s\,\mathrm{mg} \times %s = %s\,\mathrm{mg}" % (
        V["m_eq"].tex(), V["f"].tex(), fmt(d["m_eq"]), fmt(ft), fmt(w)))
    S.answer(ft, "", "Every mg of oseltamivir carboxylate equivalent needs %s mg of oseltamivir phosphate: the ethyl ester adds "
             "%s%% and the phosphate salt another %s%%. Only %s%% of the salt's mass is the active carboxylate, so the "
             "standard needs %s mg of salt." % (fmt(ft), fmt(100 * (fe - 1)), fmt(100 * (fs - 1)), fmt(100 / ft), fmt(w)))
    return S


_CARD32_MW = Card("l32_mw_ratio", "Salt and ester mass factor: prodrug salt vs active drug (practice: oseltamivir)",
    r"{{f}} = \frac{{{M_s}}}{{{M_a}}} = \frac{{{M_e}}}{{{M_a}}} \times \frac{{{M_s}}}{{{M_e}}}",
    [Var("M_s", r"M_{salt}", "oseltamivir phosphate, C₁₆H₂₈N₂O₄·H₃PO₄", "g/mol", "input", "formula, IUPAC atomic weights", 410.40),
     Var("M_e", r"M_{ester}", "oseltamivir (ethyl ester base), C₁₆H₂₈N₂O₄", "g/mol", "input", "formula, IUPAC atomic weights", 312.41),
     Var("M_a", r"M_{acid}", "oseltamivir carboxylate (active), C₁₄H₂₄N₂O₄", "g/mol", "input", "formula, IUPAC atomic weights", 284.35),
     Var("m_eq", r"m_{eq}", "carboxylate-equivalent mass needed in the standard", "mg", "input", "case", 10.00),
     Var("f", r"f", "mass of salt per mass of active carboxylate", "", "result", "calculated")],
    "A pharmacokinetics laboratory needs a calibration stock containing [[m_eq]] of oseltamivir carboxylate equivalents but "
    "has only the prodrug salt, oseltamivir phosphate ([[M_s]]). The free ester base is [[M_e]] and the active carboxylate "
    "[[M_a]]. Calculate the ester factor, the salt factor, the overall factor and the mass of salt to weigh.",
    _solve_mw,
    conversions=["1 mg = 1000 µg", "mmol = mg ÷ (g/mol)", "salt factor = M(salt)/M(base)"],
    result_name="f", source="molar-mass bookkeeping for ester prodrugs and salts (L32 notes)")


# ---- 4. First-order activation in outlier patients (mixed worked example) -----
def _solve_act(card, d):
    S = Solution(card, d); V = card.V
    _c32_pos(d, "th", "t", "fL", "fH")
    k0 = math.log(2) / d["th"]
    out = {}
    S.general().given().unit_check("t½ and t are both in hours, so k is in h⁻¹ and the product k·t is unit-free, as an "
                                   "exponent must be. (1 h = 60 min; enter both times in the same unit.)")
    S.step("Typical rate constant", r"%s = \frac{\ln 2}{%s} = \frac{0.6931}{%s\,\mathrm{h}} = %s\,\mathrm{h^{-1}}" % (
        V["k"].tex(), V["th"].tex(), fmt(d["th"]), fmt(k0)))
    for tag, f in (("L", d["fL"]), ("typical", 1.0), ("H", d["fH"])):
        k = k0 * f
        kt = k * d["t"]
        pc = 100 * (1 - math.exp(-kt))
        out[tag] = pc
        name = {"L": "Patient L", "typical": "Typical patient", "H": "Patient H"}[tag]
        if f != 1.0:
            fk = V["fL"] if tag == "L" else V["fH"]
            S.step("%s: rate constant" % name, r"k = %s \times %s = %s \times %s = %s\,\mathrm{h^{-1}}" % (
                fk.tex(), V["k"].tex(), fmt(f), fmt(k0), fmt(k)))
        S.step("%s: exponent" % name, r"k %s = %s\,\mathrm{h^{-1}} \times %s\,\mathrm{h} = %s" % (V["t"].tex(), fmt(k), fmt(d["t"]), fmt(kt)))
        S.step("%s: percent converted" % name, r"100\left(1 - e^{-%s}\right) = 100(1 - %s) = %s\%%" % (fmt(kt), fmt(math.exp(-kt)), fmt(pc)))
    sp = out["H"] - out["L"]
    S.step("Between-patient spread", r"%s - %s = %s\ \text{percentage points}" % (fmt(out["H"]), fmt(out["L"]), fmt(sp)))
    S.answer(sp, "", "At %s h, patient L has converted %s%% and patient H %s%% (typical %s%%): a %s-percentage-point spread "
             "from a %s-fold range of esterase activity. Re-solve with t½ = 0.25 h to see a faster promoiety shrink the "
             "spread." % (fmt(d["t"]), fmt(out["L"]), fmt(out["H"]), fmt(out["typical"]), fmt(sp), fmt(d["fH"] / d["fL"])))
    return S


_CARD32_ACT = Card("l32_activation_spread", "First-order prodrug activation in two outlier patients (worked example: how reliably does an ester switch on?)",
    r"{{k}} = \frac{\ln 2}{{{th}}};\quad \%_{conv} = 100\left(1 - e^{-k{{t}}}\right);\quad k_{L} = {{fL}}{{k}},\ k_{H} = {{fH}}{{k}}",
    [Var("th", r"t_{1/2}", "activation half-life in a typical patient", "h", "input", "case (illustrative)", 1.5),
     Var("t", r"t", "time point of interest", "h", "input", "case", 2.0),
     Var("fL", r"f_{L}", "esterase activity of patient L relative to typical", "", "input", "case (illustrative)", 0.3),
     Var("fH", r"f_{H}", "esterase activity of patient H relative to typical", "", "input", "case (illustrative)", 3.0),
     Var("k", r"k", "typical first-order activation rate constant", "h^-1", "intermediate", "calculated")],
    "An ester prodrug is activated by first-order hydrolysis with t½ = [[th]] in a typical patient. Patient L has [[fL]] × "
    "the typical esterase activity and patient H [[fH]] ×. What percentage has been converted to active drug in each at "
    "t = [[t]], and what is the between-patient spread?",
    _solve_act,
    conversions=["1 h = 60 min", "k in h⁻¹ ÷ 60 = k in min⁻¹", "ln 2 = 0.6931"],
    result_name=r"\text{spread}", source="first-order kinetics of enzymatic activation (L32 notes)")


# ---- 5. Conversion half-life in minutes and hours (new practice) --------------
def _solve_fos(card, d):
    S = Solution(card, d); V = card.V
    _c32_pos(d, "th", "t")
    k = math.log(2) / d["th"]
    kt = k * d["t"]
    pc = 100 * (1 - math.exp(-kt))
    t90 = math.log(10) / k
    S.general().given().unit_check("Both times are in minutes, so k is in min⁻¹ and k·t is unit-free. To report k per hour, "
                                   "multiply by 60 min/h; to report a time in hours, divide minutes by 60 min/h.")
    S.step("Rate constant", r"%s = \frac{\ln 2}{%s} = \frac{0.6931}{%s\,\mathrm{min}} = %s\,\mathrm{min^{-1}}" % (
        V["k"].tex(), V["th"].tex(), fmt(d["th"]), fmt(k)))
    S.step("Rate constant per hour", r"%s\,\mathrm{min^{-1}} \times 60\,\mathrm{min/h} = %s\,\mathrm{h^{-1}}" % (fmt(k), fmt(60 * k)))
    S.step("Exponent", r"%s %s = %s \times %s = %s" % (V["k"].tex(), V["t"].tex(), fmt(k), fmt(d["t"]), fmt(kt)))
    S.step("Fraction remaining as prodrug", r"e^{-%s} = %s" % (fmt(kt), fmt(math.exp(-kt))))
    S.step("Percent converted", r"100(1 - %s) = %s\%%" % (fmt(math.exp(-kt)), fmt(pc)))
    S.step("Time to 90% conversion",
           r"t_{90} = \frac{\ln 10}{%s} = \frac{2.303}{%s\,\mathrm{min^{-1}}} = %s\,\mathrm{min} = \frac{%s}{60}\,\mathrm{h} = %s\,\mathrm{h}" % (
               V["k"].tex(), fmt(k), fmt(t90), fmt(t90), fmt(t90 / 60)))
    S.answer(pc, "%", "After %s min (%s h), %s%% of the fosphenytoin has been converted to phenytoin; 90%% conversion takes "
             "%s min (%s h). Because each half-life halves what is left, %s min is exactly %s half-lives." % (
                 fmt(d["t"]), fmt(d["t"] / 60), fmt(pc), fmt(t90), fmt(t90 / 60), fmt(d["t"]), fmt(d["t"] / d["th"])))
    return S


_CARD32_FOS = Card("l32_conversion_halflife", "Fraction of a prodrug converted at time t, in minutes and hours (practice: fosphenytoin)",
    r"{{k}} = \frac{\ln 2}{{{th}}};\quad \%_{conv} = 100\left(1 - e^{-{{k}}{{t}}}\right);\quad t_{90} = \frac{\ln 10}{{{k}}}",
    [Var("th", r"t_{1/2}", "conversion half-life of fosphenytoin to phenytoin in plasma", "min", "input", "approximate literature value (≈ 15 min)", 15),
     Var("t", r"t", "time after the infusion ends", "min", "input", "case", 45),
     Var("k", r"k", "first-order conversion rate constant", "min^-1", "intermediate", "calculated")],
    "In the emergency department a pharmacist explains to a resident why phenytoin levels drawn right after a fosphenytoin "
    "infusion can mislead. Phosphatases convert fosphenytoin to phenytoin with a half-life of about [[th]]. What percentage "
    "has been converted [[t]] after the end of the infusion, and how long (in minutes and hours) does 90% conversion take?",
    _solve_fos,
    conversions=["1 h = 60 min", "k (h⁻¹) = 60 × k (min⁻¹)", "ln 2 = 0.6931; ln 10 = 2.303"],
    result_name=r"\%_{conv}", source="first-order activation; tripartite fosphenytoin (L32 notes)")


# ---------------------------------------------------------------------------
# L32 lecture-notes figures and calculators: prodrugs
# Every prodrug is drawn directly ABOVE its active drug on one template, so the
# shared core sits in the same place and only the promoiety changes.
# Blue halo = promoiety; scissors = the bond the activating enzyme breaks.
# Stereocentres are drawn with plain bonds (constitution only).
# ---------------------------------------------------------------------------
import math as _m
from matplotlib.patches import FancyArrowPatch as _Arrow, FancyBboxPatch as _Box

_PRO = COL["ester"]      # promoiety halo
_ACT = COL["oh"]         # group unmasked in the active drug


def _dir(a, d=1.0):
    return d * _m.cos(_m.radians(a)), d * _m.sin(_m.radians(a))


def _at(m, i, a, label="", d=1.0, order=1, hl=None, color=None, anchor=None):
    """New atom at angle a (deg) from atom i, bonded to it."""
    x, y = m.pos(i)
    dx, dy = _dir(a, d)
    k = m.add(x + dx, y + dy, label, color=color, anchor=anchor)
    m.bond(i, k, order, hl=hl)
    return k


def _phenyl_at(m, i, a):
    """Phenyl ring whose ipso carbon is one bond from atom i along angle a."""
    x, y = m.pos(i)
    cx, cy = x + 2 * _m.cos(_m.radians(a)), y + 2 * _m.sin(_m.radians(a))
    r = ring(m, cx, cy, 6, rot=a + 180)
    kekule(m, r, 0)
    m.bond(i, r[0])
    return r


def _cut(m, i, j, text="", dx=0.0, dy=-0.55, color=None):
    x1, y1 = m.pos(i); x2, y2 = m.pos(j)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    m.note(mx, my, "✂", color=color or COL["drug"], size=13)
    if text:
        m.note(mx + dx, my + dy, text, color=color or COL["drug"], size=7.5)


def _pair(top, bot, top_name, bot_name, enzyme, side_note, title, x_arrow=None, space=3.4, gap_names=0.0):
    """Stack prodrug (top) over active drug (bottom) at one scale, with a labelled arrow."""
    t0 = extent(top, pad=0.0); b0 = extent(bot, pad=0.0)
    dy = (t0[2] - space) - b0[3]
    m = Mol("pair")
    merge(m, top)
    merge(m, bot, 0, dy)
    xl = min(t0[0], b0[0])
    m.note(xl, t0[3] + 0.75, top_name, color=COL["ink"], size=9.5, ha="left", weight="bold")
    m.note(xl, b0[3] + dy + 0.75 + gap_names, bot_name, color=COL["ink"], size=9.5, ha="left", weight="bold")
    xa = x_arrow if x_arrow is not None else xl + 0.6
    y1, y2 = t0[2] - 0.25, b0[3] + dy + 1.35
    m.note(xa + 0.35, (y1 + y2) / 2 + 0.28, enzyme, color=COL["ester"], size=9, ha="left", weight="bold")
    m.note(xa + 0.35, (y1 + y2) / 2 - 0.28, side_note, color=COL["muted"], size=8.5, ha="left")

    def extra(ax):
        ax.add_patch(_Arrow((xa, y1), (xa, y2), arrowstyle="-|>", mutation_scale=15, lw=1.5, color=COL["ink"]))
    show(m, title, extra=extra)


# ---- enalapril / enalaprilat -----------------------------------------------
def enalapril(ester=True):
    m = Mol("enalapril" if ester else "enalaprilat")
    ph = ring(m, -C30, -0.5, 6, rot=30); kekule(m, ph, 0)
    p1 = m.add(C30, 0.5); m.bond(ph[0], p1)
    p2 = m.add(2 * C30, 0.0); m.bond(p1, p2)
    p3 = m.add(3 * C30, 0.5); m.bond(p2, p3)
    nh = m.add(4 * C30, 0.0, "NH"); m.bond(p3, nh)
    p5 = m.add(4 * C30, -1.0); m.bond(nh, p5)
    m.bond(p5, m.add(3 * C30, -1.5))                     # alanine CH3
    p6 = m.add(5 * C30, -1.5); m.bond(p5, p6)
    m.bond(p6, m.add(5 * C30, -2.5, "O"), 2)               # amide C=O
    R = 1 / (2 * _m.sin(_m.pi / 5))
    nx, ny = 6 * C30, -1.0
    pr = ring(m, nx + R * _m.cos(_m.radians(30)), ny + R * _m.sin(_m.radians(30)), 5, rot=210,
              labels=["N", "", "", "", ""])
    m.bond(p6, pr[0])
    ox, oy = outward(m, pr[1])
    a = _m.degrees(_m.atan2(oy, ox))
    c = _at(m, pr[1], a)
    _at(m, c, a + 60, "O", order=2)
    _at(m, c, a - 60, "OH")
    # the masked / unmasked carboxylate (zinc-binding group)
    ce = m.add(3 * C30, 1.5); m.bond(p3, ce)
    m.bond(ce, m.add(2 * C30, 2.0, "O"), 2)
    if ester:
        o = m.add(4 * C30, 2.0, "O"); m.bond(ce, o)
        c1 = m.add(5 * C30, 1.5); m.bond(o, c1, hl=_PRO)
        c2 = m.add(6 * C30, 2.0); m.bond(c1, c2, hl=_PRO)
        _cut(m, ce, o)
        m.note(5.5 * C30, 2.75, "ethyl promoiety", color=_PRO, size=8)
    else:
        o = m.add(4 * C30, 2.0, "OH", color=_ACT); m.bond(ce, o, hl=_ACT)
        m.note(5.9 * C30, 2.0, "Zn²⁺-binding carboxylate", color=_ACT, size=8, ha="left")
    return m


def fig_enalapril():
    _pair(enalapril(True), enalapril(False),
          "Enalapril (prodrug)  C₂₀H₂₈N₂O₅", "Enalaprilat (active)  C₁₈H₂₄N₂O₅",
          "carboxylesterase 1 (liver)", "+ H₂O  →  releases ethanol",
          "Enalapril → enalaprilat: an ethyl ester masks one carboxylic acid", x_arrow=-1.2)
    _md(["The ethyl ester (blue) removes one negative charge, so the prodrug is absorbed far better from the gut. "
         "Hepatic carboxylesterase cuts the acyl–oxygen bond (✂) and releases ethanol, a harmless leaving group. "
         "The freed carboxylate (red) is the group that binds the active-site **zinc ion of ACE**. "
         "*Three stereocentres (all S) are drawn with plain bonds.*"])


# ---- valacyclovir / acyclovir ---------------------------------------------
def _guanine(m):
    """Guanine, six-membered ring left, N9 lower right. Returns dict of key atoms."""
    r6 = ring(m, 0, 0, 6, rot=30, labels=["", "", "HN", "", "N", ""])  # C5, C6, N1, C2, N3, C4
    C5, C6, N1, C2, N3, C4 = r6
    r5 = fuse(m, C4, C5, 5, labels=["N", "", "N"])                      # from C5: N7, C8, N9
    N7, C8, N9 = r5[2], r5[3], r5[4]
    set_double(m, [(C2, N3), (C4, C5), (N7, C8)])
    m.bond(C6, m.add(m.pos(C6)[0], m.pos(C6)[1] + 1.0, "O"), 2)
    sub(m, C2, "H₂N", angle=210, anchor=2)
    return dict(N9=N9, C8=C8)


def valacyclovir(prodrug=True):
    m = Mol("valacyclovir" if prodrug else "acyclovir")
    g = _guanine(m)
    ids = chain(m, g["N9"], 5, heading=0, first=-1)     # CH2-O-CH2-CH2-O
    label(m, ids[2], "O")
    if not prodrug:
        label(m, ids[5], "OH")
        m.atoms[ids[5]]["color"] = _ACT
        m.bonds[-1]["hl"] = _ACT
        return m
    label(m, ids[5], "O")
    x, y = m.pos(ids[5])
    c = m.add(x + C30, y - 0.5); m.bond(ids[5], c)           # ester C=O carbon
    _at(m, c, 270, "O", order=2)
    ca = m.add(x + 2 * C30, y); m.bond(c, ca, hl=_PRO)       # valine C-alpha
    _at(m, ca, 90, "NH₂", hl=_PRO)
    cb = m.add(x + 3 * C30, y - 0.5); m.bond(ca, cb, hl=_PRO)
    _at(m, cb, -90, hl=_PRO)
    _at(m, cb, 30, hl=_PRO)
    _cut(m, ids[5], c)
    m.note(x + 3.3 * C30, y - 1.9, "L-valyl promoiety", color=_PRO, size=8, ha="left")
    return m


def fig_valacyclovir():
    _pair(valacyclovir(True), valacyclovir(False),
          "Valacyclovir (prodrug)  C₁₃H₂₀N₆O₄", "Acyclovir (released)  C₈H₁₁N₅O₃",
          "valacyclovir hydrolase (BPHL), gut wall and liver", "after uptake by the PEPT1 peptide transporter",
          "Valacyclovir → acyclovir: an amino-acid ester rides a nutrient transporter", x_arrow=-1.6)
    _md(["The L-valine ester (blue) makes the molecule look like a dipeptide, so the intestinal **PEPT1** transporter carries it in; "
         "oral bioavailability of acyclovir rises about three- to five-fold. An esterase-type hydrolase then cuts the ester (✂) and releases valine. "
         "Acyclovir is itself still a prodrug: **viral thymidine kinase** adds the first phosphate, host kinases add two more, and "
         "acyclovir triphosphate blocks the **viral DNA polymerase** (chain terminator). Because only infected cells make the first phosphate, "
         "uninfected cells are spared — activation, not binding, supplies the selectivity."])


# ---- levodopa / dopamine / carbidopa ---------------------------------------
def catechol_amine(kind):
    """kind: 'levodopa', 'dopamine', 'carbidopa'. Catechol ring left, side chain right."""
    m = Mol(kind)
    r = ring(m, 0, 0, 6, rot=0); kekule(m, r, 0)
    sub(m, r[2], "HO", angle=120, anchor=1)
    sub(m, r[3], "HO", angle=180, anchor=1)
    c1 = m.add(1 + C30, -0.5); m.bond(r[0], c1)
    c2 = m.add(1 + 2 * C30, 0.0); m.bond(c1, c2)
    if kind == "carbidopa":
        n = _at(m, c2, 90, "NH"); _at(m, n, 90, "NH₂")
        _at(m, c2, 270)                                       # alpha-methyl
    else:
        _at(m, c2, 90, "NH₂", hl=_ACT if kind == "dopamine" else None)
    if kind in ("levodopa", "carbidopa"):
        c3 = m.add(1 + 3 * C30, -0.5); m.bond(c2, c3, hl=_PRO if kind == "levodopa" else None)
        _at(m, c3, 330, "OH", hl=_PRO if kind == "levodopa" else None)
        _at(m, c3, 270, "O", order=2,
            hl=_PRO if kind == "levodopa" else None)
        if kind == "levodopa":
            _cut(m, c2, c3, "")
            m.note(1 + 3.6 * C30, 0.55, "lost as CO₂", color=_PRO, size=8, ha="left")
    return m


def fig_levodopa():
    _pair(catechol_amine("levodopa"), catechol_amine("dopamine"),
          "Levodopa (L-DOPA)  C₉H₁₁NO₄", "Dopamine (active)  C₈H₁₁NO₂",
          "aromatic L-amino-acid decarboxylase (AADC, PLP)", "inside the brain: −CO₂",
          "Levodopa → dopamine: the carboxyl group is the 'ticket' across the blood–brain barrier", x_arrow=-2.6)
    _md(["Dopamine cannot cross the blood–brain barrier. Levodopa is an amino acid, so the **LAT1** large neutral amino-acid "
         "transporter carries it into the brain (a *Trojan-horse* strategy). There **AADC** removes the carboxyl group as CO₂ (✂). "
         "No carrier group is cut off — the drug is made by a metabolic change — so levodopa behaves as a **bioprecursor**."])
    show(catechol_amine("carbidopa"), "Carbidopa  C₁₀H₁₄N₂O₄: a hydrazine AADC inhibitor that stays outside the brain")
    _md(["AADC is also present in gut wall, liver and blood vessels, so most of a levodopa dose would otherwise become dopamine in the periphery "
         "(nausea, low brain delivery). Carbidopa's hydrazine (–NH–NH₂) traps the PLP cofactor; it is too polar to cross the blood–brain barrier. "
         "The combination therefore protects the prodrug **until** it reaches the site where activation is wanted — "
         "a lesson that applies to any prodrug whose activating enzyme is widespread."])


# ---- oseltamivir / oseltamivir carboxylate ---------------------------------
def oseltamivir(ester=True):
    m = Mol("oseltamivir" if ester else "oseltamivir carboxylate")
    r = ring(m, 0, 0, 6, rot=0)          # C1 right, C2 upper right, C3 upper left, C4 left, C5 lower left, C6 lower right
    C1, C2, C3, C4, C5, C6 = r
    set_double(m, [(C1, C2)])
    c = _at(m, C1, 0)
    _at(m, c, 60, "O", order=2)
    if ester:
        o = _at(m, c, -60, "O")
        e1 = _at(m, o, 0, hl=_PRO); _at(m, e1, 60, hl=_PRO)
        _cut(m, c, o)
        m.note(m.pos(e1)[0] + 0.2, m.pos(e1)[1] - 0.6, "ethyl promoiety", color=_PRO, size=8)
    else:
        m.bond(c, m.add(m.pos(c)[0] + 0.5, m.pos(c)[1] - C30, "OH", color=_ACT), hl=_ACT)
    o3 = _at(m, C3, 120, "O")
    ch = _at(m, o3, 60)
    a1 = _at(m, ch, 120); _at(m, a1, 60)
    b1 = _at(m, ch, 0); _at(m, b1, 60)
    n = _at(m, C4, 180, "N")
    m.note(m.pos(n)[0], m.pos(n)[1] - 0.42, "H", color=COL["ink"], size=10.5)
    ac = _at(m, n, 150)
    _at(m, ac, 90, "O", order=2)
    _at(m, ac, 210)
    _at(m, C5, 240, "H₂N", anchor=2)
    return m


def fig_oseltamivir():
    _pair(oseltamivir(True), oseltamivir(False),
          "Oseltamivir (prodrug)  C₁₆H₂₈N₂O₄", "Oseltamivir carboxylate (active)  C₁₄H₂₄N₂O₄",
          "carboxylesterase 1 (liver)", "releases ethanol",
          "Oseltamivir → oseltamivir carboxylate: the same ethyl-ester trick on a neuraminidase inhibitor", x_arrow=-4.6)
    _md(["The carboxylate binds a cluster of arginines in the influenza **neuraminidase** active site, but as a free acid the drug "
         "is poorly absorbed. The ethyl ester raises oral bioavailability to roughly 80%; hepatic **CES1** removes it. "
         "Note the design is the same as enalapril: keep the pharmacophore, mask the one charge that blocks absorption. "
         "*Three ring stereocentres (3R,4R,5S) are drawn with plain bonds.*"])


# ---- fosphenytoin / phenytoin ---------------------------------------------
def hydantoin(stage):
    """stage: 'fos' (fosphenytoin), 'hm' (N-hydroxymethyl intermediate), 'phen' (phenytoin)."""
    m = Mol(stage)
    r = ring(m, 0, 0, 5, rot=180)            # v0 C5 (left), v1 N1, v2 C2, v3 N3, v4 C4
    C5, N1, C2, N3, C4 = r
    label(m, N1, "NH")
    _at(m, C4, 108, "O", order=2)
    _at(m, C2, 324, "O", order=2)
    _phenyl_at(m, C5, 158)
    _phenyl_at(m, C5, 236)
    if stage == "phen":
        label(m, N3, "NH")
        m.atoms[N3]["color"] = _ACT
        return m
    label(m, N3, "N")
    ch2 = _at(m, N3, 36, hl=_PRO)
    if stage == "hm":
        _at(m, ch2, -24, "OH", hl=_PRO)
        _cut(m, N3, ch2, "", color=COL["muted"])
        return m
    o = _at(m, ch2, -24, "O", hl=_PRO)
    p = m.add(m.pos(o)[0] + 1.35, m.pos(o)[1], "P", color=COL["phos"]); m.bond(o, p)
    m.bond(p, m.add(m.pos(p)[0], m.pos(p)[1] + 1.05, "O"), 2)
    m.bond(p, m.add(m.pos(p)[0], m.pos(p)[1] - 1.05, "OH"))
    m.bond(p, m.add(m.pos(p)[0] + 1.1, m.pos(p)[1], "OH"))
    _cut(m, o, p)
    return m


def fig_fosphenytoin():
    top, mid, bot = hydantoin("fos"), hydantoin("hm"), hydantoin("phen")
    scheme([top, ("alkaline phosphatase", "− phosphate"), mid, ("spontaneous", "− HCHO"), bot],
           title="Fosphenytoin → phenytoin: a tripartite prodrug with a self-immolative CH₂ linker", per_row=3, gap=3.4)
    _md(["**Left** fosphenytoin C₁₆H₁₅N₂O₆P · **middle** N-hydroxymethyl intermediate (not isolated) · **right** phenytoin C₁₅H₁₂N₂O₂.",
         "",
         "Phenytoin is too insoluble in water for a convenient intravenous solution. A phosphate makes the prodrug very water soluble, "
         "but a phosphate fixed directly on the imide nitrogen would be awkward to remove, so a one-carbon **spacer** (CH₂) sits between them. "
         "Phosphatases cut the P–O bond (✂); the N–CH₂OH that remains is unstable and falls apart on its own, releasing "
         "**formaldehyde** and phenytoin. That is the meaning of *self-immolative*: once the trigger is removed, the linker destroys itself."])


# ---- concept figure: four prodrug designs (Pathway figure kit) ---------------
def fig_prodrug_types():
    P = Pathway("Four ways to build a prodrug: what is cut, and what is left", 10.5, 6.6)
    bands = [(45.0, "A. Bipartite (enalapril, oseltamivir)"),
             (30.5, "B. Tripartite (fosphenytoin, disoproxils)"),
             (16.0, "C. Mutual prodrug (sultamicillin)"),
             (1.5, "D. Bioprecursor (levodopa, clopidogrel)")]
    for y0, lab in bands:
        P.compartment(1, y0, 98, 13.5, lab, "cyto", membrane=False)
    # A
    y = 50.0
    P.node("a0", 20, y, "drug–carrier", "metabolite")
    P.node("aE", 42, y + 4.6, "esterase", "enzyme", fs=8.6)
    P.drug("aD", 62, y + 1.2, "drug")
    P.node("aC", 84, y, "carrier\n(e.g. ethanol)", "cofactor", fs=8.6)
    P.catalyze("aE", "a0", "aD")
    P.arrow("aD", "aC", "indirect", "+", label_side=1)
    # B
    y = 35.5
    P.node("b0", 18, y, "drug–linker–carrier", "metabolite", fs=8.8)
    P.node("bE", 38, y + 4.4, "esterase or\nphosphatase", "enzyme", fs=8.0, w=14)
    P.node("b1", 50, y, "drug–linker", "metabolite", fs=8.8)
    P.drug("bD", 70, y + 1.2, "drug")
    P.node("bC", 88, y, "HCHO +\ncarrier", "cofactor", fs=8.6)
    P.catalyze("bE", "b0", "b1")
    P.arrow("b1", "bD", "indirect", "falls apart", label_side=1)
    P.arrow("bD", "bC", "indirect", "+", label_side=1)
    # C
    y = 21.0
    P.node("c0", 20, y, "drug A–drug B", "metabolite")
    P.node("cE", 42, y + 4.6, "esterase", "enzyme", fs=8.6)
    P.drug("cA", 62, y + 1.2, "drug A")
    P.drug("cB", 82, y + 1.2, "drug B")
    P.catalyze("cE", "c0", "cA")
    P.arrow("cA", "cB", "indirect", "+", label_side=1)
    # D
    y = 6.5
    P.node("d0", 20, y, "inactive precursor", "metabolite")
    P.node("dE", 42, y + 4.6, "CYP450 / AADC", "enzyme", fs=8.6)
    P.drug("dD", 62, y + 1.2, "drug")
    P.note(72, y + 2.6, "nothing is shed as a carrier:\nthe molecule itself is changed", fs=8.2)
    P.catalyze("dE", "d0", "dD")
    P.show()


def fig_ester_route():
    P = Pathway("Where an oral ester prodrug is switched on (enalapril)", 10.5, 6.2)
    P.compartment(1.5, 6, 19, 50, "gut lumen", "lumen", membrane=False)
    P.compartment(23, 6, 17, 50, "enterocyte", "cyto")
    P.compartment(43, 6, 25, 50, "hepatocyte", "mito")
    P.compartment(71, 6, 27.5, 50, "blood → tissue", "blood", membrane=False)
    P.node("pro", 11, 42, "enalapril\n(ester)", "metabolite")
    P.node("acid", 11, 20, "enalaprilat\n(diacid)", "metabolite")
    P.node("ent", 31.5, 42, "enalapril", "metabolite")
    P.node("hep", 52, 42, "enalapril", "metabolite")
    P.node("ces", 52, 30.5, "CES1", "enzyme")
    P.node("hacid", 52, 18, "enalaprilat", "metabolite")
    P.node("etoh", 62, 24.5, "ethanol", "cofactor", fs=8.2, w=9)
    P.drug("d", 79, 21, "enalaprilat")
    P.node("ace", 86.5, 42, "ACE", "enzyme")
    P.arrow("pro", "ent", "transport", "passive", label_side=1)
    P.arrow("ent", "hep", "transport", "portal vein", label_side=1)
    P.arrow("hep", "ces", "convert")
    P.arrow("ces", "hacid", "convert", "ester cut", label_side=-1)
    P.arrow("ces", "etoh", "indirect")
    P.arrow("hacid", "d", "transport")
    P.arrow("d", "ace", "inhibit", "Zn²⁺ site", label_side=-1)
    P.arrow("acid", (23.0, 20), "indirect", "", lw=1.4)
    P.ax.plot([23.0], [20], marker="x", ms=9, mew=2.2, color=PW["coral"], zorder=6)
    P.note(3, 13.5, "charged at pH 6.5:\npoorly absorbed", fs=8.0)
    P.note(73, 12.5, "the diacid binds the zinc\nmetalloprotease ACE", fs=8.0)
    P.note(2, 4.2, "The neutral ester crosses the gut wall; liver carboxylesterase 1 (CES1) then unmasks the carboxylate the target needs.", fs=8.2)
    P.legend()
    P.show()


# ---- calculators -----------------------------------------------------------
def activation_calc(t_half_typical_h=1.5, time_h=2.0, activity_low=0.3, activity_high=3.0):
    """First-order activation (prodrug -> drug) for a typical patient and two outliers."""
    import matplotlib.pyplot as plt
    import numpy as np
    k0 = _m.log(2) / t_half_typical_h
    rows, out = [], {}
    lines = ["**Model.** Activation is first order: [prodrug]ₜ = [prodrug]₀·e^(−kt), so % converted = 100·(1 − e^(−kt)).",
             "",
             "1. Typical patient: k = ln 2 / t½ = 0.693 / %.2f h = **%.3f h⁻¹**" % (t_half_typical_h, k0)]
    step = 2
    for name, f in (("low-esterase patient", activity_low), ("typical patient", 1.0), ("high-esterase patient", activity_high)):
        k = k0 * f
        th = _m.log(2) / k if k > 0 else float("inf")
        pc = 100 * (1 - _m.exp(-k * time_h))
        out[name] = dict(k=k, t_half=th, pct=pc)
        if f != 1.0:
            lines.append("%d. %s (enzyme activity × %.2g): k = %.3f × %.2g = **%.3f h⁻¹**, t½ = 0.693 / k = **%.2f h**"
                         % (step, name.capitalize(), f, k0, f, k, th))
            step += 1
        rows.append("| %s | %.2g | %.3f | %.2f | %.1f |" % (name, f, k, th, pc))
    lines.append("%d. Fraction converted at t = %.1f h: 100·(1 − e^(−k·%.1f)) for each patient (table)." % (step, time_h, time_h))
    spread = out["high-esterase patient"]["pct"] - out["low-esterase patient"]["pct"]
    lines += ["", "| Patient | activity × | k (h⁻¹) | t½ (h) | %% active drug at %.1f h |" % time_h, "|---|---|---|---|---|"] + rows
    lines += ["", "**Between-patient spread at %.1f h = %.1f − %.1f = %.1f percentage points.**"
              % (time_h, out["high-esterase patient"]["pct"], out["low-esterase patient"]["pct"], spread)]
    _md(lines)
    t = np.linspace(0, max(4 * t_half_typical_h / max(activity_low, 1e-3) / 2, 2 * time_h), 300)
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    for (name, d), col in zip(out.items(), (COL["dbl"], COL["ester"], COL["phos"])):
        ax.plot(t, 100 * (1 - np.exp(-d["k"] * t)), lw=2.2, color=col, label="%s (t½ %.2f h)" % (name, d["t_half"]))
    ax.axvline(time_h, color=COL["muted"], ls="--", lw=1)
    ax.set_xlabel("time after the dose (h)"); ax.set_ylabel("% converted to active drug")
    ax.set_ylim(-2, 102); ax.grid(alpha=.25); ax.legend(fontsize=8, loc="lower right")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_title("First-order activation in three patients", fontsize=10.5, loc="left")
    plt.tight_layout(); plt.show()
    out["spread_pct_points"] = spread
    return out


def logD_ester_calc(logP_acid=1.0, pKa_acid=4.0, logP_ester=2.3, pH=6.5):
    """Acid vs its ethyl ester: log D at a chosen pH (ester has no ionizable group)."""
    ratio = 10 ** (pH - pKa_acid)
    corr = _m.log10(1 + ratio)
    logD_acid = logP_acid - corr
    logD_ester = logP_ester
    diff = logD_ester - logD_acid
    _md(["*Illustrative values for a generic carboxylic-acid drug and its ethyl ester (not measured data for a named drug).*",
         "",
         "1. Acid: [A⁻]/[HA] = 10^(pH − pKa) = 10^(%.1f − %.1f) = **%.3g** → %.2f%% ionized" % (pH, pKa_acid, ratio, 100 * ratio / (1 + ratio)),
         "2. Acid: log D = log P − log₁₀(1 + 10^(pH − pKa)) = %.2f − %.2f = **%.2f**" % (logP_acid, corr, logD_acid),
         "3. Ethyl ester: no ionizable group, so log D = log P = **%.2f** at any pH" % logD_ester,
         "4. Difference: %.2f − (%.2f) = **%.2f log units** → the ester partitions about **%.0f-fold** more into octanol at pH %.1f"
         % (logD_ester, logD_acid, diff, 10 ** diff, pH),
         "",
         "Most of the gain comes from removing the charge (step 2), not from the two extra carbons (log P %.2f → %.2f)." % (logP_acid, logP_ester)])
    return dict(ratio=ratio, logD_acid=logD_acid, logD_ester=logD_ester, difference=diff, fold=10 ** diff)


# ---- prodrug design chooser (objective L32.4: design a prodrug, name the enzyme) ---
# Each problem maps to a design: the group the promoiety attaches to, the strategy,
# the activating agent and a caution. Original teaching table (no doses/intervals).
_PRODRUG_DESIGNS = {
    "An acidic drug (carboxyl/phosphate) is barely absorbed orally": dict(
        group="the carboxylic acid or phosphate",
        strategy="esterify the acid (or use a carbonate–acetal 'disoproxil' arm on a phosphonate) to mask the charge",
        enzyme="carboxylesterase 1 (CES1), mainly in the liver (alkaline phosphatase for a phosphate)",
        kind="carrier-linked, bipartite (tripartite for the phosphonate arm)",
        caution="the alcohol released must be harmless at the amount freed, and activation must beat excretion of the intact ester",
        example="enalapril → enalaprilat; tenofovir disoproxil → tenofovir"),
    "A neutral, poorly soluble drug must be given by injection": dict(
        group="a hydroxyl or imide N–H on the drug",
        strategy="attach an ionizable phosphate through a self-immolative CH2 linker for water solubility",
        enzyme="alkaline phosphatase in plasma and on the intestinal/liver brush border",
        kind="carrier-linked, tripartite (self-immolative linker)",
        caution="the linker must fall apart quickly after the phosphate is removed; it releases a small amount of formaldehyde",
        example="fosphenytoin → phenytoin"),
    "A polar drug cannot cross the blood–brain barrier": dict(
        group="add a group the drug is missing, or mask one, so a nutrient transporter recognises it",
        strategy="make the molecule resemble an amino acid (or peptide) so a solute-carrier transporter carries it in (Trojan-horse)",
        enzyme="an intracellular decarboxylase or hydrolase that regenerates the drug after uptake",
        kind="bioprecursor (levodopa) or carrier-linked amino-acid ester (valacyclovir)",
        caution="the same transporter and enzyme may act in the periphery, wasting drug; a peripheral blocker may be needed",
        example="levodopa → dopamine (with carbidopa); valacyclovir → acyclovir"),
    "A drug is too toxic to healthy tissue and should act mainly in a tumour": dict(
        group="a group that only an enzyme enriched in or near the target tissue can remove",
        strategy="mask the drug so it is unmasked preferentially where the activating enzyme is concentrated",
        enzyme="an enzyme over-expressed at the target site, e.g. thymidine phosphorylase in many tumours",
        kind="carrier-linked, multi-step activation",
        caution="selectivity is only as good as the difference in enzyme level; leak elsewhere still causes toxicity",
        example="capecitabine → 5-fluorouracil"),
    "A drug is cleared too fast and needs a longer duration": dict(
        group="a group on the drug that an enzyme can remove slowly",
        strategy="attach a slowly hydrolysed promoiety so active drug is released gradually from a circulating reservoir",
        enzyme="plasma cholinesterase (butyrylcholinesterase) or a slow esterase",
        kind="carrier-linked, bipartite (slow release)",
        caution="too slow means too little active drug; the release rate also varies between patients, widening exposure",
        example="bambuterol → terbutaline"),
    "An oral liquid for children tastes unacceptably bitter": dict(
        group="a hydroxyl on the drug",
        strategy="esterify to a very insoluble derivative so little dissolves on the tongue, then dissolves and is cleaved after swallowing",
        enzyme="intestinal and hepatic esterases",
        kind="carrier-linked, bipartite (taste masking)",
        caution="an insoluble ester must still dissolve and activate in the gut, or absorption is lost",
        example="erythromycin ethylsuccinate / estolate suspensions"),
}


def prodrug_designer(problem="(choose a problem)"):
    probs = list(_PRODRUG_DESIGNS)
    if problem not in _PRODRUG_DESIGNS:
        _md(["**Pick a design problem, then sketch your own answer before reading the suggestion.**", "",
             "Work in this order for any prodrug: (1) name the property problem and the group that causes it; "
             "(2) choose a promoiety that fixes the property and attaches to a group the drug can spare; "
             "(3) name the enzyme (or trigger) that will unmask it, and where that enzyme is; "
             "(4) check that what falls off is harmless and that activation is reliable across patients.", "",
             "Problems to choose from:"] + ["* %s" % p for p in probs])
        return {}
    d = _PRODRUG_DESIGNS[problem]
    _md(["**Problem.** %s" % problem, "",
         "| Design step | Choice |", "|---|---|",
         "| 1. Group to modify | %s |" % d["group"],
         "| 2. Promoiety strategy | %s |" % d["strategy"],
         "| 3. Prodrug class | %s |" % d["kind"],
         "| 4. Activating agent (and where) | **%s** |" % d["enzyme"],
         "| 5. Main caution | %s |" % d["caution"],
         "", "Marketed design of this kind: *%s*." % d["example"],
         "", "This is one sound answer, not the only one. Compare it with yours: a different promoiety is fine if it "
         "fixes the same property, attaches to a group the drug can spare, and is unmasked reliably by a named enzyme."])
    return d

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110})
_start_report('PHS 312', 'L32', 'Prodrugs')
EXTRA_ITEMS = _json.loads('[{"qid": "L32-X1", "stem": "A potent drug carries a phosphate group it needs for binding, but the charge keeps it from being absorbed when swallowed. You must design an orally absorbed prodrug and name what switches it on. Which plan is soundest?", "options": {"A": "Mask the phosphate as a carbonate–acetal ester arm, unmasked by esterases then self-immolation", "B": "Mask the phosphate as a permanent methyl phosphonate, which resists charge at every pH", "C": "Attach a second copy of the drug through the phosphate, cleaved by a plasma phosphatase", "D": "Leave the phosphate and add a fixed quaternary ammonium to raise aqueous solubility"}, "key": "A", "why": {"B": "A methyl phosphonate is not cleaved back to the active phosphate, so it is a new molecule, not a prodrug that regenerates the drug.", "C": "Joining two drug copies gives a mutual prodrug only when both halves are useful drugs; here it just doubles the charged load and solves nothing.", "D": "A permanent quaternary group adds a second fixed charge, which worsens rather than fixes the absorption problem.", "_correct": "A disoproxil-type carbonate–acetal masks the phosphate charge so the neutral prodrug is absorbed; gut and liver esterases cleave the carbonate and the exposed acetal then collapses on its own, regenerating the active phosphate."}, "bloom": "Create", "silo": "5", "section": "concept", "topic": "L32"}, {"qid": "L32-X2", "stem": "An antiviral is active only after a cell adds three phosphates to it, and you want it unmasked mainly inside virus-infected cells. You design a prodrug and must name the first activating step. Which design and enzyme fit the goal?", "options": {"A": "An amino-acid ester cleaved by gut esterase, then a first phosphate added by viral thymidine kinase", "B": "A methyl ester cleaved by plasma esterase, then all three phosphates added by host kinases alone", "C": "A permanent fluorine for hydrogen, with the three phosphates added by any cellular kinase", "D": "A carbonate linker cleaved by phosphatase, which removes the need for any kinase step"}, "key": "A", "why": {"B": "Relying on host kinases for the first phosphate gives no selectivity for infected cells, which is the whole point of this design.", "C": "A fluorine swap is a bioisostere, not a prodrug; it does not improve uptake and still leaves the first phosphate to a non-selective enzyme.", "D": "A phosphatase removes phosphate rather than adding it, so it cannot prepare a drug that must gain three phosphates.", "_correct": "An amino-acid ester improves oral uptake and is cleaved by an intestinal/hepatic ester hydrolase; the viral thymidine kinase then adds the first phosphate far faster in infected cells, so the triphosphate forms mainly where the virus is."}, "bloom": "Create", "silo": "5", "section": "concept", "topic": "L32"}]')
gb.add_many([Item(**d) for d in EXTRA_ITEMS if d['qid'] not in gb.items])
_log_answers(gb)
# ===== END kit =====


## Learning objectives

| # | By the end of this activity you should be able to | Bloom | Syllabus level | SILO |
|---|---|---|---|---|
| 1 | Predict how varying the promoiety changes the rate of hydrolytic activation. | Apply | IV | 4 |
| 2 | Distinguish carrier-linked prodrugs, bipartite designs, mutual prodrugs and bioprecursors. | Understand | V | 5 |
| 3 | Evaluate a proposed prodrug for activation reliability and patient variability. | Evaluate | II | 4 |
| 4 | Design a prodrug for a stated problem and name the activating enzyme. | Create | I | 5 |

*Syllabus levels follow the PHS 312 legend, in which Level I is Creating and Level VI is Remembering.*

## Part 1 - What do you already know?

Three questions before you begin. **You will not be told whether you are
right**, and that is deliberate: the point is to measure what you knew coming
in, so that at the end you can see how much the activity moved you. Answer
from what you remember from the reading. Guessing is fine.

In [ ]:
#@title P1   (run after choosing) { display-mode: "form" }
#@markdown **P1.**  In a carrier-linked prodrug, what does the promoiety do?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It protects the drug from the immune system during transport", "B. It binds the target directly once the drug has been absorbed", "C. It changes the physical properties and is later cleaved off", "D. It provides the pharmacological activity until the drug acts"]
gb.answer("P1", response[:1])

In [ ]:
#@title P2   (run after choosing) { display-mode: "form" }
#@markdown **P2.**  An ester prodrug is activated faster when the alcohol carries an electron-withdrawing aryl group. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The compound becomes more water soluble and so hydrolyses", "B. The ester bond becomes longer and therefore weaker overall", "C. The aryl group attracts esterase enzymes to the molecule", "D. The carbonyl becomes more electrophilic and easier to attack"]
gb.answer("P2", response[:1])

In [ ]:
#@title P3   (run after choosing) { display-mode: "form" }
#@markdown **P3.**  A prodrug behaves consistently in plasma but gives variable exposure across patients. What does that suggest?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Activation depends on an enzyme whose activity differs between people", "B. The prodrug is chemically unstable and decomposes before it is taken", "C. The assay used to measure plasma concentrations is poorly calibrated", "D. The drug itself has a narrow therapeutic window in these patients"]
gb.answer("P3", response[:1])

## Part 2 - Key ideas: what a prodrug is and why you would make one

*Objectives addressed: L32.2*


**Summary.** A **prodrug** is a compound with little or no activity of its own that the body converts into the active drug. The pharmacophore is kept intact; a removable group (the **promoiety** or **carrier**) is attached to fix a problem that has nothing to do with binding — the drug is too polar to be absorbed, too insoluble to inject, too short-lived, too bitter, too unstable, or active in the wrong tissue. An enzyme (sometimes just pH or light) then removes or changes the masking group where and when the drug is needed.

The design only works if three things are true: (1) the prodrug survives long enough to do its job (cross the gut wall, dissolve, reach the tissue); (2) activation is **fast and reliable** once it gets there; (3) whatever falls off is **harmless**.

| Problem with the drug | What the prodrug changes | Example (activation) |
|---|---|---|
| Poor oral absorption (charged acid, phosphate or very polar group) | masks the charge with an ester; raises log D | enalapril, oseltamivir (liver carboxylesterase); tenofovir disoproxil |
| Cannot reach the target tissue (e.g. blood–brain barrier) | borrows a nutrient transporter (*Trojan horse*) | levodopa via LAT1; valacyclovir via PEPT1 |
| Too short a duration of action | slow-release reservoir or slowly cleaved group | bambuterol → terbutaline (slow plasma cholinesterase hydrolysis) |
| Side effects or toxicity at the wrong site | drug is released mainly where an activating enzyme is concentrated | capecitabine → 5-FU (tumour thymidine phosphorylase); acyclovir (viral kinase) |
| Bad taste in an oral liquid | lowers water solubility so less dissolves on the tongue | erythromycin esters in paediatric suspensions |
| Too insoluble for an injection | adds an ionizable phosphate or amine | fosphenytoin; irinotecan |
| Chemically unstable in the bottle | locks a reactive group into a ring that reopens in the body | the reading's §14.6.7 example |
| Needs switching on at one site | external trigger (light) — the reading's "sleeping agents" | 5-aminolevulinic acid → protoporphyrin IX, a photosensitizer for photodynamic therapy |

In [ ]:
#@title 2.1 Figure — four ways to build a prodrug { display-mode: "form" }
_cell('2.1 Figure — four ways to build a prodrug')
fig_prodrug_types()

| Class | What is attached | How the drug is released | What to look for in a structure |
|---|---|---|---|
| **Carrier-linked, bipartite** | carrier bonded straight to the drug | one hydrolysis (esterase, phosphatase, peptidase) | an ester, amide, carbonate or phosphate on a group the drug needs for binding |
| **Carrier-linked, tripartite** | carrier → **linker (spacer)** → drug | enzyme removes the carrier; the linker then breaks off **spontaneously** (self-immolative) | –O–CH₂–O– or –N–CH₂–O– acetal-type links; often release formaldehyde or acetaldehyde |
| **Mutual prodrug** | a second drug acts as the carrier | one cleavage releases two active compounds | two recognizable drugs joined by an ester or azo bond |
| **Bioprecursor** | nothing — the molecule itself is changed | oxidation, reduction, decarboxylation, phosphorylation or an acid-catalysed rearrangement | the active species is a *metabolite*; no carrier is shed |

Why bother with a linker? Some bonds are hard for enzymes to reach (a crowded or weakly nucleophilic atom, such as the imide N of phenytoin) or would release a toxic carrier. A spacer moves the enzyme's cut away from the drug, and the left-over fragment is designed to be unstable, so it falls apart as soon as the carrier is gone.

<details>
<summary><b>▸ Key terms (open)</b></summary>


| Term | Meaning |
|---|---|
| Prodrug | Inactive (or much less active) compound converted in the body to the active drug |
| Promoiety / carrier | The removable group attached to the drug |
| Bipartite prodrug | Drug and carrier joined directly |
| Tripartite prodrug | Drug – linker (spacer) – carrier |
| Self-immolative linker | A spacer that decomposes spontaneously once the carrier has been cleaved |
| Mutual prodrug | Two pharmacologically active agents joined so that each carries the other |
| Bioprecursor | A prodrug with no carrier; it becomes active by a metabolic change to its own structure |
| Trojan-horse strategy | Making the drug resemble a nutrient so a membrane transporter carries it in |
| Activation | The chemical or enzymatic step that releases or creates the active drug |
| Carboxylesterase (CES1, CES2) | Serine hydrolases that cut esters, amides and carbamates; CES1 mainly liver, CES2 mainly intestine |
| Sleeping agent | The reading's term for a prodrug switched on by an external influence such as light |
| Soft drug | The reverse idea: an *active* drug designed to be inactivated quickly by metabolism |

</details>

#### Worked example: classifying four prodrugs
Classify each as bipartite, tripartite, mutual or bioprecursor, and name what is released besides the drug: (a) oseltamivir; (b) tenofovir disoproxil (each phosphonate OH is masked as –O–CH₂–O–C(=O)–O–iPr); (c) sultamicillin (ampicillin and sulbactam joined through a –O–CH₂–O– diester); (d) clopidogrel.

<details>
<summary><b>▸ Worked solution</b></summary>

1. **Oseltamivir — bipartite.** An ethyl ester sits directly on the carboxylate; CES1 releases **ethanol**.
2. **Tenofovir disoproxil — tripartite.** Esterases cut the carbonate first; the exposed –O–CH₂–O–P hemiacetal then collapses on its own. Each arm releases **isopropanol, CO₂ and formaldehyde**.
3. **Sultamicillin — mutual prodrug** (with a methylene linker). Hydrolysis releases **ampicillin** (antibiotic) and **sulbactam** (β-lactamase inhibitor) plus formaldehyde; each drug is the other's carrier.
4. **Clopidogrel — bioprecursor.** Nothing is cut off as a carrier; CYP450 oxidation (mainly CYP2C19) converts the thiophene ring, in two steps, into the active thiol.

Rule of thumb: if you can draw the drug and a separate fragment that leaves, it is carrier-linked; if the active species is a *metabolite* of the whole molecule, it is a bioprecursor.
</details>

**Practice: what does the promoiety release?** A carrier is only acceptable if what falls off is harmless at the amount released. The card below does the stoichiometry for tenofovir disoproxil (part B of the classification above).

**1. By-products released by a tripartite prodrug (practice: tenofovir disoproxil)**

$$n = \frac{{\color{#1d6fb8}{m_{pro}}}}{{\color{#c2410c}{M_{pro}}}};\quad {\color{#7c3aed}{m_{HCHO}}} = {\color{#4b5563}{n_{arms}}} \times n \times {\color{#4b5563}{M_{HCHO}}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{m_{pro}}}$ | tenofovir disoproxil (free base) hydrolysed | input | g | in-vitro study (case) |
| ${\color{#c2410c}{M_{pro}}}$ | molar mass of tenofovir disoproxil, C₁₉H₃₀N₅O₁₀P | input | g/mol | formula, IUPAC atomic weights |
| ${\color{#15803d}{M_{drug}}}$ | molar mass of tenofovir, C₉H₁₄N₅O₄P | input | g/mol | formula, IUPAC atomic weights |
| ${\color{#7c3aed}{m_{HCHO}}}$ | formaldehyde released | result | mg | calculated |
| ${\color{#4b5563}{n_{arms}}}$ | promoiety arms per molecule | constant | — | value = 2 ; structure (two phosphonate esters) |
| ${\color{#4b5563}{M_{HCHO}}}$ | molar mass of formaldehyde, CH₂O | constant | g/mol | value = 30.03 g/mol; IUPAC atomic weights |
| ${\color{#4b5563}{M_{iPrOH}}}$ | molar mass of isopropanol, C₃H₈O | constant | g/mol | value = 60.1 g/mol; IUPAC atomic weights |
| ${\color{#4b5563}{M_{CO_2}}}$ | molar mass of CO₂ | constant | g/mol | value = 44.01 g/mol; IUPAC atomic weights |
| ${\color{#4b5563}{M_{H_2O}}}$ | molar mass of water (consumed, one per arm) | constant | g/mol | value = 18.02 g/mol; IUPAC atomic weights |

*Source of the relationship:* activation chemistry of acyloxymethyl/carbonate-acetal prodrugs (L32 notes)

**2. Practice problem (case)**

A hospital drug-information pharmacist is asked how much formaldehyde the 'disoproxil' promoiety releases. In an in-vitro esterase study, $\mathbf{{\color{#1d6fb8}{1\,\mathrm{g}}}}$ of tenofovir disoproxil (molar mass $\mathbf{{\color{#c2410c}{519.4\,\mathrm{g/mol}}}}$) is hydrolysed completely. Each of its $\mathbf{{\color{#4b5563}{2}}}$ –O–CH₂–O–C(=O)–O–iPr arms releases one formaldehyde, one CO₂ and one isopropanol, leaving tenofovir ($\mathbf{{\color{#15803d}{287.2\,\mathrm{g/mol}}}}$). How many mg of each product form?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | $n = \frac{{\color{#1d6fb8}{m_{pro}}}}{{\color{#c2410c}{M_{pro}}}};\quad {\color{#7c3aed}{m_{HCHO}}} = {\color{#4b5563}{n_{arms}}} \times n \times {\color{#4b5563}{M_{HCHO}}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{m_{pro}}} = 1\,\mathrm{g}$; ${\color{#c2410c}{M_{pro}}} = 519.4\,\mathrm{g/mol}$; ${\color{#15803d}{M_{drug}}} = 287.2\,\mathrm{g/mol}$; ${\color{#4b5563}{n_{arms}}} = 2$ (constant); ${\color{#4b5563}{M_{HCHO}}} = 30.03\,\mathrm{g/mol}$ (constant); ${\color{#4b5563}{M_{iPrOH}}} = 60.1\,\mathrm{g/mol}$ (constant); ${\color{#4b5563}{M_{CO_2}}} = 44.01\,\mathrm{g/mol}$ (constant); ${\color{#4b5563}{M_{H_2O}}} = 18.02\,\mathrm{g/mol}$ (constant) |
| 3. Unit check | No unit prefixes to expand. Write the sample in milligrams: 1 g = 10³ mg. Then mg ÷ (g/mol) = 10⁻³ g ÷ (g/mol) = 10⁻³ mol = mmol, and mmol × g/mol = mg. Mol per mol stoichiometry factors carry the milli prefix through unchanged. |
| 4. Sample in mg | ${\color{#1d6fb8}{m_{pro}}} = 1\,\mathrm{g} \times 10^{3}\,\mathrm{mg/g} = 1000\,\mathrm{mg}$ |
| 5. Moles of prodrug | $n = \frac{{\color{#1d6fb8}{m_{pro}}}}{{\color{#c2410c}{M_{pro}}}} = \frac{1000\,\mathrm{mg}}{519.4\,\mathrm{g/mol}} = 1.925\,\mathrm{mmol}$ |
| 6. Moles from the two arms | $n_{arm} = {\color{#4b5563}{n_{arms}}} \times n = 2 \times 1.925 = 3.85\,\mathrm{mmol}$ |
| 7. Tenofovir released | $n \times {\color{#15803d}{M_{drug}}} = 1.925\,\mathrm{mmol} \times 287.2\,\mathrm{g/mol} = 552.9\,\mathrm{mg}$ |
| 8. Formaldehyde released | ${\color{#7c3aed}{m_{HCHO}}} = n_{arm} \times {\color{#4b5563}{M_{HCHO}}} = 3.85 \times 30.03 = 115.6\,\mathrm{mg}$ |
| 9. Isopropanol released | $n_{arm} \times {\color{#4b5563}{M_{iPrOH}}} = 3.85 \times 60.1 = 231.4\,\mathrm{mg}$ |
| 10. CO₂ released | $n_{arm} \times {\color{#4b5563}{M_{CO_2}}} = 3.85 \times 44.01 = 169.4\,\mathrm{mg}$ |
| 11. Mass balance check (water is consumed) | $552.9 + 115.6 + 231.4 + 169.4 = 1069\,\mathrm{mg} = 1000 + 69.36\,\mathrm{mg\ H_2O}$ |
| 12. Answer | $\boxed{m_{HCHO} = 115.6\,\mathrm{mg}}$ |
| 13. What it means | Complete hydrolysis of 1 g of tenofovir disoproxil gives 552.9 mg of tenofovir (55.29% of the mass) plus 115.6 mg formaldehyde, 231.4 mg isopropanol and 169.4 mg CO₂ from the two promoiety arms. The promoiety must be judged at the amount released, which is why formaldehyde-releasing linkers are scrutinized. |

**Common conversions for this calculation:** 1 g = 1000 mg; mg ÷ (g/mol) = mmol; 1 mmol = 10⁻³ mol = 1000 µmol
</details>

In [ ]:
#@title 2.2 Solve it yourself: By-products released by a tripartite prodrug (practice: tenofovir disoproxil) { display-mode: "form" }
_cell('2.2 Solve it yourself: By-products released by a tripartite prodrug (practice: tenofovir disoproxil)')
card('l32_byproducts')

## Part 3 - Masking a charge: ester and amino-acid prodrugs on one template

*Objectives addressed: L32.1 · L32.4*


The commonest prodrug problem is a **carboxylic acid** (or phosphate) that the target needs for an ionic or metal contact but that is almost fully ionized in the small intestine, so the drug crosses membranes poorly. Converting the acid to an **ester** removes the charge; esterases put it back after absorption. Three design rules follow from the chemistry of ester hydrolysis:

* **Electron-withdrawing groups on the alcohol speed hydrolysis** (they make the carbonyl carbon more electrophilic and the alkoxide a better leaving group): aryl esters such as 5-indanyl are cleaved faster than ethyl esters.
* **Steric bulk slows hydrolysis**: a pivaloyl or tert-butyl group shields the carbonyl from the esterase serine.
* **The leaving alcohol must be safe** at the amount released: ethanol, valine and 5-indanol are; methanol is avoided.

Below, each prodrug sits directly above its active drug on the same template. Only the blue promoiety changes; the scissors mark the bond the enzyme breaks.

In [ ]:
#@title 3.1 Figure — enalapril above enalaprilat { display-mode: "form" }
_cell('3.1 Figure — enalapril above enalaprilat')
fig_enalapril()

In [ ]:
#@title 3.2 Figure — oseltamivir above oseltamivir carboxylate { display-mode: "form" }
_cell('3.2 Figure — oseltamivir above oseltamivir carboxylate')
fig_oseltamivir()

In [ ]:
#@title 3.3 Figure — valacyclovir above acyclovir { display-mode: "form" }
_cell('3.3 Figure — valacyclovir above acyclovir')
fig_valacyclovir()

**1. log D of a carboxylic-acid drug and its ethyl ester prodrug (worked example: how much does an ester raise log D?)**

$${\color{#b91c1c}{\log D_{acid}}} = {\color{#1d6fb8}{\log P_{acid}}} - \log_{10}\left(1 + 10^{{\color{#7c3aed}{pH}} - {\color{#c2410c}{pK_a}}}\right);\quad {\color{#0e7490}{\log D_{ester}}} = {\color{#15803d}{\log P_{ester}}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{\log P_{acid}}}$ | log P of the neutral acid | input | — | case (illustrative) |
| ${\color{#c2410c}{pK_a}}$ | pKa of the acid | input | — | case (illustrative) |
| ${\color{#15803d}{\log P_{ester}}}$ | log P of the ethyl ester (no ionizable group) | input | — | case (illustrative) |
| ${\color{#7c3aed}{pH}}$ | pH of the upper small intestine | input | — | physiological range 6–7 |
| ${\color{#b91c1c}{\log D_{acid}}}$ | log D of the acid | intermediate | — | calculated |
| ${\color{#0e7490}{\log D_{ester}}}$ | log D of the ester | intermediate | — | calculated |

*Source of the relationship:* log D of a monoprotic acid (L32 notes, masking a charge)

**2. Practice problem (case)**

A carboxylic-acid drug has log P $\mathbf{{\color{#1d6fb8}{1}}}$ and pKa $\mathbf{{\color{#c2410c}{4}}}$. Its ethyl ester prodrug has log P $\mathbf{{\color{#15803d}{2.3}}}$ and no ionizable group. Calculate log D for each at pH $\mathbf{{\color{#7c3aed}{6.5}}}$ (upper small intestine) and the fold change in octanol/water distribution.

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{\log D_{acid}}} = {\color{#1d6fb8}{\log P_{acid}}} - \log_{10}\left(1 + 10^{{\color{#7c3aed}{pH}} - {\color{#c2410c}{pK_a}}}\right);\quad {\color{#0e7490}{\log D_{ester}}} = {\color{#15803d}{\log P_{ester}}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{\log P_{acid}}} = 1$; ${\color{#c2410c}{pK_a}} = 4$; ${\color{#15803d}{\log P_{ester}}} = 2.3$; ${\color{#7c3aed}{pH}} = 6.5$ |
| 3. Unit check | No unit prefixes to expand. log P, log D, pH and pKa are logarithms (no units); the fold change is a ratio of two octanol/water distribution coefficients, also unit-free. |
| 4. Acid: exponent | $pH - {\color{#c2410c}{pK_a}} = 6.5 - 4 = 2.5$ |
| 5. Acid: ionized/neutral ratio | $\frac{[A^{-}]}{[HA]} = 10^{2.5} = 316.2$ |
| 6. Acid: percent ionized | $\frac{100 \times 316.2}{1 + 316.2} = 99.68\%$ |
| 7. Acid: log term | $\log_{10}(1 + 316.2) = \log_{10}(317.2) = 2.501$ |
| 8. Acid: log D | ${\color{#b91c1c}{\log D_{acid}}} = {\color{#1d6fb8}{\log P_{acid}}} - 2.501 = 1 - 2.501 = -1.501$ |
| 9. Ester: no ionizable group | ${\color{#0e7490}{\log D_{ester}}} = {\color{#15803d}{\log P_{ester}}} = 2.3$ |
| 10. Difference | ${\color{#0e7490}{\log D_{ester}}} - {\color{#b91c1c}{\log D_{acid}}} = 2.3 - \left(-1.501\right) = 3.801$ |
| 11. Fold change | $10^{3.801} = 6330$ |
| 12. Split of the gain | $\text{extra carbons: } 2.3 - 1 = 1.3;\ \text{removing the charge: } 2.501$ |
| 13. Answer | $\boxed{\Delta\log D = 3.801}$ |
| 14. What it means | At pH 6.5 the acid is 99.68% ionized (log D -1.501); the ester stays neutral (log D 2.3). The ester partitions 3.801 log units, about 6330-fold, more into octanol; 2.501 of that comes from removing the charge, only 1.3 from the extra carbons. |

**Common conversions for this calculation:** D = 10^(log D); % neutral = 100 − % ionized
</details>

In [ ]:
#@title 3.4 Solve it yourself: log D of a carboxylic-acid drug and its ethyl ester prodrug (worked example: how much does an ester raise log D?) { display-mode: "form" }
_cell('3.4 Solve it yourself: log D of a carboxylic-acid drug and its ethyl ester prodrug (worked example: how much does an ester raise log D?)')
card('l32_logd_ester')

In [ ]:
#@title 3.5 Calculator — log D of an acid and its ethyl ester { display-mode: "form" }
_cell('3.5 Calculator — log D of an acid and its ethyl ester')
logP_acid = 1.0 #@param {type:"number"}
pKa_acid = 4.0 #@param {type:"number"}
logP_ester = 2.3 #@param {type:"number"}
pH = 6.5 #@param {type:"slider", min:1.0, max:8.0, step:0.1}
_ = logD_ester_calc(logP_acid, pKa_acid, logP_ester, pH)

**Try it.** Set pH to 1.5 (stomach): the acid is now mostly un-ionized and the ester's advantage shrinks to its log P difference. The prodrug matters most where the acid is charged — the intestine and blood.

**Practice: how much of the prodrug is drug?** An ester (and any salt form) adds mass that is not the active drug. Laboratories preparing standards, and anyone comparing amounts of a prodrug with amounts of its active drug, convert with the molar-mass ratio.

**1. Salt and ester mass factor: prodrug salt vs active drug (practice: oseltamivir)**

$${\color{#b91c1c}{f}} = \frac{{\color{#1d6fb8}{M_{salt}}}}{{\color{#15803d}{M_{acid}}}} = \frac{{\color{#c2410c}{M_{ester}}}}{{\color{#15803d}{M_{acid}}}} \times \frac{{\color{#1d6fb8}{M_{salt}}}}{{\color{#c2410c}{M_{ester}}}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{M_{salt}}}$ | oseltamivir phosphate, C₁₆H₂₈N₂O₄·H₃PO₄ | input | g/mol | formula, IUPAC atomic weights |
| ${\color{#c2410c}{M_{ester}}}$ | oseltamivir (ethyl ester base), C₁₆H₂₈N₂O₄ | input | g/mol | formula, IUPAC atomic weights |
| ${\color{#15803d}{M_{acid}}}$ | oseltamivir carboxylate (active), C₁₄H₂₄N₂O₄ | input | g/mol | formula, IUPAC atomic weights |
| ${\color{#7c3aed}{m_{eq}}}$ | carboxylate-equivalent mass needed in the standard | input | mg | case |
| ${\color{#b91c1c}{f}}$ | mass of salt per mass of active carboxylate | result | — | calculated |

*Source of the relationship:* molar-mass bookkeeping for ester prodrugs and salts (L32 notes)

**2. Practice problem (case)**

A pharmacokinetics laboratory needs a calibration stock containing $\mathbf{{\color{#7c3aed}{10\,\mathrm{mg}}}}$ of oseltamivir carboxylate equivalents but has only the prodrug salt, oseltamivir phosphate ($\mathbf{{\color{#1d6fb8}{410.4\,\mathrm{g/mol}}}}$). The free ester base is $\mathbf{{\color{#c2410c}{312.4\,\mathrm{g/mol}}}}$ and the active carboxylate $\mathbf{{\color{#15803d}{284.4\,\mathrm{g/mol}}}}$. Calculate the ester factor, the salt factor, the overall factor and the mass of salt to weigh.

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{f}} = \frac{{\color{#1d6fb8}{M_{salt}}}}{{\color{#15803d}{M_{acid}}}} = \frac{{\color{#c2410c}{M_{ester}}}}{{\color{#15803d}{M_{acid}}}} \times \frac{{\color{#1d6fb8}{M_{salt}}}}{{\color{#c2410c}{M_{ester}}}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{M_{salt}}} = 410.4\,\mathrm{g/mol}$; ${\color{#c2410c}{M_{ester}}} = 312.4\,\mathrm{g/mol}$; ${\color{#15803d}{M_{acid}}} = 284.4\,\mathrm{g/mol}$; ${\color{#7c3aed}{m_{eq}}} = 10\,\mathrm{mg}$ |
| 3. Unit check | Prefixes written out: $10\,\mathrm{mg} = 10 \times 10^{-3}\,\mathrm{g} = 0.01\,\mathrm{g}$. g/mol divided by g/mol cancels, so each factor is unit-free; the mass to weigh keeps the unit of the target (mg). The milli prefix carries through and is never converted. |
| 4. Ester factor | $\frac{{\color{#c2410c}{M_{ester}}}}{{\color{#15803d}{M_{acid}}}} = \frac{312.4}{284.4} = 1.099$ |
| 5. Salt factor | $\frac{{\color{#1d6fb8}{M_{salt}}}}{{\color{#c2410c}{M_{ester}}}} = \frac{410.4}{312.4} = 1.314$ |
| 6. Overall factor | ${\color{#b91c1c}{f}} = 1.099 \times 1.314 = 1.443$ |
| 7. Check directly | $\frac{{\color{#1d6fb8}{M_{salt}}}}{{\color{#15803d}{M_{acid}}}} = \frac{410.4}{284.4} = 1.443$ |
| 8. Fraction of the salt that is active drug | $\frac{1}{1.443} = 0.6929 = 69.29\%$ |
| 9. Mass of salt to weigh | ${\color{#7c3aed}{m_{eq}}} \times {\color{#b91c1c}{f}} = 10\,\mathrm{mg} \times 1.443 = 14.43\,\mathrm{mg}$ |
| 10. Answer | $\boxed{f = 1.443}$ |
| 11. What it means | Every mg of oseltamivir carboxylate equivalent needs 1.443 mg of oseltamivir phosphate: the ethyl ester adds 9.868% and the phosphate salt another 31.37%. Only 69.29% of the salt's mass is the active carboxylate, so the standard needs 14.43 mg of salt. |

**Common conversions for this calculation:** 1 mg = 1000 µg; mmol = mg ÷ (g/mol); salt factor = M(salt)/M(base)
</details>

In [ ]:
#@title 3.6 Solve it yourself: Salt and ester mass factor: prodrug salt vs active drug (practice: oseltamivir) { display-mode: "form" }
_cell('3.6 Solve it yourself: Salt and ester mass factor: prodrug salt vs active drug (practice: oseltamivir)')
card('l32_mw_ratio')

## Part 4 - Bioprecursors, self-immolative linkers and how fast activation happens

*Objectives addressed: L32.1 · L32.2 · L32.3*


Not every prodrug sheds a carrier. A **bioprecursor** becomes active because metabolism rewrites part of the molecule. Levodopa is the classic case: removing CO₂ creates dopamine, and the carboxyl group that is lost is exactly what lets an amino-acid transporter carry the molecule into the brain.

In [ ]:
#@title 4.1 Figure — levodopa above dopamine, and carbidopa { display-mode: "form" }
_cell('4.1 Figure — levodopa above dopamine, and carbidopa')
fig_levodopa()

A **tripartite** prodrug puts a spacer between drug and carrier. Fosphenytoin is the cleanest example: the phosphate (carrier) gives water solubility for an intravenous solution, the CH₂ is the self-immolative linker, and phenytoin is the drug.

In [ ]:
#@title 4.2 Figure — fosphenytoin to phenytoin in two steps { display-mode: "form" }
_cell('4.2 Figure — fosphenytoin to phenytoin in two steps')
fig_fosphenytoin()

In [ ]:
#@title 4.3 Figure — the route of an oral ester prodrug from gut lumen to target (enalapril) { display-mode: "form" }
_cell('4.3 Figure — the route of an oral ester prodrug from gut lumen to target (enalapril)')
fig_ester_route()

**Where activation happens decides what the prodrug can do.**

| Activating agent | Main location | What it does | Examples |
|---|---|---|---|
| Carboxylesterase 1 (CES1) | liver | hydrolyses esters with a large acyl part and small alcohol | enalapril, oseltamivir, capecitabine (first step) |
| Carboxylesterase 2 (CES2) | small intestine (also liver) | prefers a small acyl part and large alcohol | irinotecan → SN-38 |
| Plasma cholinesterase and other blood esterases | blood | slow hydrolysis of carbamates and some esters | bambuterol → terbutaline |
| Alkaline phosphatase | intestinal brush border, liver, plasma | removes phosphate monoesters | fosphenytoin, fosamprenavir |
| Amino-acid ester hydrolase (valacyclovirase) | intestine, liver | cuts amino-acid esters | valacyclovir, valganciclovir |
| Cytochrome P450 | liver, gut wall | oxidation, O- and N-dealkylation | clopidogrel (CYP2C19), codeine → morphine (CYP2D6) |
| Bacterial azoreductase | colon | reduces N=N | balsalazide, olsalazine → 5-aminosalicylic acid |
| Aromatic L-amino-acid decarboxylase | brain and periphery | removes CO₂ | levodopa → dopamine |
| Kinases (viral thymidine kinase first) | infected cells | add phosphate | acyclovir → acyclovir triphosphate |
| Acid (no enzyme) | acidic compartments | rearrangement | omeprazole → sulfenamide in the parietal-cell canaliculus |

**What can go wrong.**
* **Variable activation.** If the activating enzyme is polymorphic or easily inhibited, exposure varies between patients: CYP2C19 poor metabolizers make less active clopidogrel metabolite; CYP2D6 ultrarapid metabolizers make morphine from codeine faster than expected; some CES1 variants hydrolyse ester prodrugs more slowly.
* **Species differences.** Rodent plasma is rich in carboxylesterase and human plasma is not, so an ester that is cleaved instantly in rat blood may survive much longer in a patient.
* **Activation in the wrong place.** Peripheral AADC wastes levodopa (hence carbidopa); an ester hydrolysed in the gut lumen loses its absorption advantage.
* **Toxic or burdensome by-products.** Pivoxil (pivaloyloxymethyl) esters release pivalic acid, which is excreted bound to carnitine and can deplete carnitine during long courses; acyloxymethyl and N-CH₂ linkers release small amounts of formaldehyde. The promoiety must be judged at the amount actually released.
* **Incomplete activation.** Some prodrug may be excreted unchanged, so the dose of prodrug is not the dose of drug.

#### Worked example: how reliably does an ester switch on?
An ester prodrug is activated by first-order hydrolysis with t½ = 1.5 h in a typical patient. Patient L has 0.3 × typical esterase activity and patient H has 3 ×. What percentage of the dose is active drug in each at 2 h, and what is the between-patient spread? Repeat for a much faster promoiety with t½ = 0.25 h.

<details>
<summary><b>▸ Worked solution</b></summary>

1. Typical k = ln 2 / t½ = 0.693 / 1.5 = **0.462 h⁻¹**.
2. Patient L: k = 0.3 × 0.462 = 0.139 h⁻¹; % at 2 h = 100(1 − e^(−0.277)) = **24.2%**.
3. Patient H: k = 3 × 0.462 = 1.386 h⁻¹; % at 2 h = 100(1 − e^(−2.773)) = **93.8%**.
4. Spread = 93.8 − 24.2 = **69.5 percentage points** (typical patient 60.3%).
5. Faster promoiety, t½ = 0.25 h: typical k = 2.77 h⁻¹; L: k = 0.832 h⁻¹ → **81.1%**; H: k = 8.32 h⁻¹ → **≈100.0%**; spread ≈ **19 points**.

The same tenfold range of enzyme activity produces a much smaller difference in drug delivered when activation is fast relative to the time point of interest. Reproduce both cases with the card below (re-solve with t½ = 0.25) or with the calculator after it.
</details>

**1. First-order prodrug activation in two outlier patients (worked example: how reliably does an ester switch on?)**

$${\color{#b91c1c}{k}} = \frac{\ln 2}{{\color{#1d6fb8}{t_{1/2}}}};\quad \%_{conv} = 100\left(1 - e^{-k{\color{#c2410c}{t}}}\right);\quad k_{L} = {\color{#15803d}{f_{L}}}{\color{#b91c1c}{k}},\ k_{H} = {\color{#7c3aed}{f_{H}}}{\color{#b91c1c}{k}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{t_{1/2}}}$ | activation half-life in a typical patient | input | h | case (illustrative) |
| ${\color{#c2410c}{t}}$ | time point of interest | input | h | case |
| ${\color{#15803d}{f_{L}}}$ | esterase activity of patient L relative to typical | input | — | case (illustrative) |
| ${\color{#7c3aed}{f_{H}}}$ | esterase activity of patient H relative to typical | input | — | case (illustrative) |
| ${\color{#b91c1c}{k}}$ | typical first-order activation rate constant | intermediate | h^-1 | calculated |

*Source of the relationship:* first-order kinetics of enzymatic activation (L32 notes)

**2. Practice problem (case)**

An ester prodrug is activated by first-order hydrolysis with t½ = $\mathbf{{\color{#1d6fb8}{1.5\,\mathrm{h}}}}$ in a typical patient. Patient L has $\mathbf{{\color{#15803d}{0.3}}}$ × the typical esterase activity and patient H $\mathbf{{\color{#7c3aed}{3}}}$ ×. What percentage has been converted to active drug in each at t = $\mathbf{{\color{#c2410c}{2\,\mathrm{h}}}}$, and what is the between-patient spread?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#b91c1c}{k}} = \frac{\ln 2}{{\color{#1d6fb8}{t_{1/2}}}};\quad \%_{conv} = 100\left(1 - e^{-k{\color{#c2410c}{t}}}\right);\quad k_{L} = {\color{#15803d}{f_{L}}}{\color{#b91c1c}{k}},\ k_{H} = {\color{#7c3aed}{f_{H}}}{\color{#b91c1c}{k}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{t_{1/2}}} = 1.5\,\mathrm{h}$; ${\color{#c2410c}{t}} = 2\,\mathrm{h}$; ${\color{#15803d}{f_{L}}} = 0.3$; ${\color{#7c3aed}{f_{H}}} = 3$ |
| 3. Unit check | No unit prefixes to expand. t½ and t are both in hours, so k is in h⁻¹ and the product k·t is unit-free, as an exponent must be. (1 h = 60 min; enter both times in the same unit.) |
| 4. Typical rate constant | ${\color{#b91c1c}{k}} = \frac{\ln 2}{{\color{#1d6fb8}{t_{1/2}}}} = \frac{0.6931}{1.5\,\mathrm{h}} = 0.4621\,\mathrm{h^{-1}}$ |
| 5. Patient L: rate constant | $k = {\color{#15803d}{f_{L}}} \times {\color{#b91c1c}{k}} = 0.3 \times 0.4621 = 0.1386\,\mathrm{h^{-1}}$ |
| 6. Patient L: exponent | $k {\color{#c2410c}{t}} = 0.1386\,\mathrm{h^{-1}} \times 2\,\mathrm{h} = 0.2773$ |
| 7. Patient L: percent converted | $100\left(1 - e^{-0.2773}\right) = 100(1 - 0.7579) = 24.21\%$ |
| 8. Typical patient: exponent | $k {\color{#c2410c}{t}} = 0.4621\,\mathrm{h^{-1}} \times 2\,\mathrm{h} = 0.9242$ |
| 9. Typical patient: percent converted | $100\left(1 - e^{-0.9242}\right) = 100(1 - 0.3969) = 60.31\%$ |
| 10. Patient H: rate constant | $k = {\color{#7c3aed}{f_{H}}} \times {\color{#b91c1c}{k}} = 3 \times 0.4621 = 1.386\,\mathrm{h^{-1}}$ |
| 11. Patient H: exponent | $k {\color{#c2410c}{t}} = 1.386\,\mathrm{h^{-1}} \times 2\,\mathrm{h} = 2.773$ |
| 12. Patient H: percent converted | $100\left(1 - e^{-2.773}\right) = 100(1 - 0.0625) = 93.75\%$ |
| 13. Between-patient spread | $93.75 - 24.21 = 69.54\ \text{percentage points}$ |
| 14. Answer | $\boxed{\text{spread} = 69.54}$ |
| 15. What it means | At 2 h, patient L has converted 24.21% and patient H 93.75% (typical 60.31%): a 69.54-percentage-point spread from a 10-fold range of esterase activity. Re-solve with t½ = 0.25 h to see a faster promoiety shrink the spread. |

**Common conversions for this calculation:** 1 h = 60 min; k in h⁻¹ ÷ 60 = k in min⁻¹; ln 2 = 0.6931
</details>

In [ ]:
#@title 4.4 Solve it yourself: First-order prodrug activation in two outlier patients (worked example: how reliably does an ester switch on?) { display-mode: "form" }
_cell('4.4 Solve it yourself: First-order prodrug activation in two outlier patients (worked example: how reliably does an ester switch on?)')
card('l32_activation_spread')

In [ ]:
#@title 4.5 Calculator — first-order activation in two outlier patients { display-mode: "form" }
_cell('4.5 Calculator — first-order activation in two outlier patients')
t_half_typical_h = 1.5 #@param {type:"slider", min:0.1, max:12, step:0.05}
time_h = 2.0 #@param {type:"slider", min:0.5, max:12, step:0.5}
activity_low = 0.3 #@param {type:"number"}
activity_high = 3.0 #@param {type:"number"}
_ = activation_calc(t_half_typical_h, time_h, activity_low, activity_high)

**Practice in minutes and hours.** Fosphenytoin (Part 4 figure above) is converted to phenytoin by phosphatases with a half-life of minutes, so the same first-order arithmetic is usually done in minutes and reported in hours.

**1. Fraction of a prodrug converted at time t, in minutes and hours (practice: fosphenytoin)**

$${\color{#15803d}{k}} = \frac{\ln 2}{{\color{#1d6fb8}{t_{1/2}}}};\quad \%_{conv} = 100\left(1 - e^{-{\color{#15803d}{k}}{\color{#c2410c}{t}}}\right);\quad t_{90} = \frac{\ln 10}{{\color{#15803d}{k}}}$$

| Symbol | Meaning | Role in the calculation | Units | Source / value |
|---|---|---|---|---|
| ${\color{#1d6fb8}{t_{1/2}}}$ | conversion half-life of fosphenytoin to phenytoin in plasma | input | min | approximate literature value (≈ 15 min) |
| ${\color{#c2410c}{t}}$ | time after the infusion ends | input | min | case |
| ${\color{#15803d}{k}}$ | first-order conversion rate constant | intermediate | min^-1 | calculated |

*Source of the relationship:* first-order activation; tripartite fosphenytoin (L32 notes)

**2. Practice problem (case)**

In the emergency department a pharmacist explains to a resident why phenytoin levels drawn right after a fosphenytoin infusion can mislead. Phosphatases convert fosphenytoin to phenytoin with a half-life of about $\mathbf{{\color{#1d6fb8}{15\,\mathrm{min}}}}$. What percentage has been converted $\mathbf{{\color{#c2410c}{45\,\mathrm{min}}}}$ after the end of the infusion, and how long (in minutes and hours) does 90% conversion take?

<details><summary><b>▸ 3. Step-by-step solution (try it first)</b></summary>

| Step | Work |
|---|---|
| 1. General equation | ${\color{#15803d}{k}} = \frac{\ln 2}{{\color{#1d6fb8}{t_{1/2}}}};\quad \%_{conv} = 100\left(1 - e^{-{\color{#15803d}{k}}{\color{#c2410c}{t}}}\right);\quad t_{90} = \frac{\ln 10}{{\color{#15803d}{k}}}$ |
| 2. Values from the case | ${\color{#1d6fb8}{t_{1/2}}} = 15\,\mathrm{min}$; ${\color{#c2410c}{t}} = 45\,\mathrm{min}$ |
| 3. Unit check | No unit prefixes to expand. Both times are in minutes, so k is in min⁻¹ and k·t is unit-free. To report k per hour, multiply by 60 min/h; to report a time in hours, divide minutes by 60 min/h. |
| 4. Rate constant | ${\color{#15803d}{k}} = \frac{\ln 2}{{\color{#1d6fb8}{t_{1/2}}}} = \frac{0.6931}{15\,\mathrm{min}} = 0.04621\,\mathrm{min^{-1}}$ |
| 5. Rate constant per hour | $0.04621\,\mathrm{min^{-1}} \times 60\,\mathrm{min/h} = 2.773\,\mathrm{h^{-1}}$ |
| 6. Exponent | ${\color{#15803d}{k}} {\color{#c2410c}{t}} = 0.04621 \times 45 = 2.079$ |
| 7. Fraction remaining as prodrug | $e^{-2.079} = 0.125$ |
| 8. Percent converted | $100(1 - 0.125) = 87.5\%$ |
| 9. Time to 90% conversion | $t_{90} = \frac{\ln 10}{{\color{#15803d}{k}}} = \frac{2.303}{0.04621\,\mathrm{min^{-1}}} = 49.83\,\mathrm{min} = \frac{49.83}{60}\,\mathrm{h} = 0.8305\,\mathrm{h}$ |
| 10. Answer | $\boxed{\%_{conv} = 87.5\,\mathrm{\%}}$ |
| 11. What it means | After 45 min (0.75 h), 87.5% of the fosphenytoin has been converted to phenytoin; 90% conversion takes 49.83 min (0.8305 h). Because each half-life halves what is left, 45 min is exactly 3 half-lives. |

**Common conversions for this calculation:** 1 h = 60 min; k (h⁻¹) = 60 × k (min⁻¹); ln 2 = 0.6931; ln 10 = 2.303
</details>

In [ ]:
#@title 4.6 Solve it yourself: Fraction of a prodrug converted at time t, in minutes and hours (practice: fosphenytoin) { display-mode: "form" }
_cell('4.6 Solve it yourself: Fraction of a prodrug converted at time t, in minutes and hours (practice: fosphenytoin)')
card('l32_conversion_halflife')

## Part 5 - How fast does the promoiety come off?

*Objectives addressed: L32.1 · L32.3*

A carrier-linked ester prodrug is activated by hydrolysis. The left panel
shows prodrug disappearing and active drug appearing. The right panel shows
the **same prodrug in three patients** whose esterase activity differs
threefold either way.

Work through the promoieties from fastest to slowest and watch both panels.

<details>
<summary><b>Stuck? Open this before you ask a neighbour</b></summary>

The two panels answer different questions, and the second is the one that
matters clinically.

* **Left panel: how fast?** An electron-withdrawing 5-indanyl ester is
  essentially fully activated within two hours. A hindered pivaloyl ester
  takes most of a working day. An amide barely moves.
* **Right panel: how *reliably* fast?** Look at the spread between the three
  patients at two hours. A very fast promoiety is nearly complete in all
  three, so the spread is small. A very slow one has barely started in any of
  them, so the spread is also small. It is the *intermediate* rates where the
  three patients diverge most — and that is where dosing becomes
  unpredictable.

Read `patient spread at 2 h` for each promoiety before you decide which you
would choose.

</details>

In [ ]:
#@title Activate the prodrug { display-mode: "form" }
#@markdown Choose a promoiety and the patient's enzyme activity.
promoiety = "ethyl ester" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide", "no promoiety"]
patient_esterase_activity = 1.0 #@param {type:"slider", min:0.1, max:3, step:0.1}
hours_followed = 8 #@param {type:"slider", min:1, max:24, step:1}
r = sims.prodrug_kinetics(promoiety, patient_esterase_activity, hours_followed)
print(f"rate constant k        {r['k_per_h']:.3f} per hour")
hl = r['half_life_h']
print(f"half-life of activation{'  ' + format(hl, '.2f') + ' h' if hl else '  effectively none'}")
print(f"active drug at 2 h     {r['pct_active_at_2h']:.1f}% of dose")
print(f"patient spread at 2 h  {r['patient_spread_at_2h_pct']:.1f} percentage points")

**Now put two settings side by side.** Run the cell below and compare the two figures it prints, one above the other.

In [ ]:
#@title Side by side: a fast, a moderate and a slow promoiety { display-mode: "form" }
#@markdown Three promoieties, same drug, same patients.
fast_option = "5-indanyl ester (EWG)" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide"]
middle_option = "ethyl ester" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide"]
slow_option = "pivaloyl ester (hindered)" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide"]
out = []
for p in (fast_option, middle_option, slow_option):
    print("=" * 70); print(p.upper()); print("=" * 70)
    out.append((p, sims.prodrug_kinetics(p, 1.0, 8.0)))
print()
print(f"{'promoiety':<30}{'k /h':>8}{'t1/2 h':>9}{'%@2h':>8}{'spread':>9}")
for p, r in out:
    hl = f"{r['half_life_h']:.2f}" if r['half_life_h'] else "-"
    print(f"{p:<30}{r['k_per_h']:>8.2f}{hl:>9}{r['pct_active_at_2h']:>8.1f}"
          f"{r['patient_spread_at_2h_pct']:>8.1f}%")
print("\nThe largest between-patient spread is not at the fastest or the slowest rate.")

In [ ]:
#@title C1   (run after choosing) { display-mode: "form" }
#@markdown **C1.**  The ethyl ester showed a far larger between-patient spread at two hours than the indanyl ester did. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. At an intermediate rate, two hours falls where the curves differ most", "B. The ethyl ester is chemically unstable and decomposes unpredictably", "C. Esterase activity varies more widely for ethyl esters than aryl ones", "D. The indanyl ester is activated by a route that needs no enzyme at all"]
gb.answer("C1", response[:1])

In [ ]:
#@title C2   (run after choosing) { display-mode: "form" }
#@markdown **C2.**  You must choose a promoiety for a drug with a narrow therapeutic window. Which consideration should dominate?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Reliability of activation across patients, not raw speed alone", "B. The lowest possible rate constant, to avoid any sudden release", "C. The highest possible rate constant, to minimise exposure time", "D. The lowest molecular weight promoiety that can be attached"]
gb.answer("C2", response[:1])

## Part 6 - Design a prodrug for the problem in front of you

*Objectives addressed: L32.4*


Everything so far has been analysis: given a prodrug, say what it does. Design runs the other way: given a **problem**, build the prodrug. A reliable routine is four steps — (1) name the property problem and the group that causes it; (2) choose a promoiety that fixes the property and hangs off a group the drug can spare; (3) name the **activating enzyme or trigger** and where in the body it acts; (4) check that what falls off is harmless and that activation is reliable across patients.

Pick a problem in the tool below, **write your own four-step answer first**, then open the suggestion and compare. A different promoiety is right whenever it fixes the same property, attaches to a group the drug can spare, and is unmasked by a named enzyme.

In [ ]:
#@title 6.1 Tool — design a prodrug: problem → promoiety → activating enzyme { display-mode: "form" }
_cell('0.1 Tool — design a prodrug: problem → promoiety → activating enzyme')
design_problem = "(choose a problem)" #@param ["(choose a problem)", "An acidic drug (carboxyl/phosphate) is barely absorbed orally", "A neutral, poorly soluble drug must be given by injection", "A polar drug cannot cross the blood–brain barrier", "A drug is too toxic to healthy tissue and should act mainly in a tumour", "A drug is cleared too fast and needs a longer duration", "An oral liquid for children tastes unacceptably bitter"]
_ = prodrug_designer(design_problem)

In [ ]:
#@title Check L32-X1   (run after choosing) { display-mode: "form" }
#@markdown **L32-X1.** A potent drug carries a phosphate group it needs for binding, but the charge keeps it from being absorbed when swallowed. You must design an orally absorbed prodrug and name what switches it on. Which plan is soundest?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Mask the phosphate as a carbonate–acetal ester arm, unmasked by esterases then self-immolation", "B. Mask the phosphate as a permanent methyl phosphonate, which resists charge at every pH", "C. Attach a second copy of the drug through the phosphate, cleaved by a plasma phosphatase", "D. Leave the phosphate and add a fixed quaternary ammonium to raise aqueous solubility"]
gb.answer("L32-X1", response[:1])

In [ ]:
#@title Check L32-X2   (run after choosing) { display-mode: "form" }
#@markdown **L32-X2.** An antiviral is active only after a cell adds three phosphates to it, and you want it unmasked mainly inside virus-infected cells. You design a prodrug and must name the first activating step. Which design and enzyme fit the goal?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. An amino-acid ester cleaved by gut esterase, then a first phosphate added by viral thymidine kinase", "B. A methyl ester cleaved by plasma esterase, then all three phosphates added by host kinases alone", "C. A permanent fluorine for hydrogen, with the three phosphates added by any cellular kinase", "D. A carbonate linker cleaved by phosphatase, which removes the need for any kinase step"]
gb.answer("L32-X2", response[:1])

## Part 7 - Prodrugs in practice (reference)

*Objectives addressed: L32.2*


| Prodrug | Active drug | Promoiety / activation | Reason for the prodrug | Target / use |
|---|---|---|---|---|
| **Enalapril** | enalaprilat | ethyl ester; hepatic CES1 | oral absorption | angiotensin-converting enzyme (zinc metalloprotease); hypertension, heart failure |
| **Oseltamivir** | oseltamivir carboxylate | ethyl ester; hepatic CES1 | oral absorption | influenza neuraminidase; influenza A and B |
| **Valacyclovir** | acyclovir → acyclovir triphosphate | L-valyl ester; PEPT1 uptake, valacyclovirase; then viral thymidine kinase | oral absorption; activation only in infected cells | herpesvirus DNA polymerase; HSV and VZV infections |
| **Levodopa** (+ carbidopa) | dopamine | none (bioprecursor); LAT1 transport, AADC decarboxylation | crossing the blood–brain barrier | dopamine receptors; Parkinson's disease |
| **Fosphenytoin** | phenytoin | phosphate + CH₂ linker (tripartite); phosphatases, then spontaneous loss of formaldehyde | water solubility for IV use | voltage-gated Na⁺ channels; status epilepticus |
| **Tenofovir disoproxil** | tenofovir (→ diphosphate) | two carbonate–acetal arms (tripartite); esterases | oral absorption of a phosphonate | HIV reverse transcriptase, HBV polymerase |
| **Candesartan cilexetil** | candesartan | carbonate–acetal ester (tripartite); hydrolysed in the gut wall during absorption | oral absorption | AT₁ angiotensin II receptor; hypertension |
| **Capecitabine** | 5-fluorouracil | carbamate + sugar; CES1 → cytidine deaminase → thymidine phosphorylase (high in many tumours) | oral use; more drug released in tumour | thymidylate synthase (as FdUMP); colorectal and breast cancer |
| **Irinotecan** | SN-38 | piperidinopiperidine carbamate; CES2 | water solubility for IV use | topoisomerase I; colorectal cancer |
| **Bambuterol** | terbutaline | bis-dimethylcarbamate; plasma cholinesterase (slow) | long duration | β₂-adrenergic receptor agonist; asthma |
| **Clopidogrel** | active thiol metabolite | none (bioprecursor); CYP2C19 and other CYPs, two oxidations | — (discovered as such) | P2Y₁₂ ADP receptor, irreversible; prevention of arterial thrombosis |
| **Omeprazole** | sulfenamide | none; acid-catalysed rearrangement | activation only in the acidic parietal-cell canaliculus | H⁺/K⁺-ATPase (covalent); peptic ulcer, reflux |

**Looking back (L31 Bioisosteres).** A bioisostere changes a group **permanently** to fix a property; a prodrug changes it **temporarily** and relies on the body to undo the change. The two are often combined: candesartan keeps the **tetrazole** bioisostere for its acid contact and adds a **cilexetil ester** prodrug on its carboxylic acid for oral absorption. When a property problem appears in a lead, ask both questions — can the group be replaced, or only hidden until absorption?

## Part 8 - End-of-activity assessment

Five questions in the style of the course quizzes and exams, mapped to the
Student Intended Learning Outcomes for this unit (**SILO 4**).
This time you get full feedback, including why each wrong option is wrong.
Answer each one, then run the summary cell at the bottom.

In [ ]:
#@title Q1   (run after choosing) { display-mode: "form" }
#@markdown **Q1.**  SAR data: OCH2 active, NHCH2 active, SCH2 inactive, CH2CH2 inactive, CH=CH inactive. What does the pattern establish about the oxygen?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It contributes bulk that the binding site sterically requires", "B. It stabilizes the active conformation through its bond angle", "C. It raises lipophilicity enough to drive membrane permeation", "D. It acts as a hydrogen-bond acceptor at that position in the site"]
gb.answer("Q1", response[:1])

In [ ]:
#@title Q2   (run after choosing) { display-mode: "form" }
#@markdown **Q2.**  A CF3 group is introduced two bonds from a basic amine that must engage an anionic site residue. Potency falls. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. CF3 sterically blocks the amine's approach to the site residue", "B. CF3 lowers the amine's pKa, reducing the protonated fraction", "C. CF3 is metabolized to an acid that competes for the same site", "D. CF3 raises lipophilicity, which excludes the drug from plasma"]
gb.answer("Q2", response[:1])

In [ ]:
#@title Q3   (run after choosing) { display-mode: "form" }
#@markdown **Q3.**  Two esters of one drug differ only in the alcohol: ethanol versus 5-indanol. How will activation compare, and why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The indanyl ester hydrolyzes faster, as the aryl group withdraws density", "B. Neither hydrolyzes in plasma, since aryl esters require liver enzymes", "C. Both hydrolyze identically, since esterases are insensitive to structure", "D. The ethyl ester hydrolyzes faster, as smaller esters are less hindered"]
gb.answer("Q3", response[:1])

In [ ]:
#@title Q4   (run after choosing) { display-mode: "form" }
#@markdown **Q4.**  Chloramphenicol is formulated as the palmitate ester for oral suspension and the succinate ester for injection. What does this contrast demonstrate?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Two promoieties are required before any prodrug can be marketed", "B. Ester prodrugs invariably increase the aqueous solubility of a drug", "C. Promoiety choice is dictated by the route and the dosage form needed", "D. The parent drug is inactive until both esters have been hydrolyzed"]
gb.answer("Q4", response[:1])

In [ ]:
#@title Q5   (run after choosing) { display-mode: "form" }
#@markdown **Q5.**  A prodrug performs consistently in plasma incubations but gives highly variable exposure across patients. Which explanation fits best?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The promoiety itself is pharmacologically active in some patients", "B. Activating enzyme or gut flora activity differs between individuals", "C. The linkage hydrolyzes chemically rather than enzymatically in vivo", "D. Plasma incubation overestimates the drug's intrinsic potency in vivo"]
gb.answer("Q5", response[:1])

## Lab report (automatic)
Every figure you displayed, every calculation you solved with **Solve step by step** and your first answer to each check were collected as you worked. Add your name and a short discussion, then build the report: a **Word (.docx)** file to edit and submit and an **HTML** copy with the equations exactly as shown here. A file is written only when you press the button.

In [ ]:
#@title Build and download my lab report { display-mode: "form" }
_cell('Build and download my lab report')
report_panel()

## Your results

In [ ]:
#@title Show my performance and my report code { display-mode: "form" }
gb.summary()
print()
_ = gb.report_code()

### What to submit

1. Copy the single `L32|...` line printed above.
2. Paste it into the Canvas assignment **L32 Colab activity**.
3. With your partner, be ready to say one thing the simulation showed you that
   the reading did not.

Your report code records how many questions you answered correctly. It is a
participation and self-check record, not a proctored exam score.